**Risk of Incident Cytopenia/Cytoses in Clonal Hematopoiesis** 

This notebook focuses on analyzing the rate of transformation from clonal hematopoiesis (CH) to CH with cytopenia.

In [ ]:
# ! pip install lifelines

In [ ]:
# !pip uninstall -y shapely pygeos geopandas
# # Install specific versions of shapely, pygeos, and geopandas known to be compatible
# !pip install shapely==1.8.5.post1 pygeos==0.9.0 geopandas==0.10.2
# # Upgrade google-cloud-aiplatform
# !pip install -U google-cloud-bigquery

In [ ]:
# ! pip install pandas-gbq -U

In [ ]:
import pandas as pd
from typing import Dict, List, Tuple
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.plotting import add_at_risk_counts
from lifelines.statistics import logrank_test, multivariate_logrank_test
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import rcParams
import random
import os
import pytz
from unifier_functions import * # Assumes you have unifier_functions.py in your current directory 
from collections import Counter

## Defining necessary pathways
BUCKET = os.environ['WORKSPACE_BUCKET']
DATASET = os.environ['WORKSPACE_CDR']

# Defining global variables
cohort_name = 'aou'
file_path = 'pershy1/mca_cell_counts'

offset = 0

unit_of_time = 365.25 # days == 1, months == 30.4375, years == 365.25
unit_of_time_label = 'Years'
x_max = 3
y_max = 0.4

control_variables = ['gender', 'ever_smoker']
max_controls = 3
max_age_difference = 3
match_name = f'age_gender_{max_controls}_{max_age_difference}'

## Setting dataframe display dimensions
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 150)
pd.set_option('display.max_colwidth', None)

# Set the default font to DejaVu Sans
rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['DejaVu Sans']

In [ ]:
def check_column_types(df, column_types):
    """
    Check the types of columns in a DataFrame.

    Parameters:
    - df: pandas DataFrame to check.
    - column_types: dictionary where keys are column names and values are expected types (e.g., {'column_name': str}).

    Returns:
    - True if all columns are correctly typed and no columns are missing or extra.
    - False if there are issues with missing columns, extra columns, or incorrect types, and prints issues.
    """
    errors = []

    # Check if all keys in column_types are in the DataFrame columns
    missing_columns = [col for col in column_types if col not in df.columns]
    extra_columns = [col for col in df.columns if col not in column_types]
    
    if missing_columns:
        errors.append(f"Missing columns in DataFrame: {', '.join(missing_columns)}")
    if extra_columns:
        errors.append(f"Extra columns in DataFrame: {', '.join(extra_columns)}")
    
    # Check if columns have the correct types
    incorrect_types = []
    for column, expected_type in column_types.items():
        if column in df.columns:
            pandas_type = pd.api.types.pandas_dtype(expected_type)
            if not pd.api.types.is_dtype_equal(df[column].dtype, pandas_type):
                incorrect_types.append(f"{column} (expected {pandas_type}, got {df[column].dtype})")
    
    if incorrect_types:
        errors.append(f"Columns with incorrect types: {', '.join(incorrect_types)}")
    
    # Print errors and return False if any issues are found
    if errors:
        for error in errors:
            print(error)
        return False
    
    return True

# Prepare input data

Formulate a table of participants that have genetic sequencing data

In [ ]:
def check_column_types(df, column_types):
    """
    Check the types of columns in a DataFrame.

    Parameters:
    - df: pandas DataFrame to check.
    - column_types: dictionary where keys are column names and values are expected types (e.g., {'column_name': str}).

    Returns:
    - True if all columns are correctly typed and no columns are missing or extra.
    - False if there are issues with missing columns, extra columns, or incorrect types, and prints issues.
    """
    errors = []

    # Check if all keys in column_types are in the DataFrame columns
    missing_columns = [col for col in column_types if col not in df.columns]
    extra_columns = [col for col in df.columns if col not in column_types]
    
    if missing_columns:
        errors.append(f"Missing columns in DataFrame: {', '.join(missing_columns)}")
    if extra_columns:
        errors.append(f"Extra columns in DataFrame: {', '.join(extra_columns)}")
    
    # Check if columns have the correct types
    incorrect_types = []
    for column, expected_type in column_types.items():
        if column in df.columns:
            pandas_type = pd.api.types.pandas_dtype(expected_type)
            if not pd.api.types.is_dtype_equal(df[column].dtype, pandas_type):
                incorrect_types.append(f"{column} (expected {pandas_type}, got {df[column].dtype})")
    
    if incorrect_types:
        errors.append(f"Columns with incorrect types: {', '.join(incorrect_types)}")
    
    # Print errors and return False if any issues are found
    if errors:
        for error in errors:
            print(error)
        return False
    
    return True

In [ ]:
# Define chunk size (e.g., 1000 elements per chunk)
chunk_size = 50000

# Function to chunk a list into smaller lists
def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]
        

def split_vaf_rows(df):
    # Split the rows with commas in the VAF column
    rows_with_commas = df[df['VAF'].str.contains(',')]
    rows_without_commas = df[~df['VAF'].str.contains(',')]

    # Split VAF values and create new rows
    split_rows = rows_with_commas.assign(VAF=rows_with_commas['VAF'].str.split(',')).explode('VAF').drop_duplicates()

    # Concatenate the split rows with the rows without commas
    result_df = pd.concat([rows_without_commas, split_rows], ignore_index=True)

    return result_df

## Demographic and genetic data

In [ ]:
# mca_df = pd.read_table('gs://fc-secure-cb192ac6-30ba-46b9-92ee-896a6e36c63e/yp_mca_calls_021924.txt')
# autosomal_df = mca_df[mca_df['chrom']!='chrX']
# autosomal_df['arm'] = ''
# autosomal_df.loc[((autosomal_df['p_arm']=='T') | (autosomal_df['p_arm']=='T') | (autosomal_df['p_arm']=='C')) & 
#                  ~((autosomal_df['q_arm']=='T') | (autosomal_df['q_arm']=='Y')), 'arm'] = 'p'
# autosomal_df.loc[~((autosomal_df['p_arm']=='T') | (autosomal_df['p_arm']=='T') | (autosomal_df['p_arm']=='C')) & 
#                  ((autosomal_df['q_arm']=='T') | (autosomal_df['q_arm']=='Y')), 'arm'] = 'q'

# autosomal_df['type'] = autosomal_df['chrom']+autosomal_df['arm']+ " " + autosomal_df['type']
# mca_clean_df = autosomal_df[['sample_id', 'type', 'cf']]
# mca_clean_df.columns = ['person_id', 'Gene.refGene', 'AF_main']
# mca_clean_df.to_csv(f'{BUCKET}/pershy1/mca_cell_counts/mca_calls.txt',
#                                                 sep='\t', index=False)

In [ ]:
column_types = {'person_id': 'string', 'Gene.refGene': 'string', 'AF_main': 'float64'}
# variants_one = pd.read_csv(f'{BUCKET}/AoU_CHIP_Calls/CHIP_calls_per_person_nomyeloidcancers_12082022.txt', 
#                            delimiter='\t', 
#                            dtype=column_types, 
#                            low_memory=False)

# variants_two = pd.read_csv(f'{BUCKET}/AoU_CHIP_Calls/CHIP_calls_perperson_batch2_nomyeloidcancers_06162023.txt', 
#                            delimiter='\t', 
#                            dtype=column_types, 
#                            low_memory=False)

# # Concatenate variants data
# variants = pd.concat([variants_one, variants_two], axis=0).reset_index(drop=True)

variants = pd.read_table(f'{BUCKET}/pershy1/mca_cell_counts/mca_calls.txt',
                         dtype=column_types, 
                         low_memory=False).reset_index(drop=True)

# # Prepare variant data for merging
# variant_slice = variants[['person_id', 'survey_datetime', 'Gene.refGene', 'AF_main']].copy(deep=True)
variant_slice = variants[['person_id', 'Gene.refGene', 'AF_main']].copy(deep=True)

# variant_slice.rename(columns={'survey_datetime': 'index_datetime', 'Gene.refGene': 'gene', 'AF_main': 'VAF'}, inplace=True)
variant_slice.rename(columns={'Gene.refGene': 'gene', 'AF_main': 'VAF'}, inplace=True)

In [ ]:
demographics_query = f"""
        SELECT
            p.person_id,
            p.birth_datetime,
            r.concept_name AS race,
            e.concept_name AS ethnicity,
            d.death_date,
            LOWER(g.concept_name) AS gender,
            sc.survey_datetime
        FROM
            `{DATASET}.person` p
        LEFT JOIN
            `{DATASET}.concept` g
        ON
            p.sex_at_birth_concept_id = g.concept_id
        LEFT JOIN
            `{DATASET}.concept` r
        ON
            p.race_concept_id = r.concept_id
        LEFT JOIN
            `{DATASET}.concept` e
        ON
            p.ethnicity_concept_id = e.concept_id
        LEFT JOIN
            `{DATASET}.death` d
        ON
            p.person_id = d.person_id
        LEFT JOIN
            `{DATASET}.ds_survey` sc
        ON
            p.person_id = sc.person_id
        GROUP BY
            p.person_id, p.birth_datetime, r.concept_name, e.concept_name, d.death_date, g.concept_name, sc.survey_datetime
"""

demographics = pandas_gbq.read_gbq(
    demographics_query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

demographics['person_id'] = demographics['person_id'].astype('string')

# changed this to left join to include non-mCA people.
demographics = demographics.merge(variant_slice, on='person_id', how='left')
# demographics = demographics.merge(variant_slice, on='person_id', how='left')

demographics.head()

In [ ]:
demographics.rename(columns={'survey_datetime': 'index_datetime'}, inplace=True)

For EHR recorded smoking, we found all individuals with ICD9CM codes 305.1, V15.82 and ICD10CM codes Z72.0 or O99.33*. We labeled individuals with at least one of these codes on at least two separate calendar days as ever smokers.

In [ ]:
# Define the query to get smoking-related information
smoker_query = f"""
-- Step 1: Define a CTE (Common Table Expression) to identify smoking-related concept IDs
WITH concept_smoking AS (
    SELECT DISTINCT concept_id, concept_code, vocabulary_id
    FROM `{DATASET}.concept`
    WHERE 
        (concept_code = 'V15.82' AND vocabulary_id = 'ICD9CM') OR 
        (concept_code IN ('Z72.0', 'Z87.891', 'Z71.6') AND vocabulary_id = 'ICD10CM') OR
        (concept_code LIKE '305.1%' OR concept_code LIKE '649.0%' OR concept_code LIKE '989.84%' AND vocabulary_id = 'ICD9CM') OR
        (concept_code IN ('Z72.0') OR concept_code LIKE 'O99.33%' OR 
         (concept_code LIKE 'F17.2%' AND concept_code NOT LIKE 'F17.22%') OR 
         (concept_code LIKE 'T65.2%' AND concept_code NOT LIKE 'T65.21%') AND vocabulary_id = 'ICD10CM')
),

-- Step 2: Identify smoking-related observation dates for each person
observation_smoking AS (
    SELECT DISTINCT person_id, observation_date AS code_date
    FROM `{DATASET}.observation`
    WHERE observation_source_concept_id IN (SELECT concept_id FROM concept_smoking)
),

-- Step 3: Identify smoking-related condition start dates for each person
condition_smoking AS (
    SELECT DISTINCT person_id, condition_start_date AS code_date
    FROM `{DATASET}.condition_occurrence`
    WHERE condition_source_concept_id IN (SELECT concept_id FROM concept_smoking)
),

-- Step 4: Combine all smoking-related dates from observations and conditions
all_smoking AS (
    SELECT person_id, code_date
    FROM observation_smoking
    UNION ALL
    SELECT person_id, code_date
    FROM condition_smoking
),

-- Step 5: Count the number of distinct smoking-related dates for each person
smoker_counts AS (
    SELECT person_id, COUNT(DISTINCT code_date) AS count
    FROM all_smoking
    GROUP BY person_id
),

-- Step 6: Determine the smoking status for each person and retain the count of smoking-related codes
ever_smoker_status AS (
    SELECT person_id,
           COUNT(DISTINCT code_date) AS count,
           CASE 
               WHEN COUNT(DISTINCT code_date) >= 2 THEN 1  -- Consider as smoker if 2 or more distinct dates
               WHEN COUNT(DISTINCT code_date) = 1 THEN -9  -- Mark as potential smoker if exactly 1 date
               ELSE 0                                      -- Non-smoker if no dates
           END AS ever_smoke_ehr
    FROM all_smoking
    GROUP BY person_id
)

-- Step 7: Select the final smoking status and count for each person
SELECT person_id, MAX(ever_smoke_ehr) AS ever_smoker, MAX(count) AS smoke_code_count
FROM ever_smoker_status
GROUP BY person_id
"""

# Execute the query to get the DataFrame with smoking-related information
ever_smoker = pd.read_gbq(
    smoker_query, 
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

ever_smoker['person_id'] = ever_smoker['person_id'].astype('string')
ever_smoker.head()

In [ ]:
# Merge participants with ever_smoker on person_id
participants = pd.merge(demographics, ever_smoker[['person_id', 'ever_smoker']], on='person_id', how='left')

# Replace -9 with 0 in the ever_smoke_ehr column and convert the column to integer type
participants['ever_smoker'] = participants['ever_smoker'].replace(-9, 0).fillna(0).astype(int)

participants['gender'] = participants['gender'].apply(lambda x: x if x in ['male', 'female'] else 'other')

# Map race values to a simplified subset
race_map = {
    'White': 'white',
    'Black or African American': 'black',
    'American Indian or Alaska Native': 'indian_alaskan',
    'Asian': 'asian',
    'Middle Eastern or North African': 'middle_eastern',
    'What Race Ethnicity: Race Ethnicity None Of These': 'white_none',
    'Native Hawaiian or Other Pacific Islander': 'hawaiian_pacific',
    'No matching concept': 'missing'
}
participants['race'] = participants['race'].map(race_map).fillna('missing')

# Map ethnicity values to a simplified subset
ethnicity_map = {
    'Not Hispanic or Latino': 'not_hispanic',
    'Hispanic or Latino': 'hispanic',
    'What Race Ethnicity: Race Ethnicity None Of These': 'none_of_these',
    'PMI: Skip': 'missing',
    'PMI: Prefer Not To Answer': 'missing',
    np.nan: 'missing'
}
participants['ethnicity'] = participants['ethnicity'].map(ethnicity_map).fillna('missing')

# Convert birth_datetime to datetime64[ns, UTC]
participants['birth_datetime'] = pd.to_datetime(participants['birth_datetime'], utc=True)

# Convert death_date to datetime64[ns, UTC]
participants['death_date'] = pd.to_datetime(participants['death_date'], errors='coerce', utc=True)

# Convert index_datetime to datetime64[ns, UTC]
participants['index_datetime'] = pd.to_datetime(participants['index_datetime'], utc=True)

# Ensure all columns are in datetime64[ns, UTC]
participants['birth_datetime'] = participants['birth_datetime'].astype('datetime64[ns, UTC]')
participants['death_date'] = participants['death_date'].astype('datetime64[ns, UTC]')
participants['index_datetime'] = participants['index_datetime'].astype('datetime64[ns, UTC]')


participants['gender'] = participants['gender'].astype('string')
participants['race'] = participants['race'].astype('string')
participants['ethnicity'] = participants['ethnicity'].astype('string')

genetic_order = [
    'person_id', 'gene', 'VAF'
]

demographic_order = [
    'person_id', 'birth_datetime', 'death_date', 'index_datetime', 'gender', 
    'race', 'ethnicity', 'ever_smoker'
]

ch_pids = variants['person_id'].to_list() 

genetic_data = participants[genetic_order]
genetic_data = genetic_data[genetic_data['person_id'].isin(ch_pids)].reset_index(drop=True)

demographic_data = participants[demographic_order].drop_duplicates(subset='person_id').reset_index(drop=True)

In [ ]:
assert check_column_types(demographic_data, {'person_id': 'string', 
                                             'birth_datetime': 'datetime64[ns, UTC]',
                                             'death_date': 'datetime64[ns, UTC]',
                                             'index_datetime': 'datetime64[ns, UTC]',
                                             'gender': 'string',
                                             'race': 'string',
                                             'ethnicity': 'string',
                                             'ever_smoker': 'int'})

assert check_column_types(genetic_data, {'person_id': 'string', 'gene': 'string', 'VAF': 'float'})

In [ ]:
save_to_bucket(genetic_data, f'{file_path}', f'{cohort_name}_genetic_data')
save_to_bucket(demographic_data, f'{file_path}', f'{cohort_name}_demographic_data')

## Phenotype data

In [ ]:
# icd_codes = pd.read_excel('icd_codes.xlsx', 
#                           sheet_name='input_icd_codes', 
#                           dtype={'ICD': 'str'})

In [ ]:
# icd_list = icd_codes['ICD'].to_list()
icd_codes = pd.read_csv('gs://fc-secure-cb192ac6-30ba-46b9-92ee-896a6e36c63e/broganjf/chip_cytopenia/aou_phenotype_data.csv')
icd_list = list(icd_codes['ICD'].unique())

query = f"""
        SELECT person_id
            , condition_start_date
            , concept_name as ICD_string
            , vocabulary_id
            , concept_code as ICD
        FROM 
            {DATASET}.condition_occurrence    
        LEFT JOIN `{DATASET}.concept` as c on c.concept_id = condition_source_concept_id
        WHERE vocabulary_id IN ('ICD9CM', 'ICD10CM') AND concept_code IN UNNEST({icd_list})
        ORDER BY
            person_id
        """

icd_diagnoses = pd.read_gbq(query, use_bqstorage_api=True, progress_bar_type='tqdm_notebook')

In [ ]:
phenotypes = pd.merge(icd_diagnoses, 
                      icd_codes[['ICD', 'phenotype', 'phenotype_class']].drop_duplicates(), 
                      on='ICD', 
                      how='left')

phenotypes['person_id'] = phenotypes['person_id'].astype('string')
phenotypes['condition_start_date'] = pd.to_datetime(phenotypes['condition_start_date']).dt.tz_localize('UTC')
phenotypes['ICD_string'] = phenotypes['ICD_string'].astype('string')
phenotypes['vocabulary_id'] = phenotypes['vocabulary_id'].astype('string')
phenotypes['ICD'] = phenotypes['ICD'].astype('string')
phenotypes['phenotype'] = phenotypes['phenotype'].astype('string')
phenotypes['phenotype_class'] = phenotypes['phenotype_class'].astype('string')

phenotypes.head()

In [ ]:
assert check_column_types(phenotypes, {'person_id': 'string', 
                                       'condition_start_date': 'datetime64[ns, UTC]',
                                       'ICD_string': 'string',
                                       'vocabulary_id': 'string',
                                       'ICD': 'string',
                                       'phenotype': 'string',
                                       'phenotype_class': 'string'})

In [ ]:
save_to_bucket(phenotypes, file_path, f'{cohort_name}_phenotype_data')

## Laboratory measurements

In [ ]:
# Read in tables for unit processing
tables = read_tables('v0_2')
metadata = tables['metadata']
unit_map = tables['unit_map']
unit_reduce = tables['unit_reduce']

m_cids = [3000963, 3000905, 3024929, 3019897, 3023599, 3013650, 3004327]
m_vars = metadata[metadata['measurement_concept_id'].isin(m_cids)]['lab_name'].to_list()
print(f'Preparing laboratory measurements for the following variables: {m_vars}')

**Build CBC DataFrame**

In [ ]:
demographic_data = read_from_bucket(file_path, 
                                    f'{cohort_name}_demographic_data', 
                                    column_types={'person_id': 'string', 
                                                  'gender': 'string',
                                                  'race': 'string',
                                                  'ethnicity': 'string',
                                                  'ever_smoker': 'int',}, 
                                    datetime_columns=['index_datetime'])

In [ ]:
pids = demographic_data['person_id'].to_list()

# Initialize an empty list to hold dataframes
m_frames = []

# Process each chunk
for chunk in chunk_list(pids, chunk_size):
    m_chunk = participant_omop_query(m_cids, chunk)
    m_frames.append(m_chunk)

# Combine all dataframes into a single dataframe
m = pd.concat(m_frames, ignore_index=True)
query_summary(m)

In [ ]:
m.columns

In [ ]:
# Run quality control process
m_preprocessed = preprocess(m)
m_harmonized = harmonize(m_preprocessed, metadata, unit_map, unit_reduce)
m_final = trim(m_harmonized)

# Descriptive statistics after outliers are removed
m_unitdata = units_dist(m_harmonized)

In [ ]:
cbc = m_final.pivot_table(index=['person_id', 'measurement_datetime'], 
                          columns='lab_name', values='value_as_number').reset_index().rename_axis(None, axis=1)

# Define the lists
m_vars_old = ['hemoglobin', 'mean corpuscular volume', 'platelets', 'red cell distribution width', 
              'leukocyte count', 'neutrophil count', 'lymphocyte count']
m_vars_new = ['hgb', 'mcv', 'plt', 'rdw', 'wbc', 'neu', 'alc']

# Create a dictionary to map m_vars_old to m_vars_new
m_vars_dict = dict(zip(m_vars_old, m_vars_new))

# Rename the columns in the DataFrame
cbc.rename(columns=m_vars_dict, inplace=True)

m_vars = ['hgb', 'wbc', 'plt']

# Filter rows where all specified columns are not NaN
cbc = cbc.dropna(subset=m_vars)

cbc['person_id'] = cbc['person_id'].astype('string')

cbc.head()

In [ ]:
def check_column(df, column_name):
    """
    Checks if the specified column in the DataFrame has any missing values.

    Parameters:
    df (pandas.DataFrame): The DataFrame to check for missing values.
    column_name (str): The name of the column to check for missing values.

    Returns:
    bool: True if the specified column has no missing values, False if it has any missing values.
    """
    if column_name not in df.columns:
        raise ValueError(f"Column '{column_name}' does not exist in the DataFrame.")
    
    return not df[column_name].isnull().any()

In [ ]:
assert check_column(cbc, 'hgb') == True
assert check_column(cbc, 'plt') == True
assert check_column(cbc, 'wbc') == True
assert check_column(cbc, 'mcv') == False
assert check_column(cbc, 'rdw') == False
assert check_column(cbc, 'alc') == False

In [ ]:
cbc['measurement_datetime'] = cbc['measurement_datetime'].astype('datetime64[ns, UTC]')

In [ ]:
assert check_column_types(cbc, {'person_id': 'string', 
                                'measurement_datetime': 'datetime64[ns, UTC]',
                                'hgb': 'float64',
                                'mcv': 'float64',
                                'plt': 'float64',
                                'rdw': 'float64',
                                'wbc': 'float64', 
                                'neu': 'float64',
                                'alc': 'float64'})

In [ ]:
save_to_bucket(cbc, file_path, f'{cohort_name}_measurement_data')

# Process participant data

In [ ]:
demographic_data = read_from_bucket(file_path, 
                                    f'{cohort_name}_demographic_data', 
                                    column_types={'person_id': 'string', 
                                                  'gender': 'string',
                                                  'race': 'string',
                                                  'ethnicity': 'string',
                                                  'ever_smoker': 'int',}, 
                                    datetime_columns=['index_datetime'])

phenotype_data = read_from_bucket(file_path, 
                                  f'{cohort_name}_phenotype_data', 
                                  column_types={'person_id': 'string', 
                                                'ICD_string': 'string', 
                                                'vocabulary_id': 'string', 
                                                'ICD': 'string', 
                                                'phenotype': 'string', 
                                                'phenotype_class': 'string'}, 
                                  datetime_columns=['condition_start_date'])

measurement_data = read_from_bucket(file_path, 
                                    f'{cohort_name}_measurement_data', 
                                    column_types={'person_id': 'string', 
                                                  'hgb': 'float64',
                                                  'mcv': 'float64',
                                                  'plt': 'float64',
                                                  'rdw': 'float64',
                                                  'wbc': 'float64', 
                                                  'neu': 'float64'}, 
                                    datetime_columns=['measurement_datetime'])

In [ ]:
measurement_data

## Generate myeloid phenotype table

In [ ]:
def get_first_phenotype(df, phenotype_list):
    # Filter rows where phenotype is in the input list
    filtered_df = df[df['phenotype'].isin(phenotype_list)]
    
    # Sort by person_id and condition_start_date
    filtered_df = filtered_df.sort_values(by=['person_id', 'condition_start_date'])
    
    # Create column 'first_aml_mds_mf' that is equal to the phenotype that occurs first for each person_id
    filtered_df['first_aml_mds_mf'] = filtered_df.groupby('person_id')['phenotype'].transform('first')
    
    # Create column 'aml_mds_mf_date' which is the condition_start_date of the first phenotype
    filtered_df['aml_mds_mf_date'] = filtered_df.groupby('person_id')['condition_start_date'].transform('first')
    
    # Drop duplicate rows to keep only one row per person_id
    result_df = filtered_df.drop_duplicates(subset=['person_id'])
    
    # Select and reorder the final columns
    result_df = result_df[['person_id', 'first_aml_mds_mf', 'aml_mds_mf_date']]
    
    return result_df


def get_first_diagnosis(df, phenotype_list, phenotype_class, phenotype_dict):
    # Filter, sort, group, and rename columns in one go
    return (df[df['phenotype'].isin(phenotype_list)]
            .sort_values(['person_id', 'condition_start_date'])
            .groupby('person_id', as_index=False)
            .first()
            .rename(columns={'condition_start_date': f'{phenotype_dict[phenotype_class]}_date'})
            [['person_id', 
              f'{phenotype_dict[phenotype_class]}_date']])

In [ ]:
phenotype_dict = {'Acute myeloid leukemia': 'aml', 
                  'Myelodysplastic syndrome': 'mds', 
                  'Myelofibrosis': 'mf',
                  'Essential thrombocythemia': 'et',
                  'Polycythemia vera': 'pv',
                  'Lymphoid Leukemias': 'cll'
                 }

aml_list = ['Acute myeloid leukemia', 'Acute myeloblastic leukemia', 
            'Acute myelomonocytic leukemia', 'Acute promyelocytic leukemia']

mds_list = ['Myelodysplastic syndrome', 'Refractory anemia', 'Refractory cytopenia']

aml_mds_mf_list = aml_list + mds_list + ['Myelofibrosis']

cll_list = ['Lymphoid Leukemias', 'Chronic and other myelogenous leukemia']

In [ ]:
aml = get_first_diagnosis(phenotype_data, aml_list, 'Acute myeloid leukemia', phenotype_dict)
mds = get_first_diagnosis(phenotype_data, mds_list, 'Myelodysplastic syndrome', phenotype_dict)
mf = get_first_diagnosis(phenotype_data, ['Myelofibrosis'], 'Myelofibrosis', phenotype_dict)
et = get_first_diagnosis(phenotype_data, ['Essential thrombocythemia'], 'Essential thrombocythemia', phenotype_dict)
pv = get_first_diagnosis(phenotype_data, ['Polycythemia vera'], 'Polycythemia vera', phenotype_dict)

aml_mds_mf = get_first_phenotype(phenotype_data, aml_mds_mf_list)

In [ ]:
cll = get_first_diagnosis(phenotype_data, cll_list, 'Lymphoid Leukemias', phenotype_dict)
cll_cmml = get_first_phenotype(phenotype_data, cll_list)
cll_cmml.columns = ['person_id', 'first_cll', 'cll_date']

In [ ]:
p_df = demographic_data[['person_id', 'index_datetime']]
phenotype_dfs = [aml, mds, mf, et, pv]

for df in phenotype_dfs:
    p_df = p_df.merge(df, on='person_id', how='left')
    
myeloid_phenotypes = p_df.merge(aml_mds_mf, on='person_id', how='left')

In [ ]:
p_df = demographic_data[['person_id', 'index_datetime']]
phenotype_dfs = [cll]

for df in phenotype_dfs:
    p_df = p_df.merge(df, on='person_id', how='left')
    
lymphoid_phenotypes = p_df.merge(cll_cmml, on=['person_id', 'cll_date'], how='left')

In [ ]:
assert check_column_types(myeloid_phenotypes, {'person_id': 'string', 
                                               'index_datetime': 'datetime64[ns, UTC]',
                                               'aml_date': 'datetime64[ns, UTC]',
                                               'mds_date': 'datetime64[ns, UTC]',
                                               'mf_date': 'datetime64[ns, UTC]',
                                               'et_date': 'datetime64[ns, UTC]',
                                               'pv_date': 'datetime64[ns, UTC]', 
                                               'first_aml_mds_mf': 'string', 
                                               'aml_mds_mf_date': 'datetime64[ns, UTC]'})

In [ ]:
assert check_column_types(lymphoid_phenotypes, {'person_id': 'string', 
                                               'index_datetime': 'datetime64[ns, UTC]',
                                               'cll_date': 'datetime64[ns, UTC]',
                                               'first_cll': 'string'})

In [ ]:
save_to_bucket(myeloid_phenotypes, file_path, f'{cohort_name}_myeloid_phenotypes')

In [ ]:
save_to_bucket(lymphoid_phenotypes, file_path, f'{cohort_name}_lymphoid_phenotypes')

## Generate condensed complete blood count data

Baseline complete blood count (CBC) data requirements to be eligible for the cohort are implemented in the function:

`filter_cbc_data(df, time_window, total_cbc, cbc_post_index, min_risk_time)`
- Have at least one CBC within +/- 365 days of sequencing blood draw (time_window == 365)
- Have at least 3 CBC draws (total_cbc == 3)
- Have at least 2 CBC draws after date of sequencing blood draw (cbc_post_index == 2)
- Have a final CBC that is at least 120 days after the enrollment CBC (min_risk_time = 120)

In [ ]:
def filter_cbc_data(df, demographics, time_window=365, total_cbc=3, cbc_post_index=2, min_risk_time=120):
    df = pd.merge(df, demographics[['person_id', 'gender', 'index_datetime']].drop_duplicates(), on='person_id', how='left')
    
    # Ensure datetime columns are in datetime format
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    df['index_datetime'] = pd.to_datetime(df['index_datetime'])
        
    # Compute the new column 'time_from_index'
    df['time_from_index'] = (df['measurement_datetime'] - df['index_datetime']).dt.days
    
    # Create list of person_id's with at least one row within 'time_window' days of index_datetime
    pid_index_criteria = df[df['time_from_index'].abs() <= time_window]['person_id'].unique()
    
    # Create list of person_id's with at least 'total_cbc' blood draws in the time range 
    # time_from_index > -time_window
    pid_total_draw_criteria = df[df['time_from_index'] > -time_window]['person_id'].value_counts()
    pid_total_draw_criteria = pid_total_draw_criteria[pid_total_draw_criteria >= total_cbc].index
    
    # Create list of person_id's with at least 'cbc_post_index' blood draws where time_from_index > 0
    pid_post_index_criteria = df[df['time_from_index'] > 0]['person_id'].value_counts()
    pid_post_index_criteria = pid_post_index_criteria[pid_post_index_criteria >= cbc_post_index].index
    
    # Find person_id's that are in all three lists
    pid_temp_matches = set(pid_index_criteria) & set(pid_total_draw_criteria) & set(pid_post_index_criteria)
    
    # Create the 'temp_eligible' column based on the criteria
    df['temp_eligible'] = 0
    df.loc[(df['person_id'].isin(pid_temp_matches)) & (df['time_from_index'] >= -time_window), 'temp_eligible'] = 1
    
    # Create list of person_id's where max(measurement_datetime) - min(measurement_datetime) in days > min_risk_time
    pid_risk_time_criteria = df[df['temp_eligible']==1].groupby('person_id').apply(lambda x: (x['measurement_datetime'].max() - x['measurement_datetime'].min()).days >= min_risk_time)
    pid_risk_time_criteria = pid_risk_time_criteria[pid_risk_time_criteria].index
    
    # Find final eligible person_id's
    pid_matches = pid_temp_matches & set(pid_risk_time_criteria)
    
    # Create the 'eligible' column based on the criteria
    df['eligible'] = 0
    df.loc[(df['person_id'].isin(pid_matches)) & (df['time_from_index'] >= -time_window), 'eligible'] = 1
    
    df_eligible = df[df['eligible']==1].drop('temp_eligible', axis=1)
    
    return df_eligible

def cbc_checker(df, time_window=365, total_cbc=3, cbc_post_index=2, min_risk_time=120):
    # Convert datetime columns to datetime type if they're not already
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    df['index_datetime'] = pd.to_datetime(df['index_datetime'])
    
    # Group by person_id
    grouped = df.groupby('person_id')
    
    # Test 1: At least one row within time_window days of index_datetime
    test1 = grouped.apply(lambda x: (np.abs(x['time_from_index'].min()) <= time_window).any())
    
    # Test 2: At least total_cbc rows of data
    test2 = grouped.size() >= total_cbc
    
    # Test 3: At least cbc_post_index rows with time_from_index >= 0
    test3 = grouped.apply(lambda x: (x['time_from_index'] >= 0).sum() >= cbc_post_index)
    
    # Test 4: At least min_risk_time days between max and min measurement_datetime
    test4 = grouped['measurement_datetime'].apply(lambda x: (x.max() - x.min()).days >= min_risk_time)
    
    # Combine all tests
    all_tests = test1 & test2 & test3 & test4
    
    if all_tests.all():
        return True
    else:
        return all_tests[~all_tests].index.tolist()

In [ ]:
cbc_filtered = filter_cbc_data(measurement_data, demographic_data, 365, 3, 2, 120)

cbc_test_result = cbc_checker(cbc_filtered[(cbc_filtered['eligible']==1)].copy(), time_window=365, total_cbc=3, 
                              cbc_post_index=2, min_risk_time=120)

if cbc_test_result is True:
    print("All person_ids passed the tests.")
else:
    print(f"The following person_ids failed the tests: {cbc_test_result}")

Label CBC data for persistent cytopenias in eligible person_ids: 

`label_persistent_cytopenia(df, anemia_cutoff_f, anemia_cutoff_m, leukopenia_cutoff, thrombocytopenia_cutoff)`


**Clonal Cytopenia of Undetermined Significance (CCUS)**
- One or more somatic mutations otherwise found in patients with myeloid neoplasms detected in bone marrow or peripheral blood cells with an allele burden of ≥ 2%
- Persistent cytopenia (≥ 4 months) in one or more peripheral blood cell lineages
- Diagnostic criteria of myeloid neoplasm not fulfilled
- All other causes of cytopenia and molecular aberration excluded

Source: https://ascopubs.org/doi/10.1200/EDBK_239083#box1

**Cytopenia definition per WHO:**
- Anemia: hemoglobin < 12.0 g/dL in females, < 13.0 g/dL in males
- Leukopenia/neutropenia: neutrophil count < 1.8 thousand cells/microliter; **we use leukocyte count < 3.7 thousand/microliter**
- Thrombocytopenia: platelet count < 150 thousand cells/microliter

In [ ]:
anemia_f = 12.0 
anemia_m = 13.0 
leukopenia = 3.7
thrombocytopenia = 150

In [ ]:
polycythemia_f = 16.5
polycythemia_m = 18.0
leukocytoses = 11
thrombocytoses = 450

In [ ]:
def annotate_cbc_data(df, anemia_f, anemia_m, leukopenia, thrombocytopenia, 
                      polycythemia_f, polycythemia_m, leukocytoses, thrombocytoses):
    
    # Check input df only has measurements where eligible == 1
    assert (df['eligible'] == 1).all()
    
    # Label persistent cytopenias for eligible rows
    df = label_persistent_cytopenia(df, anemia_f, anemia_m, leukopenia, thrombocytopenia)
    
    # Label persistent cytopenias for eligible rows
    df = label_persistent_cytoses(df, polycythemia_f, polycythemia_m, leukocytoses, thrombocytoses)
    
    # Calculate time from first measurement for eligible rows
    df = calculate_time_from_first_measurement(df)
    
    # Calculate CBC dates for eligible rows
    df = calculate_cbc_dates(df)
    
    # Calculate CBC counts for all rows and eligible rows
    df = calculate_cbc_counts(df)
    
    df = cytopenia_at_enrollment(df)
    
    df = cytoses_at_enrollment(df)
    
    df_condensed = condense_cbc_data(df)
    
    return df, df_condensed



def label_persistent_cytopenia(df, anemia_f, anemia_m, leukopenia, thrombocytopenia):
    
    # Sort by person_id and measurement_datetime
    df = df.sort_values(by=['person_id', 'measurement_datetime'])
    
    # Create anemia column based on gender-specific cutoffs
    df['anemia'] = 0
    df.loc[(df['gender'] == 'male') & (df['hgb'] > anemia_m), 'anemia'] = 1
    df.loc[(df['gender'] == 'female') & (df['hgb'] > anemia_f), 'anemia'] = 1
    df.loc[(df['gender'] == 'Other') & (df['hgb'] > anemia_f), 'anemia'] = 1
    
    # Create leukopenia column
    df['leukopenia'] = (df['wbc'] < leukopenia).astype(int)
    
    # Create thrombocytopenia column
    df['thrombocytopenia'] = (df['plt'] < thrombocytopenia).astype(int)
    
    # Create cytopenia column
    df['cytopenia'] = ((df['anemia'] == 1) | (df['leukopenia'] == 1) | (df['thrombocytopenia'] == 1)).astype(int)
    
    df = check_persistence(df, 'anemia')
    df = check_persistence(df, 'leukopenia')
    df = check_persistence(df, 'thrombocytopenia')

    # Create cytopenia column
    df['persistent_cytopenia'] = ((df['persistent_anemia'] == 1) | 
                                  (df['persistent_leukopenia'] == 1) | 
                                  (df['persistent_thrombocytopenia'] == 1)).astype(int)
    
    return df

def label_persistent_cytoses(df, polycythemia_f, polycythemia_m, leukocytoses, thrombocytoses):
    
    # Sort by person_id and measurement_datetime
    df = df.sort_values(by=['person_id', 'measurement_datetime'])
    
    # Create anemia column based on gender-specific cutoffs
    df['polycythemia'] = 0
    df.loc[(df['gender'] == 'male') & (df['hgb'] > polycythemia_m), 'polycythemia'] = 1
    df.loc[(df['gender'] == 'female') & (df['hgb'] > polycythemia_f), 'polycythemia'] = 1
    df.loc[(df['gender'] == 'Other') & (df['hgb'] > polycythemia_f), 'polycythemia'] = 1
    
    # Create leukopenia column
    df['leukocytoses'] = (df['wbc'] > leukocytoses).astype(int)
    
    # Create thrombocytopenia column
    df['thrombocytoses'] = (df['plt'] > thrombocytoses).astype(int)
    
    # Create cytopenia column
    df['cytoses'] = ((df['polycythemia'] == 1) | (df['leukocytoses'] == 1) | (df['thrombocytoses'] == 1)).astype(int)
    
    df = check_persistence(df, 'polycythemia')
    df = check_persistence(df, 'leukocytoses')
    df = check_persistence(df, 'thrombocytoses')

    # Create cytopenia column
    df['persistent_cytoses'] = ((df['persistent_polycythemia'] == 1) | 
                                  (df['persistent_leukocytoses'] == 1) | 
                                  (df['persistent_thrombocytoses'] == 1)).astype(int)
    
    return df


def check_persistence(df, cell_line):
    # Create the new column name
    persistent_column = f'persistent_{cell_line}'
    print(persistent_column)
    # Initialize the new column with 0
    df[persistent_column] = 0
    
    # Group by person_id
    grouped = df.groupby('person_id')
    
    # List to store modified dataframes
    dfs_to_concat = []
    
    # Iterate over each group
    for name, group in grouped:
        # Reset index for each group to handle row-wise operations
        group = group.reset_index(drop=True)
        
        # Initialize variables to track condition start and end indices
        start_index = None
        end_index = None
        found_persistent_condition = False
                
        for i, row in group.iterrows():
            if row[cell_line] == 1:
                if start_index is None:
                    # Mark the start of a potential persistent condition
                    start_index = i
                    end_index = i
                else:
                    # Check the duration of the persistent condition
                    if (row['time_from_index'] - group.loc[start_index, 'time_from_index']) >= 120:
                        # Mark the first row of the persistent condition
                        group.loc[start_index, persistent_column] = 1
                        dfs_to_concat.append(group)
                        found_persistent_condition = True
                        break
                    else:
                        # Extend the end index if the duration is not met yet
                        end_index = i
            else:
                # Reset indices if a row with condition == 0 is encountered
                start_index = None
                end_index = None
        
        # If no persistent cytopenia was found, append the entire group
        if not found_persistent_condition:
            dfs_to_concat.append(group)
    
    # Concatenate all modified groups into a single dataframe
    if dfs_to_concat:
        result_df = pd.concat(dfs_to_concat, ignore_index=True)
    else:
        result_df = pd.DataFrame(columns=df.columns)  # Return an empty dataframe if no persistent cytopenia found
    
    return result_df


def calculate_time_from_first_measurement(df):
    # Ensure the measurement_datetime column is in datetime format
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    
    # Initialize the new column with zero
    df['time_from_first_measurement'] = 0
    
    # Group by person_id to process each person's data separately
    grouped = df.groupby('person_id')
    
    # Iterate over each group
    for name, group in grouped:
        # Find the first measurement datetime for the current group
        first_measurement = group['measurement_datetime'].iloc[0]
        
        # Calculate the time difference for each row in the group
        time_difference = (group['measurement_datetime'] - first_measurement).dt.days
        
        # Assign this time difference to the appropriate rows in the original dataframe
        df.loc[group.index, 'time_from_first_measurement'] = time_difference
    
    return df


def calculate_cbc_counts(df):
    # Calculate cbc_count_total for the entire dataframe
    df['cbc_count'] = df.groupby('person_id')['person_id'].transform('size')
    
    # Calculate cbc_count_post_index where time_from_index > 0
    post_index_counts = df[df['time_from_index'] > 0].groupby('person_id').size().reset_index(name='cbc_count_post_index')
    
    # Merge back to ensure all person_id have an entry, even if count is zero
    df = pd.merge(df, post_index_counts, on='person_id', how='left')
    df['cbc_count_post_index'].fillna(0, inplace=True)  # Fill NaN with 0 where no rows meet the condition
    
    return df


def calculate_cbc_dates(df):
    # Ensure the measurement_datetime column is in datetime format
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    
    # Group by person_id to process each person's data separately
    grouped = df.groupby('person_id')
    
    # Calculate the first and last measurement_datetime for each person_id
    first_cbc_datetime = grouped['measurement_datetime'].min().reset_index(name='first_cbc_datetime')
    last_cbc_datetime = grouped['measurement_datetime'].max().reset_index(name='last_cbc_datetime')
    
    # Find the measurement_datetime where persistent_cytopenia == 1 for each person_id
    persistent_cytopenia_datetime = grouped.apply(lambda x: x.loc[x['persistent_cytopenia'] == 1, 'measurement_datetime'].min()).reset_index(name='persistent_cytopenia_datetime')

    # Find the measurement_datetime where persistent_cytopenia == 1 for each person_id
    persistent_cytoses_datetime = grouped.apply(lambda x: x.loc[x['persistent_cytoses'] == 1, 'measurement_datetime'].min()).reset_index(name='persistent_cytoses_datetime')

    # Merge these dates back into the original dataframe
    df = df.merge(first_cbc_datetime, on='person_id')
    df = df.merge(last_cbc_datetime, on='person_id')
    df = df.merge(persistent_cytopenia_datetime, on='person_id')
    df = df.merge(persistent_cytoses_datetime, on='person_id')

    return df


def cytopenia_at_enrollment(df):
    # Ensure datetime columns are in datetime format
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    df['index_datetime'] = pd.to_datetime(df['index_datetime'])
    df['persistent_cytopenia_datetime'] = pd.to_datetime(df['persistent_cytopenia_datetime'], errors='coerce')
    
    # Check input df only has measurements where eligible == 1
    assert (df['eligible'] == 1).all()
    
    # First criterion: persistent_cytopenia_datetime > index_datetime or persistent_cytopenia_datetime is NaT
    condition1 = (df['persistent_cytopenia_datetime'] > df['index_datetime']) | df['persistent_cytopenia_datetime'].isna()
    pid_condition1 = df[condition1]['person_id'].unique().tolist()
    
    # Second criterion: At least one CBC without cytopenia before persistent_cytopenia_datetime
    def has_cbc_without_cytopenia_before_persistent(group):
        if group['persistent_cytopenia_datetime'].iloc[0] is pd.NaT:
            return True
        cytopenia_before_persistent = group[group['measurement_datetime'] < group['persistent_cytopenia_datetime'].iloc[0]]
        return (cytopenia_before_persistent['cytopenia'] == 0).any()
    
    pid_condition2 = df.groupby('person_id').filter(has_cbc_without_cytopenia_before_persistent)['person_id'].unique().tolist()
    
    # Find person_ids that meet both criteria
    no_cytopenia_ids = list(set(pid_condition1) & set(pid_condition2))
    cytopenia_ids = list(set(df['person_id'].unique().tolist()) - set(no_cytopenia_ids))
    
    # Create the 'chip_at_enrollment' column based on the criteria
    # -1 for participants that are not eligible
    # 0 for participants without cytopenia at time of enrollment
    # 1 for participants with cytopenia at time of enrollment
    df['cytopenia_at_enrollment'] = -1 
    df.loc[(df['person_id'].isin(no_cytopenia_ids)), 'cytopenia_at_enrollment'] = 0
    df.loc[(df['person_id'].isin(cytopenia_ids)), 'cytopenia_at_enrollment'] = 1
    
    return df


def cytoses_at_enrollment(df):
    # Ensure datetime columns are in datetime format
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    df['index_datetime'] = pd.to_datetime(df['index_datetime'])
    df['persistent_cytoses_datetime'] = pd.to_datetime(df['persistent_cytoses_datetime'], errors='coerce')
    
    # Check input df only has measurements where eligible == 1
    assert (df['eligible'] == 1).all()
    
    # First criterion: persistent_cytopenia_datetime > index_datetime or persistent_cytopenia_datetime is NaT
    condition1 = (df['persistent_cytoses_datetime'] > df['index_datetime']) | df['persistent_cytoses_datetime'].isna()
    pid_condition1 = df[condition1]['person_id'].unique().tolist()
    
    # Second criterion: At least one CBC without cytopenia before persistent_cytopenia_datetime
    def has_cbc_without_cytoses_before_persistent(group):
        if group['persistent_cytoses_datetime'].iloc[0] is pd.NaT:
            return True
        cytoses_before_persistent = group[group['measurement_datetime'] < group['persistent_cytoses_datetime'].iloc[0]]
        return (cytoses_before_persistent['cytoses'] == 0).any()
    
    pid_condition2 = df.groupby('person_id').filter(has_cbc_without_cytoses_before_persistent)['person_id'].unique().tolist()
    
    # Find person_ids that meet both criteria
    no_cytoses_ids = list(set(pid_condition1) & set(pid_condition2))
    cytoses_ids = list(set(df['person_id'].unique().tolist()) - set(no_cytoses_ids))
    
    # Create the 'chip_at_enrollment' column based on the criteria
    # -1 for participants that are not eligible
    # 0 for participants without cytopenia at time of enrollment
    # 1 for participants with cytopenia at time of enrollment
    df['cytoses_at_enrollment'] = -1 
    df.loc[(df['person_id'].isin(no_cytoses_ids)), 'cytoses_at_enrollment'] = 0
    df.loc[(df['person_id'].isin(cytoses_ids)), 'cytoses_at_enrollment'] = 1
    
    return df


def condense_cbc_data(df):
    
    # Select the desired columns
    condensed_df = df[['person_id', 
                       'measurement_datetime',
                       'gender', 
                       'index_datetime',
                       'cytopenia_at_enrollment', 
                       'cytoses_at_enrollment']]

    # Define the aggregation dictionaries
    aggregation = {
        'hgb': 'first', 
        'wbc': 'first', 
        'plt': 'first',
        'mcv': 'first',
        'rdw': 'first',
        'persistent_anemia': 'max',
        'persistent_leukopenia': 'max',
        'persistent_thrombocytopenia': 'max',
        'persistent_cytopenia': 'max',
        'persistent_cytopenia_datetime': 'max',
        'persistent_polycythemia': 'max',
        'persistent_leukocytoses': 'max',
        'persistent_thrombocytoses': 'max',
        'persistent_cytoses': 'max',
        'persistent_cytoses_datetime': 'max',
        'first_cbc_datetime': 'first',
        'last_cbc_datetime': 'first',
        'cbc_count': 'first',
        'cbc_count_post_index': 'first'
    }
    
    # Group by 'person_id' and apply the aggregation
    aggregated = df.groupby('person_id').agg(aggregation).reset_index()
    
    # Merge the aggregated DataFrame with the filtered DataFrame based on 'person_id'
    condensed_df = pd.merge(condensed_df, aggregated, on='person_id', how='left')

    # Drop the 'measurement_datetime' column
    condensed_df.drop(columns=['measurement_datetime'], inplace=True)

    # Drop duplicate rows
    condensed_df = condensed_df.drop_duplicates().reset_index(drop=True)

    return condensed_df

In [ ]:
# def annotate_cbc_data(df, anemia_f, anemia_m, leukopenia, thrombocytopenia):
    
#     # Check input df only has measurements where eligible == 1
#     assert (df['eligible'] == 1).all()
    
#     # Label persistent cytopenias for eligible rows
#     df = label_persistent_cytopenia(df, anemia_f, anemia_m, leukopenia, thrombocytopenia)
    
#     # Calculate time from first measurement for eligible rows
#     df = calculate_time_from_first_measurement(df)
    
#     # Calculate CBC dates for eligible rows
#     df = calculate_cbc_dates(df)
    
#     # Calculate CBC counts for all rows and eligible rows
#     df = calculate_cbc_counts(df)
    
#     df = cytopenia_at_enrollment(df)
    
#     df_condensed = condense_cbc_data(df)
    
#     return df, df_condensed


# def label_persistent_cytopenia(df, anemia_f, anemia_m, leukopenia, thrombocytopenia):
    
#     # Sort by person_id and measurement_datetime
#     df = df.sort_values(by=['person_id', 'measurement_datetime'])
    
#     # Create anemia column based on gender-specific cutoffs
#     df['anemia'] = 0
#     df.loc[(df['gender'] == 'male') & (df['hgb'] < anemia_m), 'anemia'] = 1
#     df.loc[(df['gender'] == 'female') & (df['hgb'] < anemia_f), 'anemia'] = 1
#     df.loc[(df['gender'] == 'Other') & (df['hgb'] < anemia_f), 'anemia'] = 1
    
#     # Create leukopenia column
#     df['leukopenia'] = (df['wbc'] < leukopenia).astype(int)
    
#     # Create thrombocytopenia column
#     df['thrombocytopenia'] = (df['plt'] < thrombocytopenia).astype(int)
    
#     # Create cytopenia column
#     df['cytopenia'] = ((df['anemia'] == 1) | (df['leukopenia'] == 1) | (df['thrombocytopenia'] == 1)).astype(int)
    
#     df = check_persistence(df, 'anemia')
#     df = check_persistence(df, 'leukopenia')
#     df = check_persistence(df, 'thrombocytopenia')

#     # Create cytopenia column
#     df['persistent_cytopenia'] = ((df['persistent_anemia'] == 1) | 
#                                   (df['persistent_leukopenia'] == 1) | 
#                                   (df['persistent_thrombocytopenia'] == 1)).astype(int)
    
#     return df


# def check_persistence(df, cell_line):
#     # Create the new column name
#     persistent_column = f'persistent_{cell_line}'
    
#     # Initialize the new column with 0
#     df[persistent_column] = 0
    
#     # Group by person_id
#     grouped = df.groupby('person_id')
    
#     # List to store modified dataframes
#     dfs_to_concat = []
    
#     # Iterate over each group
#     for name, group in grouped:
#         # Reset index for each group to handle row-wise operations
#         group = group.reset_index(drop=True)
        
#         # Initialize variables to track condition start and end indices
#         start_index = None
#         end_index = None
#         found_persistent_condition = False
                
#         for i, row in group.iterrows():
#             if row[cell_line] == 1:
#                 if start_index is None:
#                     # Mark the start of a potential persistent condition
#                     start_index = i
#                     end_index = i
#                 else:
#                     # Check the duration of the persistent condition
#                     if (row['time_from_index'] - group.loc[start_index, 'time_from_index']) >= 120:
#                         # Mark the first row of the persistent condition
#                         group.loc[start_index, persistent_column] = 1
#                         dfs_to_concat.append(group)
#                         found_persistent_condition = True
#                         break
#                     else:
#                         # Extend the end index if the duration is not met yet
#                         end_index = i
#             else:
#                 # Reset indices if a row with condition == 0 is encountered
#                 start_index = None
#                 end_index = None
        
#         # If no persistent cytopenia was found, append the entire group
#         if not found_persistent_condition:
#             dfs_to_concat.append(group)
    
#     # Concatenate all modified groups into a single dataframe
#     if dfs_to_concat:
#         result_df = pd.concat(dfs_to_concat, ignore_index=True)
#     else:
#         result_df = pd.DataFrame(columns=df.columns)  # Return an empty dataframe if no persistent cytopenia found
    
#     return result_df


# def calculate_time_from_first_measurement(df):
#     # Ensure the measurement_datetime column is in datetime format
#     df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    
#     # Initialize the new column with zero
#     df['time_from_first_measurement'] = 0
    
#     # Group by person_id to process each person's data separately
#     grouped = df.groupby('person_id')
    
#     # Iterate over each group
#     for name, group in grouped:
#         # Find the first measurement datetime for the current group
#         first_measurement = group['measurement_datetime'].iloc[0]
        
#         # Calculate the time difference for each row in the group
#         time_difference = (group['measurement_datetime'] - first_measurement).dt.days
        
#         # Assign this time difference to the appropriate rows in the original dataframe
#         df.loc[group.index, 'time_from_first_measurement'] = time_difference
    
#     return df


# def calculate_cbc_counts(df):
#     # Calculate cbc_count_total for the entire dataframe
#     df['cbc_count'] = df.groupby('person_id')['person_id'].transform('size')
    
#     # Calculate cbc_count_post_index where time_from_index > 0
#     post_index_counts = df[df['time_from_index'] > 0].groupby('person_id').size().reset_index(name='cbc_count_post_index')
    
#     # Merge back to ensure all person_id have an entry, even if count is zero
#     df = pd.merge(df, post_index_counts, on='person_id', how='left')
#     df['cbc_count_post_index'].fillna(0, inplace=True)  # Fill NaN with 0 where no rows meet the condition
    
#     return df


# def calculate_cbc_dates(df):
#     # Ensure the measurement_datetime column is in datetime format
#     df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    
#     # Group by person_id to process each person's data separately
#     grouped = df.groupby('person_id')
    
#     # Calculate the first and last measurement_datetime for each person_id
#     first_cbc_datetime = grouped['measurement_datetime'].min().reset_index(name='first_cbc_datetime')
#     last_cbc_datetime = grouped['measurement_datetime'].max().reset_index(name='last_cbc_datetime')
    
#     # Find the measurement_datetime where persistent_cytopenia == 1 for each person_id
#     persistent_cytopenia_datetime = grouped.apply(lambda x: x.loc[x['persistent_cytopenia'] == 1, 'measurement_datetime'].min()).reset_index(name='persistent_cytopenia_datetime')
    
#     # Merge these dates back into the original dataframe
#     df = df.merge(first_cbc_datetime, on='person_id')
#     df = df.merge(last_cbc_datetime, on='person_id')
#     df = df.merge(persistent_cytopenia_datetime, on='person_id')
    
#     return df


# def cytopenia_at_enrollment(df):
#     # Ensure datetime columns are in datetime format
#     df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
#     df['index_datetime'] = pd.to_datetime(df['index_datetime'])
#     df['persistent_cytopenia_datetime'] = pd.to_datetime(df['persistent_cytopenia_datetime'], errors='coerce')
    
#     # Check input df only has measurements where eligible == 1
#     assert (df['eligible'] == 1).all()
    
#     # First criterion: persistent_cytopenia_datetime > index_datetime or persistent_cytopenia_datetime is NaT
#     condition1 = (df['persistent_cytopenia_datetime'] > df['index_datetime']) | df['persistent_cytopenia_datetime'].isna()
#     pid_condition1 = df[condition1]['person_id'].unique().tolist()
    
#     # Second criterion: At least one CBC without cytopenia before persistent_cytopenia_datetime
#     def has_cbc_without_cytopenia_before_persistent(group):
#         if group['persistent_cytopenia_datetime'].iloc[0] is pd.NaT:
#             return True
#         cytopenia_before_persistent = group[group['measurement_datetime'] < group['persistent_cytopenia_datetime'].iloc[0]]
#         return (cytopenia_before_persistent['cytopenia'] == 0).any()
    
#     pid_condition2 = df.groupby('person_id').filter(has_cbc_without_cytopenia_before_persistent)['person_id'].unique().tolist()
    
#     # Find person_ids that meet both criteria
#     no_cytopenia_ids = list(set(pid_condition1) & set(pid_condition2))
#     cytopenia_ids = list(set(df['person_id'].unique().tolist()) - set(no_cytopenia_ids))
    
#     # Create the 'chip_at_enrollment' column based on the criteria
#     # -1 for participants that are not eligible
#     # 0 for participants without cytopenia at time of enrollment
#     # 1 for participants with cytopenia at time of enrollment
#     df['cytopenia_at_enrollment'] = -1 
#     df.loc[(df['person_id'].isin(no_cytopenia_ids)), 'cytopenia_at_enrollment'] = 0
#     df.loc[(df['person_id'].isin(cytopenia_ids)), 'cytopenia_at_enrollment'] = 1
    
#     return df


# def condense_cbc_data(df):
    
#     # Select the desired columns
#     condensed_df = df[['person_id', 'measurement_datetime', 'gender', 'index_datetime', 'cytopenia_at_enrollment']]

#     # Define the aggregation dictionaries
#     aggregation = {
#         'hgb': 'first', 
#         'wbc': 'first', 
#         'plt': 'first',
#         'mcv': 'first',
#         'rdw': 'first',
#         'persistent_anemia': 'max',
#         'persistent_leukopenia': 'max',
#         'persistent_thrombocytopenia': 'max',
#         'persistent_cytopenia': 'max',
#         'persistent_cytopenia_datetime': 'max',
#         'first_cbc_datetime': 'first',
#         'last_cbc_datetime': 'first',
#         'cbc_count': 'first',
#         'cbc_count_post_index': 'first'
#     }

#     # Group by 'person_id' and apply the aggregation
#     aggregated = df.groupby('person_id').agg(aggregation).reset_index()
    
#     # Merge the aggregated DataFrame with the filtered DataFrame based on 'person_id'
#     condensed_df = pd.merge(condensed_df, aggregated, on='person_id', how='left')

#     # Drop the 'measurement_datetime' column
#     condensed_df.drop(columns=['measurement_datetime'], inplace=True)

#     # Drop duplicate rows
#     condensed_df = condensed_df.drop_duplicates().reset_index(drop=True)

#     return condensed_df

**CBC Annotations**

In [ ]:
# cbc_timeseries, cbc_condensed = annotate_cbc_data(cbc_filtered, anemia_f, anemia_m, leukopenia, thrombocytopenia)
# cbc_condensed.drop(columns=['gender', 'index_datetime'], inplace=True)

In [ ]:
# cbc_timeseries['time_from_index'] = cbc_timeseries['time_from_index'].astype('int')

In [ ]:
cbc_timeseries, cbc_condensed = annotate_cbc_data(cbc_filtered, 
                                                  anemia_f, anemia_m, leukopenia, thrombocytopenia,
                                                  polycythemia_f, polycythemia_m, leukocytoses, thrombocytoses)
cbc_condensed.drop(columns=['gender', 'index_datetime'], inplace=True)
cbc_timeseries['time_from_index'] = cbc_timeseries['time_from_index'].astype(int)

In [ ]:
# Create persistent abnormal blood counts column
cbc_condensed['persistent_abnormal_blood_counts'] = ((cbc_condensed['persistent_cytopenia'] == 1) | 
                                                    (cbc_condensed['persistent_cytoses'] == 1)).astype(int)

# Create datetime column by taking the earlier date between cytopenia and cytoses
cbc_condensed['persistent_abnormal_counts_datetime'] = pd.concat([
    cbc_condensed['persistent_cytopenia_datetime'],
    cbc_condensed['persistent_cytoses_datetime']
]).groupby(level=0).min()

In [ ]:
cbc_timeseries = cbc_timeseries[['person_id', 'measurement_datetime', 'hgb', 'wbc', 'mcv',
       'plt', 'rdw', 'gender', 'index_datetime', 'time_from_index', 'eligible',
       'anemia', 'leukopenia', 'thrombocytopenia', 'cytopenia',
       'persistent_anemia', 'persistent_leukopenia',
       'persistent_thrombocytopenia', 'persistent_cytopenia', 'polycythemia',
       'leukocytoses', 'thrombocytoses', 'cytoses', 'persistent_polycythemia',
       'persistent_leukocytoses', 'persistent_thrombocytoses',
       'persistent_cytoses', 'time_from_first_measurement',
       'first_cbc_datetime', 'last_cbc_datetime',
       'persistent_cytopenia_datetime', 'persistent_cytoses_datetime',
       'cbc_count', 'cbc_count_post_index', 'cytopenia_at_enrollment',
       'cytoses_at_enrollment']]

In [ ]:
assert check_column_types(cbc_timeseries, {'person_id': 'string', 
                                           'measurement_datetime': 'datetime64[ns, UTC]',
                                           'hgb': 'float64',
                                           'wbc': 'float64',
                                           'mcv': 'float64',
                                           'plt': 'float64',
                                           'rdw': 'float64', 
                                           'gender': 'string',
                                           'index_datetime': 'datetime64[ns, UTC]',
                                           'time_from_index': 'int',
                                           'eligible': 'int',
                                           'anemia': 'int',
                                           'leukopenia': 'int',
                                           'thrombocytopenia': 'int',
                                           'cytopenia': 'int',
                                           'persistent_anemia': 'int',
                                           'persistent_leukopenia': 'int',
                                           'persistent_thrombocytopenia': 'int',
                                           'persistent_cytopenia': 'int',
                                           'polycythemia': 'int',
                                           'leukocytoses': 'int',
                                           'thrombocytoses': 'int',
                                           'cytoses': 'int',
                                           'persistent_polycythemia': 'int',
                                           'persistent_leukocytoses': 'int', 
                                           'persistent_thrombocytoses': 'int',
                                           'persistent_cytoses': 'int',
                                           'time_from_first_measurement': 'int',
                                           'first_cbc_datetime': 'datetime64[ns, UTC]',
                                           'last_cbc_datetime': 'datetime64[ns, UTC]',
                                           'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
                                           'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
                                           'cbc_count': 'int',
                                           'cbc_count_post_index': 'int', 
                                           'cytopenia_at_enrollment': 'int',
                                           'cytoses_at_enrollment': 'int'})

assert check_column_types(cbc_condensed, {'person_id': 'string', 
                                          'cytopenia_at_enrollment': 'int',
                                          'cytoses_at_enrollment': 'int',
                                          'hgb': 'float64',
                                          'wbc': 'float64',
                                          'plt': 'float64',
                                          'mcv': 'float64',
                                          'rdw': 'float64',
                                          'persistent_anemia': 'int',
                                          'persistent_leukopenia': 'int',
                                          'persistent_thrombocytopenia': 'int',
                                          'persistent_cytopenia': 'int',
                                          'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
                                          'persistent_polycythemia': 'int',
                                          'persistent_leukocytoses': 'int',
                                          'persistent_thrombocytoses': 'int',
                                          'persistent_cytoses': 'int',
                                          'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
                                          'first_cbc_datetime': 'datetime64[ns, UTC]',
                                          'last_cbc_datetime': 'datetime64[ns, UTC]',
                                          'cbc_count': 'int',
                                          'cbc_count_post_index': 'int', 
                                          'persistent_abnormal_blood_counts': 'int',
                                          'persistent_abnormal_counts_datetime': 'datetime64[ns, UTC]'})

In [ ]:
save_to_bucket(cbc_timeseries, file_path, f'{cohort_name}_cbc_timeseries')
save_to_bucket(cbc_condensed, file_path, f'{cohort_name}_cbc_condensed')

# Assemble cohort

## Screen cohort

In [ ]:
demographic_data = read_from_bucket(file_path, 
                                    f'{cohort_name}_demographic_data', 
                                    column_types={'person_id': 'string', 
                                                  'gender': 'string',
                                                  'race': 'string',
                                                  'ethnicity': 'string',
                                                  'ever_smoker': 'int',}, 
                                    datetime_columns=['index_datetime', 
                                                      'birth_datetime', 
                                                      'death_date'])

genetic_data = read_from_bucket(file_path, 
                                f'{cohort_name}_genetic_data', 
                                column_types={'person_id': 'string', 
                                              'gene': 'string', 
                                              'VAF': 'float64'})

myeloid_phenotypes = read_from_bucket(file_path, 
                                      f'{cohort_name}_myeloid_phenotypes', 
                                      column_types={'person_id': 'string', 
                                                    'first_aml_mds_mf': 'string'}, 
                                      datetime_columns=['index_datetime', 
                                                        'aml_mds_mf_date', 
                                                        'aml_date', 
                                                        'mds_date', 
                                                        'mf_date', 
                                                        'et_date', 
                                                        'pv_date'])


cbc_condensed = read_from_bucket(file_path, 
                                 f'{cohort_name}_cbc_condensed', 
                                 column_types={'person_id': 'string', 
                                               'cytopenia_at_enrollment': 'int',
                                               'cytoses_at_enrollment': 'int',
                                               'hgb': 'float64',
                                               'wbc': 'float64',
                                               'plt': 'float64',
                                               'mcv': 'float64',
                                               'rdw': 'float64',
                                               'persistent_anemia': 'int',
                                               'persistent_leukopenia': 'int',
                                               'persistent_thrombocytopenia': 'int',
                                               'persistent_cytopenia': 'int',
                                               'persistent_polycythemia': 'int',
                                               'persistent_leukocytoses': 'int',
                                               'persistent_thrombocytoses': 'int',
                                               'persistent_cytoses': 'int',
                                               'persistent_abnormal_blood_counts': 'int',
                                               'cbc_count': 'int',
                                               'cbc_count_post_index': 'int'}, 
                                 datetime_columns=['persistent_cytopenia_datetime', 
                                                   'first_cbc_datetime', 
                                                   'last_cbc_datetime', 'persistent_abnormal_counts_datetime'])

In [ ]:
lymphoid_phenotypes['index_datetime'] = lymphoid_phenotypes['index_datetime'].dt.tz_localize(None)
lymphoid_phenotypes['cll_date'] = lymphoid_phenotypes['cll_date'].dt.tz_localize(None)
lymphoid_phenotypes['prior_cll'] = (lymphoid_phenotypes['index_datetime'] + pd.DateOffset(months=offset) > lymphoid_phenotypes['cll_date']).astype(int)

In [ ]:
demographic_data['age'] = ((demographic_data['index_datetime'] - demographic_data['birth_datetime']).dt.days / 365.25).round(1)
d = demographic_data[['person_id', 'birth_datetime', 'gender', 'race', 'ethnicity', 'ever_smoker', 'age']].copy()
d = d[d['age'] >= 18]

genetic_data['mca'] = (genetic_data['gene'].notna()).astype(int)
genetic_data.drop_duplicates(subset='person_id', inplace=True)
g = genetic_data[['person_id', 'mca']].copy()

c = cbc_condensed[['person_id', 'cytopenia_at_enrollment', 'cytoses_at_enrollment', 'cbc_count', 'cbc_count_post_index']].copy()

myeloid_phenotypes['prior_aml_mds_mf'] = (myeloid_phenotypes['index_datetime'] + pd.DateOffset(months=offset) > myeloid_phenotypes['aml_mds_mf_date']).astype(int)
mp = myeloid_phenotypes[['person_id', 'prior_aml_mds_mf']].copy()

lymphoid_phenotypes['prior_cll'] = (lymphoid_phenotypes['index_datetime'] + pd.DateOffset(months=offset) > lymphoid_phenotypes['cll_date']).astype(int)
lp = lymphoid_phenotypes[['person_id', 'prior_cll']].copy()

participants = d.merge(g, on='person_id', how='left')
participants = participants.merge(c, on='person_id', how='left')
participants = participants.merge(mp, on='person_id', how='left')
participants = participants.merge(lp, on='person_id', how='left')

participants['cytopenia_at_enrollment'].fillna(-1, inplace=True)
participants['cytopenia_at_enrollment'] = participants['cytopenia_at_enrollment'].astype(int)

participants['cytoses_at_enrollment'].fillna(-1, inplace=True)
participants['cytoses_at_enrollment'] = participants['cytopenia_at_enrollment'].astype(int)

participants['cbc_count'].fillna(-1, inplace=True)
participants['cbc_count'] = participants['cbc_count'].astype(int)

participants['cbc_count_post_index'].fillna(-1, inplace=True)
participants['cbc_count_post_index'] = participants['cbc_count_post_index'].astype(int)

participants['mca'].fillna(0, inplace=True)
participants['mca'] = participants['mca'].astype(int)

In [ ]:
# Label the participants who are eligible for survival analysis as 1, all else as 0
survival_mask = (participants['cbc_count'] >= 3) & \
                (participants['cbc_count_post_index'] >= 1) & \
                (participants['cytopenia_at_enrollment'] == 0) & \
                (participants['cytoses_at_enrollment'] == 0) & \
                (participants['prior_aml_mds_mf'] == 0)

# Apply the mask to set the 'survival' column
participants['eligible'] = np.where(survival_mask, 1, 0)

We then assign cases and controls:
- Screen participants for eligibility to become a case or control
- Match based on *control variables* with *max_controls* matches and a *max_age_difference*

In [ ]:
def find_matches(df, case_column, control_variables, max_controls, max_age_difference):
    # Select only necessary columns
    columns_needed = control_variables + ['person_id', 'age', case_column]
    df_subset = df[columns_needed].copy()
    
    # Separate cases and controls based on case_column
    cases = df_subset[df_subset[case_column] == 1].copy()
    controls = df_subset[df_subset[case_column] == 0].copy()
        
    # Round age to the nearest year
    cases['rounded_age'] = cases['age'].floordiv(1)
    controls['rounded_age'] = controls['age'].floordiv(1)
        
    # Merge cases with controls on control variables and age within max_age_difference
    merged_df = pd.merge(cases, controls, on=control_variables, suffixes=('_case', '_control'))
    merged_df = merged_df[abs(merged_df['age_case'] - merged_df['age_control']) <= max_age_difference]
        
    # Group by case and aggregate control ids into lists
    grouped_df = merged_df.groupby('person_id_case')['person_id_control'].apply(list).reset_index(name='control_list')
    
    # Shuffle the control ids within each case
    grouped_df['control_list'] = grouped_df['control_list'].apply(lambda x: random.sample(x, len(x)))
    
    # Rename columns
    grouped_df.rename(columns={'person_id_case': 'case'}, inplace=True)
        
    # Explode control_list into separate rows
    exploded_rows = []
        
    for idx, row in grouped_df.iterrows():
        for i, control_id in enumerate(row['control_list'][:max_controls], start=1):
            exploded_rows.append([row['case'], control_id, i])
    exploded_df = pd.DataFrame(exploded_rows, columns=['case', 'control_id', 'count'])
    
    # Pivot to reshape the dataframe
    matched_df = exploded_df.pivot(index='case', columns='count', values='control_id').reset_index()
    matched_df.columns = ['case'] + [f'control_{i}' for i in range(1, matched_df.shape[1])]
    
    return matched_df


def reshape_matches(df):
    # Initialize an empty list to store rows
    rows = []
    
    # Iterate through each row in the DataFrame
    for index, row in df.iterrows():
        case_person_id = row['case']
        # Add 'case' row with person_id as int
        rows.append(['case', int(case_person_id)])
        
        # Iterate through control columns
        for col in df.columns[df.columns.str.startswith('control_')]:
            control_person_id = row[col]
            if pd.notna(control_person_id):  # Check if control value is not NaN
                # Add 'control' row with person_id as int
                rows.append(['control', int(control_person_id)])
    
    # Create a new DataFrame from the rows list
    reshaped_df = pd.DataFrame(rows, columns=['status', 'person_id']).astype({'person_id': 'string'})
    
    # Create 'case' column where 1 if status is 'case', else 0 if status is 'control'
    reshaped_df['case'] = reshaped_df['status'].apply(lambda x: 1 if x == 'case' else 0)
    
    return reshaped_df


def match_participants(df, case_column, control_variables, max_controls, max_age_difference):
    # Select only necessary columns for matching
    columns_needed = control_variables + ['person_id', 'age', case_column]
    eligible_participants = df[df['eligible']==1][columns_needed].copy()
    
    # Find matches and get the matched DataFrame
    matched_df = find_matches(eligible_participants, case_column, control_variables, max_controls, max_age_difference)
        
    matches = reshape_matches(matched_df)
    
    # Merge only the necessary columns back with the original dataframe
    df = pd.merge(df, matches[['person_id', 'case']], on='person_id', how='left')
    
    # Fill NaN values in 'case' column with -1
    df['case'] = df['case'].fillna(-1)
    df['case'] = df['case'].astype(int)
    
    df = df.drop_duplicates()
    
    return df

In [ ]:
participants = match_participants(participants, 'mca', control_variables, max_controls, max_age_difference)

In [ ]:
assert check_column_types(participants, {'person_id': 'string', 
                                         'birth_datetime': 'datetime64[ns, UTC]',
                                         'gender': 'string',
                                         'race': 'string',
                                         'ethnicity': 'string',
                                         'ever_smoker': 'int',
                                         'age': 'float64',
                                         'mca': 'int',
                                         'cytopenia_at_enrollment': 'int',
                                         'cytoses_at_enrollment': 'int',
                                         'cbc_count': 'int',
                                         'cbc_count_post_index': 'int', 
                                         'prior_aml_mds_mf': 'int', 
                                         'prior_cll': 'int', 
                                         'eligible': 'int', 
                                         'case': 'int'})

In [ ]:
save_to_bucket(participants, file_path, f'{cohort_name}_participants')

## Build cohort

In [ ]:
participants = read_from_bucket(file_path, 
                                f'{cohort_name}_participants', 
                                column_types={'person_id': 'string', 
                                              'gender': 'string',
                                              'race': 'string',
                                              'ethnicity': 'string',
                                              'ever_smoker': 'int',
                                              'age': 'float64',
                                              'mca': 'int',
                                              'cytopenia_at_enrollment': 'int',
                                              'cytoses_at_enrollment': 'int',
                                              'cbc_count': 'int',
                                              'cbc_count_post_index': 'int', 
                                              'prior_aml_mds_mf': 'int', 
                                              'prior_cll': 'int',
                                              'eligible': 'int', 
                                              'case': 'int'}, 
                                datetime_columns=['birth_datetime'])

demographic_data = read_from_bucket(file_path, 
                                    f'{cohort_name}_demographic_data', 
                                    column_types={'person_id': 'string', 
                                                  'gender': 'string',
                                                  'race': 'string',
                                                  'ethnicity': 'string',
                                                  'ever_smoker': 'int',}, 
                                    datetime_columns=['index_datetime', 
                                                      'birth_datetime', 
                                                      'death_date'])

genetic_data = read_from_bucket(file_path, 
                                f'{cohort_name}_genetic_data', 
                                column_types={'person_id': 'string', 
                                              'gene': 'string', 
                                              'VAF': 'float64'})

myeloid_phenotypes = read_from_bucket(file_path, 
                                      f'{cohort_name}_myeloid_phenotypes', 
                                      column_types={'person_id': 'string', 
                                                    'first_aml_mds_mf': 'string'}, 
                                      datetime_columns=['index_datetime', 
                                                        'aml_mds_mf_date', 
                                                        'aml_date', 
                                                        'mds_date', 
                                                        'mf_date', 
                                                        'et_date', 
                                                        'pv_date'])

lymphoid_phenotypes = read_from_bucket(file_path, 
                                      f'{cohort_name}_lymphoid_phenotypes', 
                                      column_types={'person_id': 'string', 
                                                    'first_cll': 'string'}, 
                                      datetime_columns=['index_datetime', 
                                                        'cll_date'])

cbc_condensed = read_from_bucket(file_path, 
                                 f'{cohort_name}_cbc_condensed', 
                                 column_types={'person_id': 'string', 
                                               'cytopenia_at_enrollment': 'int',
                                               'cytoses_at_enrollment': 'int',
                                               'hgb': 'float64',
                                               'wbc': 'float64',
                                               'plt': 'float64',
                                               'mcv': 'float64',
                                               'rdw': 'float64',
                                               'persistent_anemia': 'int',
                                               'persistent_leukopenia': 'int',
                                               'persistent_thrombocytopenia': 'int',
                                               'persistent_cytopenia': 'int',
                                               'persistent_polycythemia': 'int',
                                               'persistent_leukocytoses': 'int',
                                               'persistent_thrombocytoses': 'int',
                                               'persistent_cytoses': 'int',
                                               'persistent_abnormal_blood_counts': 'int',
                                               'cbc_count': 'int',
                                               'cbc_count_post_index': 'int'}, 
                                 datetime_columns=['persistent_cytopenia_datetime', 
                                                   'persistent_cytoses_datetime',
                                                   'first_cbc_datetime', 
                                                   'last_cbc_datetime', 
                                                   'persistent_abnormal_counts_datetime'])

In [ ]:
p_ids = participants[participants['case'].isin([0, 1])]['person_id'].to_list()

p = participants[['person_id', 'age', 'mca', 'prior_aml_mds_mf', 'prior_cll', 'case']]
p_d = demographic_data[demographic_data['person_id'].isin(p_ids)]
p_g = genetic_data[genetic_data['person_id'].isin(p_ids)]
p_mp = myeloid_phenotypes[myeloid_phenotypes['person_id'].isin(p_ids)].drop(columns=['index_datetime'])
p_lp = lymphoid_phenotypes[lymphoid_phenotypes['person_id'].isin(p_ids)].drop(columns=['index_datetime'])
p_cbc = cbc_condensed[cbc_condensed['person_id'].isin(p_ids)]

In [ ]:
c = p_d.merge(p, on='person_id', how='left')
c = c.merge(p_g, on='person_id', how='left')
c = c.merge(p_cbc, on='person_id', how='left')
c = c.merge(p_mp, on='person_id', how='left')
c = c.merge(p_lp, on='person_id', how='left')

In [ ]:
c['cll_date'] = c['cll_date'].dt.tz_localize('UTC')

In [ ]:
assert check_column_types(c, {'person_id': 'string',
                       'birth_datetime': 'datetime64[ns, UTC]',
                       'death_date': 'datetime64[ns, UTC]',
                       'index_datetime': 'datetime64[ns, UTC]', 
                       'gender': 'string',
                       'race': 'string',
                       'ethnicity': 'string',
                       'ever_smoker': 'int',
                       'age': 'float64',
                       'mca': 'int',
                       'prior_aml_mds_mf': 'int',
                       'prior_cll': 'int',
                       'case': 'int',
                       'gene': 'string',
                       'VAF': 'float64',
                       'cytopenia_at_enrollment': 'int',
                       'cytoses_at_enrollment': 'int',
                       'hgb': 'float64',
                       'wbc': 'float64',
                       'plt': 'float64',
                       'mcv': 'float64',
                       'rdw': 'float64',
                       'persistent_anemia': 'int',
                       'persistent_leukopenia': 'int',
                       'persistent_thrombocytopenia': 'int',
                       'persistent_cytopenia': 'int',
                       'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
                       'persistent_polycythemia': 'int',
                       'persistent_leukocytoses': 'int',
                       'persistent_thrombocytoses': 'int',
                       'persistent_cytoses': 'int',
                       'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
                       'first_cbc_datetime': 'datetime64[ns, UTC]',
                       'last_cbc_datetime': 'datetime64[ns, UTC]',
                       'cbc_count': 'int',
                       'cbc_count_post_index': 'int',
                       'persistent_abnormal_blood_counts': 'int',
                       'persistent_abnormal_counts_datetime': 'datetime64[ns, UTC]',
                       'aml_date': 'datetime64[ns, UTC]',
                       'mds_date': 'datetime64[ns, UTC]',
                       'mf_date': 'datetime64[ns, UTC]',
                       'et_date': 'datetime64[ns, UTC]',
                       'pv_date': 'datetime64[ns, UTC]', 
                       'first_aml_mds_mf': 'string', 
                       'aml_mds_mf_date': 'datetime64[ns, UTC]',
                        'first_cll': 'string',
                        'cll_date': 'datetime64[ns, UTC]'})

In [ ]:
save_to_bucket(c, file_path, f'{cohort_name}_cohort')

# Analysis

In [ ]:
c = read_from_bucket(file_path, f'{cohort_name}_cohort')

In [ ]:
c.columns

In [ ]:
# # Function to determine gene_class for each group
# def determine_gene_class(genes):
    
#     # Define gene types
#     gene_types = ['DNMT3A', 'TET2', 'ASXL1', 'JAK2']
#     tp53_ppm1d = ['TP53', 'PPM1D']

#     if genes.isna().all():
#         return 'reference'
#     unique_genes = genes.dropna().unique()
#     if len(unique_genes) == 0:
#         return 'reference'
#     elif len(unique_genes) > 1:
#         return 'Multiple'
#     else:
#         gene = unique_genes[0]
#         if gene in gene_types:
#             return gene
#         elif gene in tp53_ppm1d:
#             return 'TP53_PPM1D'
#         else:
#             return 'Other'

# Function to determine gene_class for each group
def determine_gene_class(genes):
    
    # Define gene types
    high_risk_mcas = ['chr6 Loss', 'chr6q Loss', 
                      'chr11 Loss', 'chr11q Loss',
                      'chr13 Loss', 'chr13q Loss', 
                      'chr17 Loss', 'chr17p Loss', 'chr17q Loss',
                      'chr12 Gain', 'chr12p Gain', 'chr12q Gain',
                      'chr13 CN-LOH', 'chr13q CN-LOH']
    
    lymphoid_mcas = ['chr10 Loss', 'chr10p Loss', 'chr10q Loss',
                     'chr11 Loss', 'chr11q Loss', 
                     'chr13 Loss', 'chr13q Loss',
                     'chr14 Loss', 'chr14q Loss',
                     'chr15 Loss', 'chr15q Loss',
                     'chr17 Loss', 'chr17p Loss',
                     'chr1 Loss', 'chr1p Loss', 'chr1q Loss',
                     'chr22 Loss', 'chr22q Loss',
                     'chr6 Loss', 'chr6q Loss',
                     'chr7 Loss', 'chr7q Loss',
                     'chr8 Loss', 'chr8p Loss',
                     'chr12 Gain', 'chr12q Gain',
                     'chr15 Gain', 'chr15q Gain',
                     'chr17 Gain', 'chr17q Gain',
                     'chr22 Gain', 'chr22q Gain',
                     'chr2 Gain', 'chr2p Gain',
                     'chr3 Gain', 'chr3q Gain',
                     'chr8 Gain', 'chr8q Gain',
                     'chr9 Gain', 'chr9q Gain',
                     'chr16 CN-LOH', 'chr16p CN-LOH',
                     'chr1 CN-LOH', 'chr1q CN-LOH',
                     'chr7 CN-LOH', 'chr7q CN-LOH',
                     'chr13 CN-LOH', 'chr13q CN-LOH',
                     'chr12 CN-LOH', 'chr12q CN-LOH',
                     'chr9 CN-LOH', 'chr9q CN-LOH',
                     'chr18 Gain',
                     'chr19 Gain']

    myeloid_mcas = ['chr12q Loss', 'chr12 Loss',
                    'chr20q Loss', 'chr20 Loss',
                    'chr5 Loss', 'chr5q Loss',
                    'chr1 Gain', 'chr1q Gain', 
                    'chr9 Gain', 'chr9p Gain',
                    'chr22 CN-LOH', 'chr22q CN-LOH',
                    'chr9 CN-LOH', 'chr9p CN-LOH', 
                    'chr14 CN-LOH', 'chr14q CN-LOH',
                    'chr8 Gain']
    
    a_mcas = ['chr21 Gain', 'chr21q Gain',
              'chr11 CN-LOH', 'chr11 CN-LOH',
              'chr16 CN-LOH', 'chr16 CN-LOH',
              'chr1 CN-LOH', 'chr1p CN-LOH',
              'chr17 CN-LOH', 'chr17p CN-LOH']
        
    if genes.isna().all():
        return 'reference'
    
    unique_genes = genes.dropna().unique()
    
    if len(unique_genes) == 0:
        return 'reference'
    
    elif len(set(unique_genes).intersection(set(myeloid_mcas)))>0 and len(set(unique_genes).intersection(set(lymphoid_mcas)))>0:
        return 'a_mca'
    
    elif len(unique_genes) > 1:
        return 'multiple'
    
    else:
        gene = unique_genes[0]
        if gene in high_risk_mcas:
            return 'high_risk'
        elif gene in myeloid_mcas:
            return 'm_mca'
        elif gene in lymphoid_mcas:
            return 'l_mca'
        elif gene in a_mcas:
            return 'a_mca'
        else:
            return 'other'

Group variables and code variables as dummy variables for modeling

In [ ]:
# Convert all relevant columns to datetime
date_columns = ['index_datetime', 
                'last_cbc_datetime', 
                'death_date', 
                'persistent_cytopenia_datetime', 
                'aml_mds_mf_date', 
                'cll_date']


# Convert all date columns including index_datetime
date_columns = ['index_datetime', 'death_date', 'persistent_cytopenia_datetime', 'aml_mds_mf_date', 'cll_date', 'last_cbc_datetime']
for col in date_columns:
    c[col] = pd.to_datetime(c[col], errors='coerce')
    # Remove timezone info if present
    if c[col].dt.tz is not None:
        c[col] = c[col].dt.tz_localize(None)

# Then proceed with the rest of your calculations
earliest_dates = pd.Series(index=c.index, dtype='datetime64[ns]')
for col in ['death_date', 'persistent_cytopenia_datetime', 'aml_mds_mf_date', 'cll_date']:
    mask = c[col].notna()
    earliest_dates = earliest_dates.combine_first(c[col])
    earliest_dates = pd.Series(np.where(mask & (c[col] < earliest_dates), 
                                      c[col], 
                                      earliest_dates), 
                             index=c.index, 
                             dtype='datetime64[ns]')

c['earliest_date'] = earliest_dates
c['final_date'] = c['earliest_date'].combine_first(c['last_cbc_datetime'])

# Now this should work
c['duration'] = (c['final_date'] - c['index_datetime']).dt.days

# Create age variables
# Groups: < 60, 60-80, > 80
c['age_over_65'] = c['age'].apply(lambda x: 1 if x >= 65 else 0)
c['age_group'] = pd.cut(c['age'], bins=[0, 60, 80, np.inf], labels=[0, 1, 2])

c['duration_years'] = (c['duration'] / 365.25).round(2)

# Create MCV and RDW group
c['mcv_100'] = c['mcv'].apply(lambda x: 1 if x >= 100 else 0)
c['rdw_15'] = c['rdw'].apply(lambda x: 1 if x >= 15 else 0)

# Gender as integer construct
c['gender_num'] = c['gender'].map({'female': 0, 'male': 1})

# Create VAF columns based on the specified ranges
c['VAF_20'] = (c['VAF'] >= 0.20).astype(int)
c['VAF_range'] = pd.cut(c['VAF'], bins=[0, 0.1, 0.2, 1], labels=[0, 1, 2])

c['persistent_cytopenia_count'] = c['persistent_anemia'] + c['persistent_thrombocytopenia'] + c['persistent_leukopenia']
c['persistent_cytoses_count'] = c['persistent_polycythemia'] + c['persistent_thrombocytoses'] + c['persistent_leukocytoses']
c['persistent_abnormal_count'] = c['persistent_cytopenia_count'] + c['persistent_cytoses_count']

# Map the genes to their groups
c['gene_group'] = c['gene'].fillna('Other')

# Determine 'gene_class' for each group
c['gene_class'] = c.groupby('person_id')['gene'].transform(determine_gene_class)

# Map gene_class to gene_class_num
gene_class_to_num = {
    'reference': 0,
    'high_risk': 1,
    'l_mca': 2,
    'm_mca': 3,
    'a_mca': 4,
    'multiple': 5,
    'other': 6,
}

c['gene_class_num'] = c['gene_class'].map(gene_class_to_num)


high_risk_mcas = ['chr6 Loss', 'chr6q Loss', 
                  'chr11 Loss', 'chr11q Loss',
                  'chr13 Loss', 'chr13q Loss', 
                  'chr17 Loss', 'chr17p Loss', 'chr17q Loss',
                  'chr12 Gain', 'chr12p Gain', 'chr12q Gain',
                  'chr13 CN-LOH', 'chr13q CN-LOH']

# Create the 'high_risk_genes' column
c['high_risk_genes'] = c['gene'].isin(high_risk_mcas).astype(int)

# Filter out rows where 'gene' is NaN
c_non_na = c.dropna(subset=['gene'])

# Calculate the number of mutations for each person_id
mutation_counts = c_non_na.groupby('person_id')['gene'].size().reset_index(name='mca_mutations')

# Merge the mutation_counts back into the original dataframe
c = c.merge(mutation_counts, on='person_id', how='left')

c['mca_mutations'] = c['mca_mutations'].fillna(0)

c['two_or_more_mcas'] = (c['mca_mutations'] >= 2).astype(int)

# Create a new column 'chip_mutation_group' based on 'chip_mutations'
c['chip_mutations_group'] = c['mca_mutations'].apply(lambda x: 0 if x == 0 else (1 if x == 1 else 2))

c.head()

In [ ]:
cases = c[c['case']==1].copy()
controls = c[c['case']==0].copy()

In [ ]:
def compute_log_rank(df, group):
    """
    Compute the log-rank test for the given DataFrame and group.
    
    Parameters:
    - df: DataFrame containing the data
    - group: Column name to group by
    
    Returns:
    - p-value of the log-rank test
    """
    groups = df[group].unique()
    if len(groups) != 2:
        raise ValueError("Log-rank test requires exactly two groups")

    group1 = df[df[group] == groups[0]]
    group2 = df[df[group] == groups[1]]

    result = logrank_test(group1['duration'], group2['duration'],
                          event_observed_A=group1['event'],
                          event_observed_B=group2['event'])
    return result.p_value


def compute_multivariate_log_rank(df, group):
    """
    Compute the multivariate log-rank test for the given DataFrame and group for more than two groups.
    
    Parameters:
    - df: DataFrame containing the data
    - group: Column name to group by
    
    Returns:
    - p-value of the multivariate log-rank test
    """
    event_times = df['duration']
    groups = df[group]
    event_observed = df['event']

    result = multivariate_logrank_test(event_times, groups, event_observed)
    return result.p_value


def compute_cph(df, group):
    """
    Compute the Cox Proportional Hazards model for unique person_id values in the given DataFrame and group.
    
    Parameters:
    - df: DataFrame containing the data. Must include 'person_id', 'duration', 'event', and other covariates.
    - group: Column name to group by in the model formula.
    
    Returns:
    - hazard ratio, 95% confidence interval lower bound, 95% confidence interval upper bound, p-value
    """
    # Ensure 'person_id' is in the DataFrame
    if 'person_id' not in df.columns:
        raise ValueError("'person_id' column is required in the DataFrame")

    # Drop duplicates to get unique person_id values
    df_unique = df.drop_duplicates(subset='person_id')
    
    # Create a Cox Proportional-Hazards model instance
    cph = CoxPHFitter()
    
    # Fit the model
    cph.fit(df_unique, duration_col='duration', event_col='event', formula=group)
    
    # Extract summary
    summary = cph.summary
    
    # Assuming you want to extract results for the first covariate in the formula
    hr = summary['exp(coef)'][0]
    ci_lower = summary['exp(coef) lower 95%'][0]
    ci_upper = summary['exp(coef) upper 95%'][0]
    p_value = summary['p'][0]
    
    return hr, ci_lower, ci_upper, p_value


def compute_hazard_ratio(df, group_col, duration_col='duration', event_col='event'):
    """
    Compute the hazard ratio for groups based on event occurrence.
    
    Parameters:
    - df: DataFrame containing the data
    - group_col: Column name to group by
    - duration_col: Column name indicating the duration
    - event_col: Column name indicating the event
    
    Returns:
    - DataFrame with hazard ratios, 95% confidence intervals, and p-values for each group
    """
    results = []
    for group, group_data in df.groupby(group_col):
        cph = CoxPHFitter()
        cph.fit(group_data[[duration_col, event_col]], duration_col=duration_col, event_col=event_col)
        summary = cph.summary
        hr = summary.loc[event_col, 'exp(coef)']  # Hazard ratio for the event occurrence
        ci_lower = summary.loc[event_col, 'exp(coef) lower 95%']  # Lower bound of CI
        ci_upper = summary.loc[event_col, 'exp(coef) upper 95%']  # Upper bound of CI
        p_value = summary.loc[event_col, 'p']  # p-value for the coefficient
        
        results.append({
            'Group': group,
            'Hazard Ratio': hr,
            'CI Lower': ci_lower,
            'CI Upper': ci_upper,
            'p-value': p_value
        })

    return pd.DataFrame(results)


def plot_cumulative_incidence(df, group, title, unit_of_time, unit_of_time_label, labels, file_path, x_max = 48, y_max=0.30):
    """
    Plot the cumulative incidence of cytopenia stratified by a given group.
    
    Parameters:
    - df: DataFrame containing the data
    - group: Column name to group by
    - title: Title of the plot
    - unit_of_time: Unit of time for the x-axis scaling
    """
    plt.figure(figsize=(14, 8))
    
    # Ensure 'person_id' is in the DataFrame
    if 'person_id' not in df.columns:
        raise ValueError("'person_id' column is required in the DataFrame")

    # Drop duplicates to get unique person_id values
    df_unique = df[['person_id', 'duration', 'event', group]].drop_duplicates(subset='person_id')

    # Fit and plot the data for each group
    kmf_fits = []
    for case_group, case_group_data in df_unique.groupby(group):
        kmf = KaplanMeierFitter()
        label = labels[case_group]
        kmf.fit(case_group_data['duration'] / unit_of_time, event_observed=case_group_data['event'], label=label)
        kmf.plot_cumulative_density(ci_show=False)
        kmf_fits.append(kmf)

    # Add title and labels
    plt.title(title)
    plt.xlabel(unit_of_time_label)
    plt.ylabel('Cumulative Incidence')
    plt.ylim(0.0, y_max)
    plt.xlim(0.0, x_max)
    plt.legend(loc='upper right')
    plt.grid(True)

    # Add the number at risk below the x-axis for each mutation group
    ax = plt.gca()
    
    if unit_of_time_label == 'Months':
        xtick_space = 6
    elif unit_of_time_label == 'Years':
        xtick_space = 1

    # Set monthly tick marks
    ax.set_xticks(np.arange(0, x_max + 1, xtick_space))

    # Customize the plot to drop right and top borders
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    # Remove vertical and horizontal grid lines
    ax.grid(False)

    add_at_risk_counts(*kmf_fits, ax=ax, rows_to_show=['At risk'])
    
    # Compute and display the log-rank test p-value
    unique_groups = df_unique[group].nunique()
    if unique_groups == 2:
        p_value = compute_log_rank(df_unique, group)
    else:
        p_value = compute_multivariate_log_rank(df_unique, group)
    plt.text(0.8, 0.25, f'Log-Rank Test p-value: {p_value:.4f}', transform=ax.transAxes)
    
    # Compute and display the hazard ratio with 95% confidence interval and p-value
    hr, ci_lower, ci_upper, _ = compute_cph(df_unique, group)
    plt.text(0.74, 0.18, f'Hazard Ratio (95% CI): {hr:.2f} ({ci_lower:.2f} - {ci_upper:.2f})', transform=ax.transAxes)

    # Save the plot as a PDF file
    plt.savefig(file_path, format='pdf', bbox_inches='tight')
    
    # Show plot
    plt.tight_layout()
    plt.show()


def cumulative_incidence_at_timepoint(df, time_point, unit_of_time, group_col=None):
    """
    Compute the cumulative incidence at a given time-point using Kaplan-Meier estimator for multiple groups.

    Parameters:
    - df: DataFrame containing the survival data. Must include 'duration', 'event', and optionally a 'group_col' column.
    - time_point: Time-point (in the unit of time) at which to return the cumulative incidence.
    - unit_of_time: Unit of time for the time-point (e.g., 'Months', 'Years'). Default is 'Months'.
    - group_col: Column name for grouping. If None, no grouping is performed.

    Returns:
    - cumulative_incidence_by_group: Dictionary with group values as keys and cumulative incidence at the specified time-point as values.
    """
    
    # Ensure 'person_id' is in the DataFrame
    if 'person_id' not in df.columns:
        raise ValueError("'person_id' column is required in the DataFrame")

    # Ensure the DataFrame contains the necessary columns
    if 'duration' not in df.columns or 'event' not in df.columns:
        raise ValueError("'duration' and 'event' columns are required in the DataFrame")
    
    cumulative_incidence_by_group = {}

    if group_col:
        # Group by the specified column
        grouped = df.groupby(group_col)
    else:
        # No grouping, treat entire DataFrame as a single group
        grouped = [('All', df)]

    for group_name, group_df in grouped:
        # Drop duplicates to get unique person_id values
        group_df_unique = group_df.drop_duplicates(subset='person_id')
        
        # Fit Kaplan-Meier estimator
        kmf = KaplanMeierFitter()
        kmf.fit(group_df_unique['duration'] / unit_of_time, event_observed=group_df_unique['event'])
        
        # Calculate cumulative incidence at the specified time-point
        try:
            survival_func = kmf.survival_function_at_times([time_point])
            if survival_func.empty:
                raise ValueError(f"No survival function value at time-point {time_point}")
            ci_at_time = survival_func[time_point]
            cumulative_incidence = 1 - ci_at_time  # Convert survival probability to cumulative incidence
            cumulative_incidence_by_group[group_name] = cumulative_incidence
        except Exception as e:
            print(f"Error calculating cumulative incidence for group '{group_name}': {e}")
            cumulative_incidence_by_group[group_name] = None  # or handle error as needed

    return cumulative_incidence_by_group
    
    
def create_forest_plot(df, variables_of_interest, control_variables, label_dict, file_path):
    # Initialize lists to store results for each variable of interest
    hr_list = []
    ci_lower_list = []
    ci_upper_list = []
    p_values_list = []
    labels = []

    for var in variables_of_interest:
        # Create dummy variables for categorical variables and drop the first category (reference group)
        dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        
        for dummy_var in dummies.columns:
            # Combine the current dummy variable of interest with control variables
            all_vars = [dummy_var] + control_variables

            # Initialize the CoxPHFitter model
            cph = CoxPHFitter()

            # Fit the model using the combined variables
            cph.fit(df[all_vars + ['duration', 'event']], duration_col='duration', event_col='event')

            # Get summary of the fitted model
            summary = cph.summary

            # Extract the required information for the current dummy variable to include in the forest plot
            hr = summary.loc[dummy_var, 'exp(coef)']
            ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
            ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
            p_value = summary.loc[dummy_var, 'p']

            # Append the results to the lists
            hr_list.append(hr)
            ci_lower_list.append(ci_lower)
            ci_upper_list.append(ci_upper)
            p_values_list.append(p_value)
            labels.append(label_dict.get(dummy_var, f'{var}={dummy_var.split("_")[-1]} vs {var}=0'))

    # Prepare the data for the plot
    hr = pd.Series(hr_list, index=labels)
    ci_lower = pd.Series(ci_lower_list, index=labels)
    ci_upper = pd.Series(ci_upper_list, index=labels)
    p_values = pd.Series(p_values_list, index=labels)

    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)

    ax.set_xscale('log')
    
    # Set x-axis to be logarithmic
    ax.set_xlabel('Hazard Ratio')

    # Calculate the position for text labels
    text_x_position = max(ci_upper) * 1.1

    # Add text beside each point
    for i, (h, cl, cu, p) in enumerate(zip(hr, ci_lower, ci_upper, p_values)):
        text = f'{h:.2f} [{cl:.2f}, {cu:.2f}], p < {p:.2e}'
        ax.text(text_x_position, i, text, verticalalignment='center', fontsize=10)

    # Save the plot as a PDF file
    plt.savefig(file_path, format='svg', bbox_inches='tight')
    
    # Show plot
    plt.tight_layout()
    plt.show()
    

def create_multivariate_forest_plot(df, variables_of_interest, control_variables, label_dict):
    # Initialize lists to store results for each variable of interest
    hr_list = []
    ci_lower_list = []
    ci_upper_list = []
    p_values_list = []
    labels = []

    # Create a copy of the dataframe to avoid modifying the original one
    df = df.copy()

    # Initialize the CoxPHFitter model
    cph = CoxPHFitter()

    # Fit the model using all variables
    all_vars = control_variables[:]
    for var in variables_of_interest:
        if df[var].dtype == 'object' or len(df[var].unique()) > 2:
            dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
            df = pd.concat([df, dummies], axis=1)
            all_vars += list(dummies.columns)
        else:
            # For binary or continuous variables
            all_vars.append(var)
    
    # Ensure all necessary columns are included in the DataFrame
    all_vars += ['duration', 'event']

    # Fit the model on the DataFrame
    cph.fit(df[all_vars], duration_col='duration', event_col='event')

    # Get summary of the fitted model
    summary = cph.summary

    for var in variables_of_interest:
        if df[var].dtype == 'object' or len(df[var].unique()) > 2:
            # For categorical variables, use the dummy variable labels
            dummies = [col for col in df.columns if col.startswith(var)]
            for dummy_var in dummies:
                if dummy_var in summary.index:
                    # Extract the required information for the current dummy variable to include in the forest plot
                    hr = summary.loc[dummy_var, 'exp(coef)']
                    ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
                    ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
                    p_value = summary.loc[dummy_var, 'p']

                    # Append the results to the lists
                    hr_list.append(hr)
                    ci_lower_list.append(ci_lower)
                    ci_upper_list.append(ci_upper)
                    p_values_list.append(p_value)
                    labels.append(label_dict.get(dummy_var, f'{dummy_var}'))
        else:
            # For binary or continuous variables
            if var in summary.index:
                hr = summary.loc[var, 'exp(coef)']
                ci_lower = summary.loc[var, 'exp(coef) lower 95%']
                ci_upper = summary.loc[var, 'exp(coef) upper 95%']
                p_value = summary.loc[var, 'p']

                # Append the results to the lists
                hr_list.append(hr)
                ci_lower_list.append(ci_lower)
                ci_upper_list.append(ci_upper)
                p_values_list.append(p_value)
                labels.append(label_dict.get(var, var))

    # Prepare the data for the plot
    hr = pd.Series(hr_list, index=labels)
    ci_lower = pd.Series(ci_lower_list, index=labels)
    ci_upper = pd.Series(ci_upper_list, index=labels)
    p_values = pd.Series(p_values_list, index=labels)

    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)

    # Set x-axis to be logarithmic
    ax.set_xlabel('Hazard Ratio')

    # Calculate the position for text labels
    text_x_position = max(ci_upper) * 1.1

    # Add text beside each point
    for i, (h, cl, cu, p) in enumerate(zip(hr, ci_lower, ci_upper, p_values)):
        text = f'{h:.2f} [{cl:.2f}, {cu:.2f}], p < {p:.2e}'
        ax.text(text_x_position, i, text, verticalalignment='center', fontsize=10)

    plt.tight_layout()
    plt.show()

## Flow diagram of cohort

In [ ]:
p = read_from_bucket(file_path, f'{cohort_name}_participants')

# Number of participants with genome sequencing
n_sequenced = p['person_id'].nunique()

# Number of participants with prior AML, MDS, MF
n_prior_ca = p[(p['prior_aml_mds_mf'] == 1) | (p['prior_cll'] == 1)]['person_id'].nunique()

# Number of participants without multi-timepoint CBC
n_insufficient_cbc = p[((p['prior_aml_mds_mf'] == 0) & (p['prior_cll'] == 0)) & 
                       (p['cbc_count'] < 3) & 
                       (p['cbc_count_post_index'] < 1)]['person_id'].nunique()

# Number of participants without multi-timepoint CBC
n_potential = p[((p['prior_aml_mds_mf'] == 0) & (p['prior_cll'] == 0)) & 
                (p['cbc_count'] >= 3) & 
                (p['cbc_count_post_index'] >= 1)]['person_id'].nunique()

# Number of participants with cytopenia at time of enrollment
n_enrollment_cytopenia = p[((p['prior_aml_mds_mf'] == 0) & (p['prior_cll'] == 0)) & 
                           ((p['cytopenia_at_enrollment'] == 1) | (p['cytoses_at_enrollment']==1))]['person_id'].nunique()

# Number of unique participants that are eligible
n_eligible = p[(p['eligible'] == 1)]['person_id'].nunique()

# Number of unique participants that are eligible with CHIP
n_eligible_chip = p[(p['eligible'] == 1) & (p['mca'] == 1)]['person_id'].nunique() 

# Number of unique participants that are eligible without CHIP
n_eligible_no_chip = p[(p['eligible'] == 1) & (p['mca'] == 0)]['person_id'].nunique()

# Number of unique participants that matched as cases
n_cases = p[(p['case'] == 1)]['person_id'].nunique()

# Number of unique participants that matched as controls
n_controls = p[(p['case'] == 0)]['person_id'].nunique()

print('Number of participants age ≥ 18 years with genetic sequencing:', n_sequenced)
print('Prior AML, MDS, MF, or CLL', n_prior_ca)
print('Insufficient CBC data', n_insufficient_cbc)
print('Potential participants', n_potential)
print('Cytopenia or cytoses at enrollment', n_enrollment_cytopenia)
print('Eligible', n_eligible)
print('Eligible - mCA', n_eligible_chip)
print('Eligible - no mCA', n_eligible_no_chip)
print('Matched cases', n_cases)
print('Matched controls', n_controls)

assert n_sequenced == n_prior_ca + n_insufficient_cbc + n_enrollment_cytopenia + n_eligible
assert n_potential == n_eligible + n_enrollment_cytopenia
assert n_eligible == n_eligible_chip + n_eligible_no_chip

## Descriptive statistics of participants

In [ ]:
def compute_statistics(dfs_dict, variable_map):
    all_statistics = []

    for df_name, df in dfs_dict.items():
        # Filter for unique person_id
        df_unique = df.drop_duplicates(subset='person_id')
        statistics = []

        for var, (var_type, stat_type) in variable_map.items():
            if var_type == 'continuous':
                if stat_type == 'mean':
                    mean_value = df_unique[var].mean()
                    std_value = df_unique[var].std()
                    result = f"{mean_value:.2f} (± {std_value:.2f})"
                elif stat_type == 'median':
                    median_value = df_unique[var].median()
                    p25 = df_unique[var].quantile(0.25)
                    p75 = df_unique[var].quantile(0.75)
                    result = f"{median_value:.2f} [{p25:.2f}, {p75:.2f}]"
                else:
                    raise ValueError(f"Unknown stat_type for continuous variable: {stat_type}")

                statistics.append({
                    'variable': var,
                    'statistic': result,
                    'data_frame': df_name
                })

            elif var_type == 'categorical':
                if stat_type == 'count':
                    count_value = df_unique[var].value_counts()
                    percentages = (count_value / count_value.sum() * 100).round(2).astype(str) + '%'
                    for category, (val, pct) in zip(count_value.index, zip(count_value, percentages)):
                        statistics.append({
                            'variable': f"{var}_{category}",
                            'statistic': f"{val} ({pct})",
                            'data_frame': df_name
                        })
                else:
                    raise ValueError(f"Unknown stat_type for categorical variable: {stat_type}")
            
            elif var_type == 'id':
                if stat_type == 'unique':
                    nunique_value = df_unique[var].nunique()
                    result = f"{nunique_value}"
                    statistics.append({
                        'variable': var,
                        'statistic': result,
                        'data_frame': df_name
                    })
                else:
                    raise ValueError(f"Unknown stat_type for id variable: {stat_type}")
            
            else:
                raise ValueError(f"Unknown var_type: {var_type}")

        # Add the statistics to the overall list
        all_statistics.extend(statistics)

    # Create the final DataFrame
    final_df = pd.DataFrame(all_statistics)

    # Pivot the final DataFrame to have variables as rows and DataFrame names as columns
    final_df_pivot = final_df.pivot_table(index='variable', columns='data_frame', values='statistic', aggfunc=lambda x: ' '.join(x)).reset_index()
    final_df_pivot.columns.name = None  # Remove the columns name

    return final_df_pivot


def calculate_genetic_statistics(df):
        
    # Calculate count and percentage of person_id for each gene_group
    gene_group_counts = df.groupby('gene_group')['person_id'].count()
    total_persons = df['person_id'].nunique()
    gene_group_percentages = (gene_group_counts / total_persons) * 100

    # Calculate count and percentage of person_id for each VAF_range
    VAF_counts = df.groupby('VAF_20')['person_id'].count()
    VAF_percentages = (VAF_counts / total_persons) * 100

    # Create DataFrame for the statistics
    statistics = pd.DataFrame({
        'gene_group_count': gene_group_counts,
        'gene_group_percentage': gene_group_percentages,
        'VAF_count': VAF_counts,
        'VAF_percentage': VAF_percentages
    })
    
    return statistics


def process_single_df(df, columns, df_name):
    # Create a DataFrame to hold the unique person counts for each phenotype
    unique_df = df.groupby('person_id').max().reset_index()
    
    # Calculate the total number of unique participants
    total_participants = unique_df['person_id'].nunique()
    
    # Calculate participant count and percentage for each phenotype
    phenotype_stats = []
    for col in columns:
        participant_count = unique_df[col].sum()
        participant_percent = participant_count / total_participants
        phenotype_stats.append({
            'phenotype': col,
            f'{df_name}': f"{participant_count} ({participant_percent:.2%})"
        })
    
    return pd.DataFrame(phenotype_stats)


def phenotype_statistics(df_dict, columns):
    # Initialize result DataFrame with phenotypes from the first columns_list
    result = pd.DataFrame({'phenotype': columns})
    
    for key, df in df_dict.items():
        processed_df = process_single_df(df, columns, key)
        result = result.merge(processed_df, on='phenotype', how='left')
    
    return result


def compute_incidence_rate(df, condition_col, duration_col, time_span=100000):
    """
    Compute the incidence rate of a condition per specified person-years.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    condition_col (str): The column name indicating the condition (1 if the condition is present, 0 if not).
    duration_col (str): The column name indicating the duration in days.
    time_span (int): The number of person-years for the incidence rate calculation (default is 100,000).
    
    Returns:
    float: The incidence rate per specified person-years.
    """
    
    # Filter for unique person_id
    df_unique = df.drop_duplicates(subset='person_id')
    
    # Count the number of unique person_id with the condition
    num_persons_with_condition = df_unique[df_unique[condition_col] == 1]['person_id'].nunique()
    
    # Calculate total person-years for persons with the condition
    total_person_years = df_unique[duration_col].sum()
    
    # Calculate the incidence rate per specified person-years
    incidence_rate = (num_persons_with_condition / total_person_years) * time_span
    
    return incidence_rate

In [ ]:
cohorts_dict = {'cases': cases, 'controls': controls}


table_one_map = {
    'person_id': ['id', 'unique'],
    'age': ['continuous', 'median'], 
    'gender': ['categorical', 'count'], 
    'ever_smoker': ['categorical', 'count'],
    'hgb': ['continuous', 'median'],
    'plt': ['continuous', 'median'],
    'wbc': ['continuous', 'median'],
    'mcv': ['continuous', 'median'],
    'rdw': ['continuous', 'median'],
    'duration_years': ['continuous', 'median'],
    'persistent_cytopenia_count': ['categorical', 'count'],
    'persistent_anemia': ['categorical', 'count'],
    'persistent_thrombocytopenia': ['categorical', 'count'],
    'persistent_leukopenia': ['categorical', 'count'],
    'persistent_polycythemia': ['categorical', 'count'],
    'persistent_thrombocytoses': ['categorical', 'count'],
    'persistent_leukocytoses': ['categorical', 'count'],
    'persistent_cytoses_count': ['categorical', 'count'],
    'persistent_abnormal_count': ['categorical', 'count']
}

In [ ]:
compute_statistics(cohorts_dict, table_one_map)

In [ ]:
compute_incidence_rate(cases, 'persistent_cytopenia', 'duration_years'), compute_incidence_rate(controls, 'persistent_cytopenia', 'duration_years')

In [ ]:
compute_incidence_rate(cases, 'persistent_cytoses', 'duration_years'), compute_incidence_rate(controls, 'persistent_cytoses', 'duration_years')

In [ ]:
# Compute follow-up time
df_case_control = c[(c['case']==0) | (c['case']==1)].copy()
compute_statistics({'case_control': df_case_control}, {'duration_years': ['continuous', 'median']})

In [ ]:
# # At time of sample collection (<= index_datetime)
# phenotype_statistics(cohorts_dict, phenotypes_abbrev)

In [ ]:
# # Assuming 'cases' is a DataFrame containing the relevant data
# genetic_cases = calculate_genetic_statistics(cases)

# # Convert sorting columns to appropriate data types
# genetic_cases['gene_group_count'] = genetic_cases['gene_group_count'].astype(float)
# genetic_cases['VAF_count'] = genetic_cases['VAF_count'].astype(float)

# # Sort the DataFrame
# genetic_cases_sorted = genetic_cases.sort_values(by=['gene_group_count', 'VAF_count'], ascending=False)

In [ ]:
# Count unique genes per person_id
unique_genes_per_person = c.groupby('person_id')['gene'].nunique()

# Plot histogram
plt.figure(figsize=(8, 6))
plt.hist(unique_genes_per_person, bins=range(1, unique_genes_per_person.max() + 3), edgecolor='black', align='left', width=0.8)

# Annotate each bar with its frequency
for count, freq in zip(range(1, unique_genes_per_person.max() + 1), unique_genes_per_person.value_counts().sort_index()):
    plt.annotate(f'{freq}', xy=(count, freq), xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

# Set labels and title
plt.xlabel('CHIP Mutations')
plt.ylabel('Participants')
plt.title('CHIP Mutation Frequency Per Participant')

# Show plot
plt.grid(axis='y', alpha=0.75)
plt.xticks(range(1, unique_genes_per_person.max() + 1))
plt.tight_layout()
plt.show()

## Case-Control Survival Analysis

We first build the survival analysis dataframe with the following columns:
1. duration = time from blood sample for genetic sequencing to developing a persistent cytopenia or final CBC
2. event = persistent cytopenia or censoring (final CBC)
    - For people without cytopenia pull date of last blood sample
    - For people with cytopenia pull date of first persistent cytopenia
    - time 0 (blood sample for genetic sequencing)
    - time 1 (first persistent cytopenia or final blood sample)

In [ ]:
survival_columns = ['person_id', 'mca', 'duration', 
                    #'event', 
                    'persistent_cytopenia', 'persistent_cytoses',
                    'gender', 'mca_mutations', 
                    'chip_mutations_group', 
                    #'chip_mutations_DNMT3A', 
                    'age_group', 'VAF_20', 
                    'VAF_range', 'gene_group', 'gene_class', 'gene_class_num', 'high_risk_genes', 
                    'rdw_15', 'mcv_100', 'ever_smoker']

# Select relevant columns for survival analysis
cases_survival_df = cases[survival_columns].drop_duplicates('person_id').reset_index(drop=True)
cases_survival_df['case'] = 1

##################################################################################################################

# Select relevant columns for survival analysis
controls_survival_df = controls[survival_columns].drop_duplicates('person_id').reset_index(drop=True)
controls_survival_df['case'] = 0

case_control_survival_df = pd.concat([cases_survival_df, controls_survival_df], ignore_index=True)
# case_control_survival_df['chip_mutations_DNMT3A'] = case_control_survival_df['chip_mutations_DNMT3A'].replace(0, '0')
# case_control_survival_df['chip_mutations_DNMT3A'] = case_control_survival_df['chip_mutations_DNMT3A'].replace(1, '1')

case_control_survival_df.head()

**Cumulative incidence curve for time to cytopenia in cases (CHIP) and controls (no CHIP)**

In [ ]:
case_control_survival_df['event'] = case_control_survival_df['persistent_cytoses']
plot_cumulative_incidence(case_control_survival_df, 
                          'case', 
                          f'{cohort_name}: cumulative incidence of cytoses', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'no mCA', 1: '≥ 1 mCA'},
                          f'{cohort_name}_ci_case_control_cytoses_by_chip_presence.pdf',
                          x_max = x_max,
                          y_max = y_max)

**Cumulative incidence curve for time to cytopenia by number of CHIP mutations**

In [ ]:
plot_cumulative_incidence(case_control_survival_df, 
                          'gene_class_num', 
                          'Cumulative Incidence of Cytoses Stratified by CHIP Mutation Gene', 
                          unit_of_time, 
                          unit_of_time_label,
#                           {0: 'Reference',1: 'DNMT3A', 2: 'TET2', 
#                            3: 'ASXL1', 4: 'JAK2', 5: 'TP53_PPM1D', 
#                            6: 'Other', 7: 'Multiple mutations'},
                          {0: 'reference',
                           1: 'high_risk',
                           2: 'l_mca',
                           3: 'm_mca',
                           4: 'a_mca',
                           5: 'multiple',
                           6: 'other'},
                          f'{cohort_name}_ci_case_control_cytoses_by_gene.pdf',
                          x_max = x_max,
                          y_max = y_max)

In [ ]:
plot_cumulative_incidence(case_control_survival_df, 
                          'chip_mutations_group', 
                          'Cumulative Incidence of Cytoses Stratified by Number of mCAs', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: '0 mCAs', 1: '1 mCA', 2: '≥ 2 mCAs'},
                          f'{cohort_name}_ci_case_control_cytopenia_by_chip_count_group.pdf',
                          x_max = x_max,
                          y_max = y_max)

In [ ]:
plot_cumulative_incidence(case_control_survival_df, 
                          'gene_class_num', 
                          'Cumulative Incidence of Cytopenia Stratified by CHIP Mutation Gene', 
                          unit_of_time, 
                          unit_of_time_label,
#                           {0: 'Reference',1: 'DNMT3A', 2: 'TET2', 
#                            3: 'ASXL1', 4: 'JAK2', 5: 'TP53_PPM1D', 
#                            6: 'Other', 7: 'Multiple mutations'},
                          {0: 'reference',
                           1: 'high_risk',
                           2: 'l_mca',
                           3: 'm_mca',
                           4: 'a_mca',
                           5: 'multiple',
                           6: 'other'},
                          f'{cohort_name}_ci_case_control_cytopenia_by_gene.pdf',
                          x_max = x_max,
                          y_max = y_max)

In [ ]:
# plot_cumulative_incidence(case_control_survival_df, 
#                           'chip_mutations_DNMT3A', 
#                           'Cumulative Incidence of Cytopenia Stratified by Number of CHIP Mutations', 
#                           unit_of_time, 
#                           unit_of_time_label,
#                           {'0': '0 CHIP mutations', '1': '1 CHIP mutation', '>=2': '≥ 2 CHIP mutations', 
#                            '>=2 DNMT3A': '≥ 2 DNMT3A CHIP mutations'},
#                           f'{cohort_name}_ci_case_control_cytopenia_by_DNMT3A.pdf',
#                           x_max = x_max,
#                           y_max = y_max)

In [ ]:
plot_cumulative_incidence(case_control_survival_df[case_control_survival_df['case']==0], 
                          'rdw_15', 
                          'Cumulative Incidence of Cytopenia Stratified by RDW in Controls without CHIP', 
                          unit_of_time, 
                          unit_of_time_label,
                          {0: 'RDW < 15', 1: 'RDW ≥ 15'},
                          f'{cohort_name}_ci_control_cytopenia_by_rdw.pdf',
                          x_max = x_max,
                          y_max = y_max)

In [ ]:
plot_cumulative_incidence(case_control_survival_df[case_control_survival_df['case']==0], 
                          'mcv_100', 
                          'Cumulative Incidence of Cytopenia Stratified by MCV in Controls without CHIP', 
                          unit_of_time, 
                          unit_of_time_label,
                          {0: 'MCV < 100', 1: 'MCV ≥ 100'},
                          f'{cohort_name}_ci_control_cytopenia_by_mcv.pdf',
                          x_max = x_max,
                          y_max = y_max)

## CHIP Survival Analysis

In this section we construct Kaplan Meier (KM) curves starting with 100% of participants that met the following criteria:
1. At least one CHIP-defining mutation
2. No history of persistent cytopenia before or on the index CBC (i.e. CHIP at sequencing, not CCUS)
3. At least 1 CBC measurement after the index CBC (i.e. enable multi-timepoint analysis after genome sequenced).

In [ ]:
c['event'] = c['persistent_cytoses']

In [ ]:
# chip_enrollment = c[(c['chip']==1) & (c['survival']==1)]
chip_enrollment = c[c['mca']==1]
num_chip_enrollment = chip_enrollment['person_id'].nunique()
print(f'Number of participants with CHIP at enrollment: {num_chip_enrollment}')

In [ ]:
# Select relevant columns for survival analysis
survival_df = chip_enrollment[['person_id', 'mca', 'duration', 'event', 'age_over_65', 'age_group', 
                               'gender_num', 'ever_smoker', 'gene_class_num', 
                               'high_risk_genes', 'VAF_range', 'VAF_20', 'mca_mutations', 
                               'two_or_more_mcas', 'chip_mutations_group', 'mcv_100', 
                               'rdw_15']].drop_duplicates('person_id').reset_index(drop=True)

survival_df.head()

**Cumulative incidence curve for time to cytopenia by CHIP mutation gene**

In [ ]:
plot_cumulative_incidence(survival_df, 
                          'high_risk_genes', 
                          'Cumulative Incidence of Cytopenia or Blood Cancer Stratified by CHIP Mutation Gene Class', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'Not high risk mutation', 1: 'High risk mutation'},
                          f'{cohort_name}_ci_chip_cytopenia_by_high_risk.pdf',
                          x_max = x_max,
                          y_max = y_max)

**Cumulative incidence curve for time to cytopenia by VAF range**

In [ ]:
plot_cumulative_incidence(survival_df, 
                          'VAF_20', 
                          'Cumulative Incidence of Cytopenia or Blood Cancer Stratified by VAF range', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'VAF < 0.20', 1: 'VAF ≥ 0.20'},
                          f'{cohort_name}_ci_chip_cytopenia_by_VAF.pdf',
                          x_max = x_max,
                          y_max = y_max)


In [ ]:
plot_cumulative_incidence(survival_df, 
                          'VAF_20', 
                          'Cumulative Incidence of Cytopenia or Blood Cancer Stratified by VAF range', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'VAF < 0.20', 1: 'VAF ≥ 0.20'},
                          f'{cohort_name}_ci_chip_cytopenia_by_VAF.pdf',
                          x_max = x_max,
                          y_max = y_max)

**Cumulative incidence curve for time to cytopenia by age**

In [ ]:
plot_cumulative_incidence(survival_df, 
                          'age_over_65', 
                          'Cumulative Incidence of Cytopenia or Blood Cancer Stratified by age', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'Age < 65', 1: 'Age ≥ 65'},
                          f'{cohort_name}_ci_chip_cytopenia_by_age.pdf',
                          x_max = x_max,
                          y_max = y_max)

**Cumulative incidence curve for time to cytopenia by RDW**

In [ ]:
plot_cumulative_incidence(survival_df, 
                          'rdw_15', 
                          'Cumulative Incidence of Cytopenia or Blood Cancer Stratified by RDW', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'RDW < 15', 1: 'RDW ≥ 15'},
                          f'{cohort_name}_ci_chip_cytopenia_by_rdw.pdf',
                          x_max = x_max,
                          y_max = y_max)

**Cumulative incidence curve for time to cytopenia by MCV**

In [ ]:
plot_cumulative_incidence(survival_df, 
                          'mcv_100', 
                          'Cumulative Incidence of Cytopenia or Blood Cancer Stratified by MCV', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'MCV < 100', 1: 'MCV ≥ 100'},
                          f'{cohort_name}_ci_chip_cytopenia_by_mcv.pdf',
                          x_max = x_max,
                          y_max = y_max)

### Cox Proportional Hazards Modeling

**Cox Proportional Hazards model controlling for control_variables**

Results of Cox regression analyses for incident cytopenia by specific CHIP genotypes. Models run with controls that do not have CHIP mutations as reference group and were adjusted for sex, history of prior cancer and any history of smoking. Forest plot indicates HR and 95% confidence intervals. 

In [ ]:
def subgroup_counts(df, variable_list, group_variable):
    total_participants = df.shape[0]
    subgroup_data = []
    
    for variable in variable_list:
        unique_values = df[variable].unique()
        
        for subgroup_value in unique_values:
            group_variable_1_count = df[(df[variable] == subgroup_value) & (df[group_variable] == 1)].shape[0]
            group_variable_0_count = df[(df[variable] == subgroup_value) & (df[group_variable] == 0)].shape[0]
            total_count = group_variable_1_count + group_variable_0_count
            percent_of_total = (total_count / total_participants) * 100
            
            subgroup_data.append({
                'variable': variable,
                'subgroup': f'{variable}_{subgroup_value}',
                'group_variable_1': group_variable_1_count,
                'group_variable_0': group_variable_0_count,
                'total_count': total_count,
                'percent_of_total': percent_of_total
            })
    
    subgroup_df = pd.DataFrame(subgroup_data)
    return subgroup_df


def compute_univariate_hr(df, input_variables, control_variables):
    results = []
    
    for var in input_variables:
        # Create dummy variables for categorical variables and drop the first category (reference group)
        dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
        df_with_dummies = pd.concat([df, dummies], axis=1)
        
        for dummy_var in dummies.columns:
            # Combine the current dummy variable of interest with control variables
            all_vars = [dummy_var] + control_variables

            # Initialize the CoxPHFitter model
            cph = CoxPHFitter()

            # Fit the model using the combined variables
            cph.fit(df_with_dummies[all_vars + ['duration', 'event']], duration_col='duration', event_col='event')

            # Get summary of the fitted model
            summary = cph.summary

            # Extract the required information for the current dummy variable
            hr = summary.loc[dummy_var, 'exp(coef)']
            ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
            ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
            p_value = summary.loc[dummy_var, 'p']

            # Append the results to the list
            results.append({
                'variable': var,
                'subgroup': dummy_var,
                'reference_group': f'{var}_0',
                'hazard_ratio': hr,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'p_value': p_value
            })
    
    # Convert results list to DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df


def build_subgroup_hazard_table(df, variable_list, group_variable, input_variables, control_variables):
    # Get the subgroup counts
    subgroup_df = subgroup_counts(df, variable_list, group_variable)
    
    # Get the univariate hazard ratios
    hr_df = compute_univariate_hr(df, input_variables, control_variables)
    
    # Merge the two dataframes on the variable and subgroup columns
    merged_df = pd.merge(subgroup_df, hr_df, on=['variable', 'subgroup'], how='left')
    
    # Sort the final dataframe
    merged_df = merged_df.sort_values(by=['variable', 'subgroup'], ascending=[True, False])
    
    return merged_df

In [ ]:
chip_forest_variables = ['age_over_65', 'rdw_15', 'mcv_100', 'high_risk_genes', 'two_or_more_mcas', 
                         'VAF_20', 'gender_num', 'ever_smoker']
control_variables = [] # no control ['gender_num', 'ever_smoker']
chip_forest_labels = {'age_over_65_1': 'Age ≥ 65', 'rdw_15_1': 'RDW ≥ 15', 
                      'mcv_100_1': 'MCV ≥ 100', 'high_risk_genes_1': 'High risk genes', 
                      'two_or_more_mcas': '≥ 2 mutations', 'VAF_20_1': 'VAF ≥ 0.20', 
                      'gender_num_1.0': 'Male', 'gender_num_2': 'Other gender', 
                      'ever_smoker_1': 'Smoker'}
create_forest_plot(survival_df, chip_forest_variables, control_variables, chip_forest_labels, 
                   f'{cohort_name}_chip_forest_plot.pdf')

In [ ]:
create_forest_plot(survival_df, chip_forest_variables, control_variables, chip_forest_labels, 
                   f'{cohort_name}_chip_forest_plot.pdf')

In [ ]:
univariate_vars_chip_hazard = build_subgroup_hazard_table(survival_df, 
                                                          chip_forest_variables, 
                                                          'event', 
                                                          chip_forest_variables, 
                                                          [])

univariate_vars_chip_hazard

In [ ]:
gene_forest_label = {'gene_class_num_1': 'high_risk', 'gene_class_num_2': 'l_mca','gene_class_num_3': 'm_mca', 
                     'gene_class_num_4': 'a_mca', 'gene_class_num_5': 'multiple', 
                     'gene_class_num_6': 'other'}


create_forest_plot(case_control_survival_df, ['gene_class_num'], [], gene_forest_label, f'{cohort_name}_gene_forest_plot.pdf')

In [ ]:
univariate_case_control_genes_hazard = build_subgroup_hazard_table(case_control_survival_df, 
                                                                   ['gene_class_num'], 
                                                                   'event', 
                                                                   ['gene_class_num'], 
                                                                   [])

In [ ]:
univariate_case_control_genes_hazard

In [ ]:
# Function to generate summary statistics for a given column
def generate_summary_stats(df, group_col):
    summary = df.groupby(group_col).agg(
        person_id_count=('person_id', 'nunique'),
        duration_median=('duration', 'median'),
        duration_25th_percentile=('duration', lambda x: np.percentile(x, 25)),
        duration_75th_percentile=('duration', lambda x: np.percentile(x, 75)),
        event_count=('event', 'sum')
    ).reset_index().rename(columns={group_col: 'group_value'})
    summary['group_type'] = group_col
    return summary

# Generate summary statistics for each group type
age_group_summary = generate_summary_stats(survival_df, 'age_over_65')
gender_summary = generate_summary_stats(survival_df, 'gender_num')
vaf_range_summary = generate_summary_stats(survival_df, 'VAF_20')
gene_group_summary = generate_summary_stats(survival_df, 'gene_class_num')

# Concatenate all summaries into a single DataFrame
summary_df = pd.concat(
    [
        age_group_summary,
        gender_summary,
        vaf_range_summary,
        gene_group_summary
    ],
    ignore_index=True
)

# Reorder columns to make 'group_type' the first column
summary_df = summary_df[['group_type', 'group_value', 'person_id_count', 'duration_median', 'duration_25th_percentile', 'duration_75th_percentile', 'event_count']]
summary_df

In [ ]:
def plot_gene_frequencies(df, n, file_path):
    # Get top n genes by frequency
    top_genes = df.nlargest(n, 'count')
    # Sort for plotting
    top_genes = top_genes.sort_values(by='count', ascending=True).reset_index(drop=True)
    
    # Plotting
    fig, ax = plt.subplots(figsize=(5, 5))
    bars = ax.barh(top_genes['gene'], top_genes['count'], color='#4e79a7', height=0.7)
    
    # Set x-axis to logarithmic scale
    ax.set_xscale('log')
    
    # Adding percentages at a fixed distance from the left end of bars
    for bar, pct in zip(bars, top_genes['percent']):
        ax.text(bar.get_width() * 0.8, bar.get_y() + bar.get_height()/2, f'{pct:.1f}%', 
                va='center', ha='left', color='white', fontweight='bold', fontsize=10)
    
    # Removing top and right borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Set x-axis label
    ax.set_xlabel('Number of individuals', fontsize=12)
    
    # Set x-axis ticks and labels
    ax.set_xticks([1, 10, 100, 1000, 10000])
    ax.set_xticklabels(['1', '10', '10²', '10³', '10⁴'])
    
    # Adjust y-axis
    ax.set_yticks(range(len(top_genes)))
    ax.set_yticklabels(top_genes['gene'], fontsize=10)
    
    # Invert y-axis to make bars start on the right
    ax.invert_xaxis()
    
    # Remove x-axis minor ticks
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    
    # Save the plot as a PDF file
    plt.savefig(file_path, format='pdf', bbox_inches='tight')
    
    plt.show()

In [ ]:
gene_counts = c.groupby('gene')['person_id'].nunique().reset_index()
gene_counts.columns = ['gene', 'count']
gene_counts['total_count'] = gene_counts['count'].sum()
gene_counts['percent'] = ((gene_counts['count'] / gene_counts['total_count']) * 100).round(2)
gene_counts.head(10)

In [ ]:
plot_gene_frequencies(gene_counts, 10, f'{cohort_name}_chip_top_10_genes.pdf')

In [ ]:
chip_gene_counts = chip_enrollment.groupby('gene')['person_id'].nunique().reset_index()
chip_gene_counts.columns = ['gene', 'count']
chip_gene_counts['total_count'] = chip_gene_counts['count'].sum()
chip_gene_counts['percent'] = ((chip_gene_counts['count'] / chip_gene_counts['total_count']) * 100).round(2)
chip_gene_counts.head(10)

In [ ]:
plot_gene_frequencies(chip_gene_counts, 10, f'{cohort_name}_chip_cases_top_10_genes.pdf')

## Incident myeloid neoplasms

In [ ]:
c['date_first_mn'] = c['aml_mds_mf_date']

In [ ]:
c['prior_mn'] = c['prior_aml_mds_mf']

In [ ]:
c['incident_mn'] = (~c['first_aml_mds_mf'].isna()).astype(int)

In [ ]:
c['date_first_cll'] = c['cll_date']
c['prior_cll'] = c['prior_cll']
c['incident_cll'] = (~c['first_cll'].isna()).astype(int)

In [ ]:
def compute_counts_and_percentages(df):
    df['persistent_cytopenia_datetime'] = pd.to_datetime(df['persistent_cytopenia_datetime'])
#     df['date_first_mn'] = pd.to_datetime(df['date_first_mn'])
    df['date_first_cll'] = pd.to_datetime(df['date_first_cll'])

    results = {}
    
    for chip_value in [0, 1]:
        sub_df = df[(df['mca'] == chip_value) & 
                    (~df['cbc_count'].isna()) & 
#                     (df['prior_mn'] == 0) & 
                    (df['prior_cll']==0)]
        
#         cond1_person_id = (sub_df['cytopenia_at_enrollment'] == 0) & (sub_df['persistent_cytopenia'] == 0)
#         cond2_person_id = (sub_df['cytopenia_at_enrollment'] == 0) & (sub_df['persistent_cytopenia'] == 1)
#         cond3_person_id = (sub_df['cytopenia_at_enrollment'] == 0) & (sub_df['persistent_cytopenia'] == 1)
#         cond4_person_id = (sub_df['cytopenia_at_enrollment'] == 1)
        
        cond1_person_id = (sub_df['cytoses_at_enrollment'] == 0) & (sub_df['persistent_cytoses'] == 0)
        cond2_person_id = (sub_df['cytoses_at_enrollment'] == 0) & (sub_df['persistent_cytoses'] == 1)
        cond3_person_id = (sub_df['cytoses_at_enrollment'] == 0) & (sub_df['persistent_cytoses'] == 1)
        cond4_person_id = (sub_df['cytoses_at_enrollment'] == 1)

        
        # Condition 1
        cond1 = cond1_person_id & (sub_df['incident_cll'] == 1)
        count1 = sub_df[cond1]['person_id'].nunique()
        total1 = sub_df[cond1_person_id]['person_id'].nunique()
        perc1 = round(count1 / total1 * 100, 2) if total1 > 0 else 0
        
        # Condition 2
        cond2 = cond2_person_id & (sub_df['persistent_cytoses_datetime'] <= sub_df['date_first_cll']) & (sub_df['incident_cll'] == 1)
        count2 = sub_df[cond2]['person_id'].nunique()
        total2 = sub_df[cond2_person_id]['person_id'].nunique()
        perc2 = round(count2 / total2 * 100, 2) if total2 > 0 else 0
        
        # Condition 3
        cond3 = cond3_person_id & (sub_df['persistent_cytoses_datetime'] > sub_df['date_first_cll']) & (sub_df['incident_cll'] == 1)
        count3 = sub_df[cond3]['person_id'].nunique()
        total3 = sub_df[cond3_person_id]['person_id'].nunique()
        perc3 = round(count3 / total3 * 100, 2) if total3 > 0 else 0
        
        # Condition 4
        cond4 = cond4_person_id & (sub_df['incident_cll'] == 1)
        count4 = sub_df[cond4]['person_id'].nunique()
        total4 = sub_df[cond4_person_id]['person_id'].nunique()
        perc4 = round(count4 / total4 * 100, 2) if total4 > 0 else 0
                
        results[f'chip_{chip_value}'] = {
            'No incident cytoses': f'{count1}/{total1} ({perc1}%)',
            'Incident cytoses before CLL': f'{count2}/{total2} ({perc2}%)',
            'Incident cytoses after CLL': f'{count3}/{total3} ({perc3}%)',
            'Prevalent cytoses': f'{count4}/{total4} ({perc4}%)',
        }
        
    df_final = pd.DataFrame.from_dict(results)
    
    return df_final

In [ ]:
compute_counts_and_percentages(c)

In [ ]:
# participants = read_from_bucket(file_path, f'{cohort_name}_participants')
# c = c.merge(participants[['person_id', 'death_date']], how='left', on='person_id')

# Convert datetime columns to datetime type if they aren't already
datetime_columns = ['index_datetime', 'date_first_mn', 'death_date']
for col in datetime_columns:
    c[col] = pd.to_datetime(c[col])
    
c['last_follow_up'] = pd.to_datetime('2023-12-31').tz_localize('UTC')

# Calculate duration (in days) until event or censoring for all rows
c['duration_mn'] = (c[['last_follow_up', 'date_first_mn', 'death_date']].min(axis=1) - c['index_datetime']).dt.days

# Convert duration from days to years
c['duration_mn_years'] = c['duration_mn'] / 365.25

# Create a mask for valid durations (not NaN and not negative)
valid_duration_mask = (c['duration_mn'].notna()) & (c['duration_mn'] >= 0) & (c['date_first_mn'].notna())

# Determine if the event (incident myeloid neoplasm) occurred for rows with valid duration
c['event_mn'] = np.where(valid_duration_mask, 1, 0)

# cases = c[c[match_name]==1].copy()
# controls = c[c[match_name]==0].copy()

# Select relevant columns for survival analysis
incident_mn_cases = cases[['person_id', 
                              'chip', 
                              'persistent_cytopenia', 
                              'duration_mn', 
                              'event_mn']] \
                            .rename(columns={'duration_mn': 'duration', 'event_mn': 'event'}) \
                            .drop_duplicates('person_id').reset_index(drop=True)

# Select relevant columns for survival analysis
incident_mn_controls = controls[['person_id', 
                              'chip', 
                              'persistent_cytopenia', 
                              'duration_mn', 
                              'event_mn']] \
                            .rename(columns={'duration_mn': 'duration', 'event_mn': 'event'}) \
                            .drop_duplicates('person_id').reset_index(drop=True)

In [ ]:
plot_cumulative_incidence(incident_mn_cases, 
                          'persistent_cytopenia', 
                          f'{cohort_name}: cumulative incidence of myeloid neoplasm in chip cases', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'no cytopenia', 1: 'cytopenia'},
                          f'{cohort_name}_ci_myeloid_neoplasm_by_cytopenia_cases.pdf',
                          x_max = 5,
                          y_max = 0.08)

In [ ]:
plot_cumulative_incidence(incident_mn_controls, 
                          'persistent_cytopenia', 
                          f'{cohort_name}: cumulative incidence of myeloid neoplasm in controls', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'no cytopenia', 1: 'cytopenia'},
                          f'{cohort_name}_ci_myeloid_neoplasm_by_cytopenia_controls.pdf',
                          x_max = 5,
                          y_max = 0.08)

In [ ]:
incident_mn = c[['person_id', 'index_datetime', 'survival']].copy()

ca_df = pid_phenotypes[['person_id', 'ls_phenotype', 'ls_phenotype_subclass', 'date_first_ls', 
                        'mn_phenotype', 'mn_phenotype_subclass', 'date_first_mn']].copy()

incident_mn = incident_mn.merge(ca_df, how='left', on='person_id')

# Convert datetime columns to datetime type if they aren't already
incident_mn['index_datetime'] = pd.to_datetime(incident_mn['index_datetime'])
incident_mn['date_first_mn'] = pd.to_datetime(incident_mn['date_first_mn']).dt.tz_localize(pytz.UTC)

offset = 0

# Create columns for myeloid neoplasm and lymphoid/solid neoplasms diagnosed BEFORE enrollment
incident_mn['prior_mn'] = np.where(incident_mn['index_datetime'] + pd.DateOffset(months=offset) >= incident_mn['date_first_mn'], 1, 0)

# Create columns for myeloid neoplasm and lymphoid/solid neoplasms diagnosed AFTER enrollment
incident_mn['incident_mn'] = np.where(incident_mn['date_first_mn'] > incident_mn['index_datetime'] + pd.DateOffset(months=offset), 1, 0)

incident_mn[(incident_mn['incident_mn']==1) & (incident_mn['survival']==1)]['mn_phenotype_subclass'].value_counts()

## ASH Abstract Analysis

In [ ]:
def add_feature_count_columns(df, columns_list):
    """
    Adds two new columns to the DataFrame:
    1. 'feature_count': Number of columns in columns_list that are == 1 for each row.
    2. 'three_or_more_features': 1 if 'feature_count' >= 3, else 0.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns_list (list): List of column names to check for value == 1.

    Returns:
    pd.DataFrame: The DataFrame with the two new columns added.
    """
    # Ensure all columns in the list are present in the DataFrame
    if not all(col in df.columns for col in columns_list):
        raise ValueError("One or more columns in columns_list are not in the DataFrame")
    
    # Calculate feature_count
    df['feature_count'] = df[columns_list].sum(axis=1)
    
    # Calculate three_or_more_features
    df['three_or_more_features'] = (df['feature_count'] >= 3).astype(int)
    
    return df

In [ ]:
# Example usage
features = ['chip_mutations_two_or_more', 'high_risk_genes', 'age_over_65', 'gender_num', 'mcv_100', 'rdw_15']
outcome_var = 'persistent_cytopenia'
id_var = 'person_id'

cases_features = add_feature_count_columns(cases, features)
compute_cph(cases_features, 'three_or_more_features')

In [ ]:
plot_cumulative_incidence(cases_features, 
                          'feature_count', 
                          f'{cohort_name}: cumulative incidence of cytopenia', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: '0 high risk features', 
                           1: '1 high risk features', 
                           2: '2 high risk features', 
                           3: '3 high risk features', 
                           4: '4 high risk features', 
                           5: '5 high risk features'},
                          f'{cohort_name}_ci_case_cytopenia_by_high_risk_feature_count.pdf',
                          x_max = x_max,
                          y_max = y_max)

In [ ]:
plot_cumulative_incidence(cases_features, 
                          'three_or_more_features', 
                          f'{cohort_name}: cumulative incidence of cytopenia', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: '< 3 high risk features', 1: '≥ 3 high risk features'},
                          f'{cohort_name}_ci_case_cytopenia_by_high_risk_features.pdf',
                          x_max = x_max,
                          y_max = y_max)

In [ ]:
cumulative_incidence_at_timepoint(cases_features, 2, unit_of_time, 'three_or_more_features')

# Grouped analysis

In [ ]:
def compute_statistics(dfs_dict, variable_map):
    all_statistics = []

    for df_name, df in dfs_dict.items():
        # Filter for unique person_id
        df_unique = df.drop_duplicates(subset='person_id')
        statistics = []

        for var, (var_type, stat_type) in variable_map.items():
            if var_type == 'continuous':
                if stat_type == 'mean':
                    mean_value = df_unique[var].mean()
                    std_value = df_unique[var].std()
                    result = f"{mean_value:.2f} (± {std_value:.2f})"
                elif stat_type == 'median':
                    median_value = df_unique[var].median()
                    p25 = df_unique[var].quantile(0.25)
                    p75 = df_unique[var].quantile(0.75)
                    result = f"{median_value:.2f} [{p25:.2f}, {p75:.2f}]"
                else:
                    raise ValueError(f"Unknown stat_type for continuous variable: {stat_type}")

                statistics.append({
                    'variable': var,
                    'statistic': result,
                    'data_frame': df_name
                })

            elif var_type == 'categorical':
                if stat_type == 'count':
                    count_value = df_unique[var].value_counts()
                    percentages = (count_value / count_value.sum() * 100).round(2).astype(str) + '%'
                    for category, (val, pct) in zip(count_value.index, zip(count_value, percentages)):
                        statistics.append({
                            'variable': f"{var}_{category}",
                            'statistic': f"{val} ({pct})",
                            'data_frame': df_name
                        })
                else:
                    raise ValueError(f"Unknown stat_type for categorical variable: {stat_type}")
            
            elif var_type == 'id':
                if stat_type == 'unique':
                    nunique_value = df_unique[var].nunique()
                    result = f"{nunique_value}"
                    statistics.append({
                        'variable': var,
                        'statistic': result,
                        'data_frame': df_name
                    })
                else:
                    raise ValueError(f"Unknown stat_type for id variable: {stat_type}")
            
            else:
                raise ValueError(f"Unknown var_type: {var_type}")

        # Add the statistics to the overall list
        all_statistics.extend(statistics)

    # Create the final DataFrame
    final_df = pd.DataFrame(all_statistics)

    # Pivot the final DataFrame to have variables as rows and DataFrame names as columns
    final_df_pivot = final_df.pivot_table(index='variable', columns='data_frame', values='statistic', aggfunc=lambda x: ' '.join(x)).reset_index()
    final_df_pivot.columns.name = None  # Remove the columns name

    return final_df_pivot


def compute_incidence_rate(df, condition_col, duration_col, time_span=100000):
    """
    Compute the incidence rate of a condition per specified person-years.
    
    Parameters:
    df (pd.DataFrame): The DataFrame containing the data.
    condition_col (str): The column name indicating the condition (1 if the condition is present, 0 if not).
    duration_col (str): The column name indicating the duration in days.
    time_span (int): The number of person-years for the incidence rate calculation (default is 100,000).
    
    Returns:
    float: The incidence rate per specified person-years.
    """
    
    # Filter for unique person_id
    df_unique = df.drop_duplicates(subset='person_id')
    
    # Count the number of unique person_id with the condition
    num_persons_with_condition = df_unique[df_unique[condition_col] == 1]['person_id'].nunique()
    
    # Calculate total person-years for persons with the condition
    total_person_years = df_unique[duration_col].sum()
    
    # Calculate the incidence rate per specified person-years
    incidence_rate = (num_persons_with_condition / total_person_years) * time_span
    
    return incidence_rate


def determine_gene_class(genes):
    
    # Define gene types
    gene_types = ['DNMT3A', 'TET2', 'ASXL1', 'JAK2']
    tp53_ppm1d = ['TP53', 'PPM1D']
    idh = ['IDH1', 'IDH2']
    ssuz= ['SF3B1', 'SRSF2', 'U2AF1', 'ZRSR2']

    if genes.isna().all():
        return 'reference'
    unique_genes = genes.dropna().unique()
    if len(unique_genes) == 0:
        return 'reference'
    elif len(unique_genes) > 1:
        return 'multiple'
    else:
        gene = unique_genes[0]
        if gene in gene_types:
            return gene
        elif gene in tp53_ppm1d:
            return 'TP53_or_PPM1D'
        elif gene in idh:
            return 'IDH1_or_IDH2'
        elif gene in ssuz:
            return 'SF3B1_or_SRSF2_or_U2AF1_or_ZRSR2'
        else:
            return 'other'


def compute_log_rank(df, group, duration_col, event_col):
    """
    Compute the log-rank test for the given DataFrame and group.
    
    Parameters:
    - df: DataFrame containing the data
    - group: Column name to group by
    
    Returns:
    - p-value of the log-rank test
    """
    groups = df[group].unique()
    if len(groups) != 2:
        raise ValueError("Log-rank test requires exactly two groups")

    group1 = df[df[group] == groups[0]]
    group2 = df[df[group] == groups[1]]

    result = logrank_test(group1[duration_col], group2[duration_col],
                          event_observed_A=group1[event_col],
                          event_observed_B=group2[event_col])
    return result.p_value


def compute_multivariate_log_rank(df, group, duration_col, event_col):
    """
    Compute the multivariate log-rank test for the given DataFrame and group for more than two groups.
    
    Parameters:
    - df: DataFrame containing the data
    - group: Column name to group by
    
    Returns:
    - p-value of the multivariate log-rank test
    """
    event_times = df[duration_col]
    groups = df[group]
    event_observed = df[event_col]

    result = multivariate_logrank_test(event_times, groups, event_observed)
    return result.p_value


def compute_cph(df, group, duration_col, event_col):
    """
    Compute the Cox Proportional Hazards model for unique person_id values in the given DataFrame and group.
    
    Parameters:
    - df: DataFrame containing the data. Must include 'person_id', 'duration', 'event', and other covariates.
    - group: Column name to group by in the model formula.
    
    Returns:
    - hazard ratio, 95% confidence interval lower bound, 95% confidence interval upper bound, p-value
    """
    # Ensure 'person_id' is in the DataFrame
    if 'person_id' not in df.columns:
        raise ValueError("'person_id' column is required in the DataFrame")

    # Drop duplicates to get unique person_id values
    df_unique = df.drop_duplicates(subset='person_id')
    
    # Create a Cox Proportional-Hazards model instance
    cph = CoxPHFitter()
    
    # Fit the model
    cph.fit(df_unique, duration_col=duration_col, event_col=event_col, formula=group)
    
    # Extract summary
    summary = cph.summary
    
    # Assuming you want to extract results for the first covariate in the formula
    hr = summary['exp(coef)'][0]
    ci_lower = summary['exp(coef) lower 95%'][0]
    ci_upper = summary['exp(coef) upper 95%'][0]
    p_value = summary['p'][0]
    
    return hr, ci_lower, ci_upper, p_value


def analyze_cohorts(df, duration_col, event_col, time_point, unit_of_time):
    """
    Analyze each cohort in the DataFrame and calculate hazard ratios, lower 95% CI, upper 95% CI, and p-values.
    """
    if 'cohort' not in df.columns:
        raise ValueError("The DataFrame does not have a 'cohort' column.")

    # Get unique cohorts
    cohorts = df['cohort'].unique()

    # Create a list to hold results
    results = []
    
    hr, ci_lower, ci_upper, p_value = compute_cph(df, 'mca', duration_col, event_col)
    cumulative_incidence = cumulative_incidence_at_timepoint(df, duration_col, event_col, time_point, unit_of_time, 'mca')
        
    results.append({
        'cohort': 'all',
        'hazard_ratio': hr,
        'lower_95_CI': ci_lower,
        'upper_95_CI': ci_upper,
        'p_value': p_value,
        f'ci_cases_{time_point}_years': cumulative_incidence[1],
        f'ci_controls_{time_point}_years': cumulative_incidence[0]
    })

    for cohort in cohorts:
        cohort_df = df[df['cohort'] == cohort]
        
        # Ensure the DataFrame is not empty
        if cohort_df.empty:
            continue
        
        hr, ci_lower, ci_upper, p_value = compute_cph(cohort_df, 'mca', duration_col, event_col)
        
        
        
        cumulative_incidence = cumulative_incidence_at_timepoint(cohort_df, duration_col, event_col, time_point, unit_of_time, 'mca')
        
        results.append({
            'cohort': cohort,
            'hazard_ratio': hr,
            'lower_95_CI': ci_lower,
            'upper_95_CI': ci_upper,
            'p_value': p_value,
            f'ci_cases_{time_point}_years': cumulative_incidence[1],
            f'ci_controls_{time_point}_years': cumulative_incidence[0]
        })

    # Create DataFrame from results
    results_df = pd.DataFrame(results)
    
    return results_df


def plot_cumulative_incidence(df, 
                              group, 
                              duration_col,
                              event_col,
                              title, 
                              unit_of_time, 
                              unit_of_time_label, 
                              labels, 
                              colors, 
                              file_path, 
                              x_max = 48, 
                              y_max=0.30, 
                              ci=True):
    """
    Plot the cumulative incidence of cytopenia stratified by a given group.
    
    Parameters:
    - df: DataFrame containing the data
    - group: Column name to group by
    - title: Title of the plot
    - unit_of_time: Unit of time for the x-axis scaling
    """
    plt.figure(figsize=(14, 8))
    
    # Ensure 'person_id' is in the DataFrame
    if 'person_id' not in df.columns:
        raise ValueError("'person_id' column is required in the DataFrame")

    # Drop duplicates to get unique person_id values
    df_unique = df[['person_id', duration_col, event_col, group]].drop_duplicates(subset='person_id')

    # Fit and plot the data for each group
    kmf_fits = []
    for case_group, case_group_data in df_unique.groupby(group):
        kmf = KaplanMeierFitter()
        label = labels[case_group]
        color = colors[case_group]
        kmf.fit(case_group_data[duration_col] / unit_of_time, event_observed=case_group_data[event_col], label=label)
        kmf.plot_cumulative_density(ci_show=ci, color=color)
        kmf_fits.append(kmf)

    # Add title and labels
    plt.title(title)
    plt.xlabel(unit_of_time_label)
    plt.ylabel('Cumulative Incidence')
    plt.ylim(0.0, y_max)
    plt.xlim(0.0, x_max)
    plt.legend(loc='upper right')
    plt.grid(True)

    # Add the number at risk below the x-axis for each mutation group
    ax = plt.gca()
    
    if unit_of_time_label == 'Months':
        xtick_space = 6
    elif unit_of_time_label == 'Years':
        xtick_space = 1

    # Set monthly tick marks
    ax.set_xticks(np.arange(0, x_max + 1, xtick_space))

    # Customize the plot to drop right and top borders
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    # Remove vertical and horizontal grid lines
    ax.grid(False)

    add_at_risk_counts(*kmf_fits, ax=ax, rows_to_show=['At risk'])
    
    # Compute and display the log-rank test p-value
    unique_groups = df_unique[group].nunique()
    if unique_groups == 2:
        p_value = compute_log_rank(df_unique, group, duration_col, event_col)
    else:
        p_value = compute_multivariate_log_rank(df_unique, group, duration_col, event_col)
#     plt.text(0.8, 0.25, f'Log-Rank Test p-value: {p_value:.4f}', transform=ax.transAxes)
    
    # Compute and display the hazard ratio with 95% confidence interval and p-value
    hr, ci_lower, ci_upper, _ = compute_cph(df_unique, group, duration_col, event_col)
#     plt.text(0.74, 0.18, f'Hazard Ratio (95% CI): {hr:.2f} ({ci_lower:.2f} - {ci_upper:.2f})', transform=ax.transAxes)

    # Save the plot as a PDF file
    plt.savefig(file_path, format='svg', bbox_inches='tight')
    
    # Show plot
    plt.tight_layout()
    plt.show()


def cumulative_incidence_at_timepoint(df, duration_col, event_col, time_point, unit_of_time, group_col=None):
    """
    Compute the cumulative incidence at a given time-point using Kaplan-Meier estimator for multiple groups.

    Parameters:
    - df: DataFrame containing the survival data. Must include 'duration', 'event', and optionally a 'group_col' column.
    - time_point: Time-point (in the unit of time) at which to return the cumulative incidence.
    - unit_of_time: Unit of time for the time-point (e.g., 'Months', 'Years'). Default is 'Months'.
    - group_col: Column name for grouping. If None, no grouping is performed.

    Returns:
    - cumulative_incidence_by_group: Dictionary with group values as keys and cumulative incidence at the specified time-point as values.
    """
    
    # Ensure 'person_id' is in the DataFrame
    if 'person_id' not in df.columns:
        raise ValueError("'person_id' column is required in the DataFrame")

    # Ensure the DataFrame contains the necessary columns
    if duration_col not in df.columns or event_col not in df.columns:
        raise ValueError("'duration' and 'event' columns are required in the DataFrame")
    
    cumulative_incidence_by_group = {}

    if group_col:
        # Group by the specified column
        grouped = df.groupby(group_col)
    else:
        # No grouping, treat entire DataFrame as a single group
        grouped = [('All', df)]

    for group_name, group_df in grouped:
        # Drop duplicates to get unique person_id values
        group_df_unique = group_df.drop_duplicates(subset='person_id')
        
        # Fit Kaplan-Meier estimator
        kmf = KaplanMeierFitter()
        kmf.fit(group_df_unique[duration_col] / unit_of_time, event_observed=group_df_unique[event_col])
        
        # Calculate cumulative incidence at the specified time-point
        try:
            survival_func = kmf.survival_function_at_times([time_point])
            if survival_func.empty:
                raise ValueError(f"No survival function value at time-point {time_point}")
            ci_at_time = survival_func[time_point]
            cumulative_incidence = 1 - ci_at_time  # Convert survival probability to cumulative incidence
            cumulative_incidence_by_group[group_name] = cumulative_incidence.round(4)
        except Exception as e:
            print(f"Error calculating cumulative incidence for group '{group_name}': {e}")
            cumulative_incidence_by_group[group_name] = None  # or handle error as needed

    return cumulative_incidence_by_group


def forest_plot(df, title, file_path, hr_col='hazard_ratio', ci_lower_col='lower_95_CI', ci_upper_col='upper_95_CI', 
                p_value_col='p_value', cohort_col='cohort', case_incidence_col='ci_cases_2_years',
                control_incidence_col='ci_controls_2_years', cohort_map=None, 
                cohort_order=None, figsize=(14, 6), xlabel='Hazard Ratio'):
    # If cohort_order is provided, reorder the DataFrame
    if cohort_order:
        df = df.set_index(cohort_col).loc[cohort_order].reset_index()

    # Prepare the data for the plot
    hr = df[hr_col]
    ci_lower = df[ci_lower_col]
    ci_upper = df[ci_upper_col]
    p_values = df[p_value_col]
    labels = df[cohort_col]
    case_incidences = df[case_incidence_col]
    control_incidences = df[control_incidence_col]

    # Apply cohort mapping if provided
    if cohort_map:
        labels = labels.map(lambda x: cohort_map.get(x, x))

    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)
    
    ax.tick_params(top=False,
                   bottom=False,
                   left=False,
                   right=False,
                   labelleft=True,
                   labelbottom=True)

    # Set x-axis to be linear with custom ticks and a narrower range
    ax.set_xscale('linear')
    ax.set_xlim(0.8, 1.6)
    ax.set_xticks([0.8, 1.0, 1.2, 1.4])
    ax.set_xlabel(xlabel)

    # Remove all spines except the bottom one
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    # Remove all grid lines
    ax.grid(False)

    # Calculate the position for text labels
    hr_text_x_position = 1.4
    case_incidence_x_position = 1.7
    control_incidence_x_position = 1.9

    # Add column headers
    ax.text(hr_text_x_position*1.05, len(hr), 'Hazard Ratio', fontweight='bold', verticalalignment='bottom', fontsize=8)
    ax.text(case_incidence_x_position*0.97, len(hr), 'Case Incidence (2 yr)', fontweight='bold', verticalalignment='bottom', fontsize=8)
    ax.text(control_incidence_x_position*0.97, len(hr), 'Control Incidence (2 yr)', fontweight='bold', verticalalignment='bottom', fontsize=8)

    # Add text beside each point
    for i, (h, cl, cu, p, case_inc, control_inc) in enumerate(zip(hr, ci_lower, ci_upper, p_values, case_incidences, control_incidences)):
        hr_text = f'{h:.2f} [{cl:.2f}, {cu:.2f}], p = {p:.2e}'
        ax.text(hr_text_x_position, i, hr_text, verticalalignment='center', fontsize=10)
        ax.text(case_incidence_x_position, i, f'{100*case_inc:.2f}%', verticalalignment='center', fontsize=10)
        ax.text(control_incidence_x_position, i, f'{100*control_inc:.2f}%', verticalalignment='center', fontsize=10)

    # Adjust layout to prevent clipping
    plt.tight_layout()
    
    ax.text(0.7, len(hr) + 0.5, title, fontsize=10, fontweight='bold', va='bottom', ha='left')

    # Save the plot as a PDF file
    plt.savefig(file_path, format='pdf', bbox_inches='tight')
    
    # Show the plot
    plt.show()
    

def create_forest_plot(df, 
                       variables_of_interest, 
                       control_variables, 
                       duration_col, 
                       event_col, 
                       label_dict, 
                       file_path, 
                       title, 
                       counts=False, 
                       sort='count', 
                       reference_dict={}):
    """
    Generate a forest plot from a DataFrame with hazard ratios, confidence intervals, and p-values.

    Parameters:
    - df: DataFrame containing the data
    - variables_of_interest: List of variables of interest
    - control_variables: List of control variables
    - label_dict: Dictionary mapping dummy variable names to human-readable labels
    - file_path: Path to save the plot
    - title: Title of the plot
    - counts: Boolean indicating whether to include count of unique person_id
    - sort: Sorting method, 'count' to sort by number of participants or 'alphabetical' to sort by hazard ratio category names
    """
    # Initialize lists to store results for each variable of interest
    hr_list = []
    ci_lower_list = []
    ci_upper_list = []
    p_values_list = []
    labels = []
    n_counts = []

    for var in variables_of_interest:
        # Create dummy variables for categorical variables and drop the first category (reference group)
        dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        
        for dummy_var in dummies.columns:
            # Combine the current dummy variable of interest with control variables
            all_vars = [dummy_var] + control_variables + [duration_col, event_col]
            label = label_dict.get(dummy_var, f'{var}={dummy_var.split("_")[-1]} vs {var}=0')
            
            # If counts is True, calculate the number of unique person_id where dummy_var == 1
            if counts:
                subset_df = df[df[dummy_var] == 1]
                n_unique = subset_df['person_id'].nunique()
                n_counts.append(n_unique)
                label = f'{label} (N={n_unique})'
            else:
                n_counts.append(None)  # Placeholder if counts is False

            labels.append(label)

            # Initialize the CoxPHFitter model
            cph = CoxPHFitter()

            # Fit the model using the combined variables
            cph.fit(df[all_vars], duration_col=duration_col, event_col=event_col)

            # Get summary of the fitted model
            summary = cph.summary

            # Extract the required information for the current dummy variable to include in the forest plot
            hr = summary.loc[dummy_var, 'exp(coef)']
            ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
            ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
            p_value = summary.loc[dummy_var, 'p']

            # Append the results to the lists
            hr_list.append(hr)
            ci_lower_list.append(ci_lower)
            ci_upper_list.append(ci_upper)
            p_values_list.append(p_value)
    
    if reference_dict:
        # Add reference group manually
        reference_label = reference_dict['label']
        reference_hr = reference_dict['hr']
        reference_count = reference_dict['count']
        hr_list.append(reference_hr)
        n_counts.append(reference_count)
        labels.append(f'{reference_label} (N={reference_count})')
        ci_lower_list.append(np.nan)
        ci_upper_list.append(np.nan)
        p_values_list.append(np.nan)
    
    # Create a DataFrame for the results
    results_df = pd.DataFrame({
        'Label': labels,
        'HR': hr_list,
        'CI Lower': ci_lower_list,
        'CI Upper': ci_upper_list,
        'P Value': p_values_list,
        'Count': n_counts
    })

    # Sort the DataFrame based on the 'sort' parameter
    if sort == 'count' and counts:
        results_df.sort_values(by='Count', ascending=True, inplace=True)
    elif sort == 'alphabetical':
        results_df.sort_values(by='Label', ascending=True, inplace=True)

    # Prepare the data for the plot
    hr = results_df['HR']
    ci_lower = results_df['CI Lower']
    ci_upper = results_df['CI Upper']
    p_values = results_df['P Value']
    labels = results_df['Label']
    
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)

    
    ax.tick_params(top=False,
                   bottom=False,
                   left=False,
                   right=False,
                   labelleft=True,
                   labelbottom=True)

    # Set x-axis to be logarithmic
    ax.set_xlabel('Hazard Ratio')

    
    # Remove all spines except the bottom one
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Remove all grid lines
    ax.grid(False)

    # Calculate the position for text labels
    text_x_position = max(ci_upper) * 1.1

    # Add text beside each point
    for i, (h, cl, cu, p) in enumerate(zip(hr, ci_lower, ci_upper, p_values)):
        if np.isnan(p):
            ax.text(text_x_position, i, 'Reference', verticalalignment='center', fontsize=10)
        else:
            text = f'{h:.2f} [{cl:.2f}, {cu:.2f}], p = {p:.2e}'
            ax.text(text_x_position, i, text, verticalalignment='center', fontsize=10)

    ax.text(0, len(hr), title, fontsize=10, fontweight='bold', va='bottom', ha='left')
        
    # Save the plot as a PDF file
    plt.savefig(file_path, format='svg', bbox_inches='tight')
    
    # Show plot
    plt.tight_layout()
    plt.show()
    
    return results_df
     
    
def create_multivariate_forest_plot(df, variables_of_interest, control_variables, label_dict):
    # Initialize lists to store results for each variable of interest
    hr_list = []
    ci_lower_list = []
    ci_upper_list = []
    p_values_list = []
    labels = []

    # Create a copy of the dataframe to avoid modifying the original one
    df = df.copy()

    # Initialize the CoxPHFitter model
    cph = CoxPHFitter()

    # Fit the model using all variables
    all_vars = control_variables[:]
    for var in variables_of_interest:
        if df[var].dtype == 'object' or len(df[var].unique()) > 2:
            dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
            df = pd.concat([df, dummies], axis=1)
            all_vars += list(dummies.columns)
        else:
            # For binary or continuous variables
            all_vars.append(var)
    
    # Ensure all necessary columns are included in the DataFrame
    all_vars += ['duration', 'event']

    # Fit the model on the DataFrame
    cph.fit(df[all_vars], duration_col='duration', event_col='event')

    # Get summary of the fitted model
    summary = cph.summary

    for var in variables_of_interest:
        if df[var].dtype == 'object' or len(df[var].unique()) > 2:
            # For categorical variables, use the dummy variable labels
            dummies = [col for col in df.columns if col.startswith(var)]
            for dummy_var in dummies:
                if dummy_var in summary.index:
                    # Extract the required information for the current dummy variable to include in the forest plot
                    hr = summary.loc[dummy_var, 'exp(coef)']
                    ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
                    ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
                    p_value = summary.loc[dummy_var, 'p']

                    # Append the results to the lists
                    hr_list.append(hr)
                    ci_lower_list.append(ci_lower)
                    ci_upper_list.append(ci_upper)
                    p_values_list.append(p_value)
                    labels.append(label_dict.get(dummy_var, f'{dummy_var}'))
        else:
            # For binary or continuous variables
            if var in summary.index:
                hr = summary.loc[var, 'exp(coef)']
                ci_lower = summary.loc[var, 'exp(coef) lower 95%']
                ci_upper = summary.loc[var, 'exp(coef) upper 95%']
                p_value = summary.loc[var, 'p']

                # Append the results to the lists
                hr_list.append(hr)
                ci_lower_list.append(ci_lower)
                ci_upper_list.append(ci_upper)
                p_values_list.append(p_value)
                labels.append(label_dict.get(var, var))

    # Prepare the data for the plot
    hr = pd.Series(hr_list, index=labels)
    ci_lower = pd.Series(ci_lower_list, index=labels)
    ci_upper = pd.Series(ci_upper_list, index=labels)
    p_values = pd.Series(p_values_list, index=labels)

    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)

    # Set x-axis to be logarithmic
    ax.set_xlabel('Hazard Ratio')

    # Calculate the position for text labels
    text_x_position = max(ci_upper) * 1.1

    # Add text beside each point
    for i, (h, cl, cu, p) in enumerate(zip(hr, ci_lower, ci_upper, p_values)):
        text = f'{h:.2f} [{cl:.2f}, {cu:.2f}], p < {p:.2e}'
        ax.text(text_x_position, i, text, verticalalignment='center', fontsize=10)

    plt.tight_layout()
    plt.show()
    

def subgroup_counts(df, variable_list, group_variable):
    total_participants = df.shape[0]
    subgroup_data = []
    
    for variable in variable_list:
        unique_values = df[variable].unique()
        
        for subgroup_value in unique_values:
            group_variable_1_count = df[(df[variable] == subgroup_value) & (df[group_variable] == 1)].shape[0]
            group_variable_0_count = df[(df[variable] == subgroup_value) & (df[group_variable] == 0)].shape[0]
            total_count = group_variable_1_count + group_variable_0_count
            percent_of_total = (total_count / total_participants) * 100
            
            subgroup_data.append({
                'variable': variable,
                'subgroup': f'{variable}_{subgroup_value}',
                'group_variable_1': group_variable_1_count,
                'group_variable_0': group_variable_0_count,
                'total_count': total_count,
                'percent_of_total': percent_of_total
            })
    
    subgroup_df = pd.DataFrame(subgroup_data)
    return subgroup_df


def compute_univariate_hr(df, input_variables, control_variables):
    results = []
    
    for var in input_variables:
        # Create dummy variables for categorical variables and drop the first category (reference group)
        dummies = pd.get_dummies(df[var], prefix=var, drop_first=True)
        df_with_dummies = pd.concat([df, dummies], axis=1)
        
        for dummy_var in dummies.columns:
            # Combine the current dummy variable of interest with control variables
            all_vars = [dummy_var] + control_variables

            # Initialize the CoxPHFitter model
            cph = CoxPHFitter()

            # Fit the model using the combined variables
            cph.fit(df_with_dummies[all_vars + ['duration', 'event']], duration_col='duration', event_col='event')

            # Get summary of the fitted model
            summary = cph.summary

            # Extract the required information for the current dummy variable
            hr = summary.loc[dummy_var, 'exp(coef)']
            ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
            ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
            p_value = summary.loc[dummy_var, 'p']

            # Append the results to the list
            results.append({
                'variable': var,
                'subgroup': dummy_var,
                'reference_group': f'{var}_0',
                'hazard_ratio': hr,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'p_value': p_value
            })
    
    # Convert results list to DataFrame
    results_df = pd.DataFrame(results)
    
    return results_df


def build_subgroup_hazard_table(df, variable_list, group_variable, input_variables, control_variables):
    # Get the subgroup counts
    subgroup_df = subgroup_counts(df, variable_list, group_variable)
    
    # Get the univariate hazard ratios
    hr_df = compute_univariate_hr(df, input_variables, control_variables)
    
    # Merge the two dataframes on the variable and subgroup columns
    merged_df = pd.merge(subgroup_df, hr_df, on=['variable', 'subgroup'], how='left')
    
    # Sort the final dataframe
    merged_df = merged_df.sort_values(by=['variable', 'subgroup'], ascending=[True, False])
    
    return merged_df

In [ ]:
! gsutil cp ukb_participants.csv {BUCKET}/pershy1/mca_cell_counts/

In [ ]:
! gsutil cp ukb_cohort.csv {BUCKET}/pershy1/mca_cell_counts/

In [ ]:
##########################
###### PARTICIPANTS ######
##########################

participant_columns = {'person_id': 'string', 
                       'gender': 'string',
                       'race': 'string',
                       'ethnicity': 'string',
                       'ever_smoker': 'int',
                       'age': 'float64',
                       'chip': 'int',
                       'cytopenia_at_enrollment': 'int',
                       'cbc_count': 'int',
                       'cbc_count_post_index': 'int', 
                       'prior_aml_mds_mf': 'int', 
                       'eligible': 'int', 
                       'case': 'int'}

p_aou = read_from_bucket(file_path, 
                         'aou_participants', 
                         column_types=participant_columns, 
                         datetime_columns=['birth_datetime'])

# p_biovu = read_from_bucket(file_path, 
#                            'biovu_participants', 
#                            column_types=participant_columns, 
#                            datetime_columns=['birth_datetime'])

p_ukb = read_from_bucket(file_path, 
                         'ukb_participants', 
                         column_types=participant_columns, 
                         datetime_columns=['birth_datetime'])

p_aou['cohort'] = 'aou'
# p_biovu['cohort'] = 'biovu'
p_ukb['cohort'] = 'ukb'

p_aou['person_id'] = 'a' + p_aou['person_id']
# p_biovu['person_id'] = 'b' + p_biovu['person_id']
p_ukb['person_id'] = 'u' + p_ukb['person_id']

p_grouped = pd.concat([p_aou, 
#                        p_biovu, 
                       p_ukb])

p_grouped['cohort'] = p_grouped['cohort'].astype('string')

assert check_column_types(p_grouped, {'person_id': 'string', 
                                      'birth_datetime': 'datetime64[ns, UTC]',
                                      'gender': 'string',
                                      'race': 'string',
                                      'ethnicity': 'string',
                                      'ever_smoker': 'int',
                                      'age': 'float64',
                                      'mca': 'int',
                                      'cytopenia_at_enrollment': 'int',
                                      'cytoses_at_enrollment': 'int',
                                      'cbc_count': 'int',
                                      'cbc_count_post_index': 'int', 
                                      'prior_aml_mds_mf': 'int', 
                                      'prior_cll': 'int',
                                      'eligible': 'int', 
                                      'case': 'int', 
                                      'cohort': 'string'})

In [ ]:
##########################
######### COHORT #########
##########################

cohort_name = 'all'

cohort_columns = {'person_id': 'string', 
                  'gender': 'string',
                  'race': 'string',
                  'ethnicity': 'string',
                  'ever_smoker': 'int',
                  'age': 'float64',
                  'chip': 'int',
                  'prior_aml_mds_mf': 'int',
                  'case': 'int',
                  'gene': 'string',
                  'VAF': 'float64',
                  'cytopenia_at_enrollment': 'int',
                  'hgb': 'float64',
                  'wbc': 'float64',
                  'plt': 'float64',
                  'mcv': 'float64',
                  'rdw': 'float64',
                  'persistent_anemia': 'int',
                  'persistent_leukopenia': 'int',
                  'persistent_thrombocytopenia': 'int',
                  'persistent_cytopenia': 'int',
                  'cbc_count': 'int',
                  'cbc_count_post_index': 'int', 
                  'first_aml_mds_mf': 'string', 
                  'duration': 'int', 
                  'event': 'int', 
                  'censoring_reason': 'string'}

cohort_datetimes = ['birth_datetime', 
                    'death_date', 
                    'index_datetime', 
                    'persistent_cytopenia_datetime', 
                    'first_cbc_datetime', 
                    'last_cbc_datetime', 
                    'aml_date', 
                    'mds_date', 
                    'mf_date', 
                    'et_date', 
                    'pv_date', 
                    'aml_mds_mf_date']

c_aou = read_from_bucket(file_path, 
                         'aou_cohort', 
                         column_types=cohort_columns, 
                         datetime_columns=cohort_datetimes)

# Function to safely convert to UTC timezone
def to_utc(series):
    try:
        # If timezone naive, localize to UTC
        return pd.to_datetime(series, errors='coerce').dt.tz_localize('UTC')
    except TypeError:
        # If already timezone aware, convert to UTC
        return pd.to_datetime(series, errors='coerce').dt.tz_convert('UTC')

# Convert datetime columns to UTC
datetime_cols = ['persistent_cytoses_datetime', 
                'persistent_abnormal_counts_datetime', 
                'cll_date']

for col in datetime_cols:
    c_aou[col] = to_utc(c_aou[col])

# Convert first_cll to string type, handling NaN values
c_aou['first_cll'] = c_aou['first_cll'].astype('string')
c_aou.loc[c_aou['first_cll'] == '', 'first_cll'] = np.nan
    
# c_biovu = read_from_bucket(file_path, 
#                            'biovu_cohort', 
#                            column_types=cohort_columns, 
#                            datetime_columns=cohort_datetimes)

c_ukb = read_from_bucket(file_path, 
                         'ukb_cohort',
                         column_types=cohort_columns, 
                         datetime_columns=cohort_datetimes)

for col in datetime_cols:
    c_ukb[col] = to_utc(c_ukb[col])

# Convert first_cll to string type, handling NaN values
c_ukb['first_cll'] = c_ukb['first_cll'].astype('string')
c_ukb.loc[c_ukb['first_cll'] == '', 'first_cll'] = np.nan



c_aou['person_id'] = 'a' + c_aou['person_id']
# c_biovu['person_id'] = 'b' + c_biovu['person_id']
c_ukb['person_id'] = 'u' + c_ukb['person_id']



# Localize each datetime column to UTC (All of Us mf_date has no values and lost localization)
for column in cohort_datetimes:
    # Explicitly convert the column to datetime, handling errors
    c_aou[column] = pd.to_datetime(c_aou[column], errors='coerce')
    
    # Check if the column is naive and localize it to UTC
    if c_aou[column].dt.tz is None:
        c_aou[column] = c_aou[column].dt.tz_localize('UTC')
        
assert check_column_types(c_aou, 
                          {'person_id': 'string',
                           'birth_datetime': 'datetime64[ns, UTC]',
                           'death_date': 'datetime64[ns, UTC]',
                           'index_datetime': 'datetime64[ns, UTC]', 
                           'gender': 'string',
                           'race': 'string',
                           'ethnicity': 'string',
                           'ever_smoker': 'int',
                           'age': 'float64',
                           'mca': 'int',
                           'prior_aml_mds_mf': 'int',
                           'prior_cll': 'int',
                           'case': 'int',
                           'gene': 'string',
                           'VAF': 'float64',
                           'cytopenia_at_enrollment': 'int',
                           'cytoses_at_enrollment': 'int',
                           'hgb': 'float64',
                           'wbc': 'float64',
                           'plt': 'float64',
                           'mcv': 'float64',
                           'rdw': 'float64',
                           'persistent_anemia': 'int',
                           'persistent_leukopenia': 'int',
                           'persistent_thrombocytopenia': 'int',
                           'persistent_cytopenia': 'int',
                           'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
                           'first_cbc_datetime': 'datetime64[ns, UTC]',
                           'persistent_polycythemia': 'int', 
                           'persistent_leukocytoses': 'int', 
                           'persistent_thrombocytoses': 'int',
                           'persistent_cytoses': 'int',
                           'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
                           'persistent_abnormal_blood_counts': 'int',
                           'persistent_abnormal_counts_datetime': 'datetime64[ns, UTC]',
                           'last_cbc_datetime': 'datetime64[ns, UTC]',
                           'cbc_count': 'int',
                           'cbc_count_post_index': 'int',
                           'aml_date': 'datetime64[ns, UTC]',
                           'mds_date': 'datetime64[ns, UTC]',
                           'mf_date': 'datetime64[ns, UTC]',
                           'et_date': 'datetime64[ns, UTC]',
                           'pv_date': 'datetime64[ns, UTC]', 
                           'cll_date': 'datetime64[ns, UTC]',
                           'first_aml_mds_mf': 'string', 
                           'first_cll': 'string',
                           'aml_mds_mf_date': 'datetime64[ns, UTC]'})

# assert check_column_types(c_biovu, 
#                           {'person_id': 'string',
#                            'birth_datetime': 'datetime64[ns, UTC]',
#                            'death_date': 'datetime64[ns, UTC]',
#                            'index_datetime': 'datetime64[ns, UTC]', 
#                            'gender': 'string',
#                            'race': 'string',
#                            'ethnicity': 'string',
#                            'ever_smoker': 'int',
#                            'age': 'float64',
#                            'chip': 'int',
#                            'prior_aml_mds_mf': 'int',
#                            'case': 'int',
#                            'gene': 'string',
#                            'VAF': 'float64',
#                            'cytopenia_at_enrollment': 'int',
#                            'hgb': 'float64',
#                            'wbc': 'float64',
#                            'plt': 'float64',
#                            'mcv': 'float64',
#                            'rdw': 'float64',
#                            'persistent_anemia': 'int',
#                            'persistent_leukopenia': 'int',
#                            'persistent_thrombocytopenia': 'int',
#                            'persistent_cytopenia': 'int',
#                            'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
#                            'first_cbc_datetime': 'datetime64[ns, UTC]',
#                            'last_cbc_datetime': 'datetime64[ns, UTC]',
#                            'cbc_count': 'int',
#                            'cbc_count_post_index': 'int',
#                            'aml_date': 'datetime64[ns, UTC]',
#                            'mds_date': 'datetime64[ns, UTC]',
#                            'mf_date': 'datetime64[ns, UTC]',
#                            'et_date': 'datetime64[ns, UTC]',
#                            'pv_date': 'datetime64[ns, UTC]', 
#                            'first_aml_mds_mf': 'string', 
#                            'aml_mds_mf_date': 'datetime64[ns, UTC]'})

assert check_column_types(c_ukb, 
                          {'person_id': 'string',
                           'birth_datetime': 'datetime64[ns, UTC]',
                           'death_date': 'datetime64[ns, UTC]',
                           'index_datetime': 'datetime64[ns, UTC]', 
                           'gender': 'string',
                           'race': 'string',
                           'ethnicity': 'string',
                           'ever_smoker': 'int',
                           'age': 'float64',
                           'mca': 'int',
                           'prior_aml_mds_mf': 'int',
                           'prior_cll': 'int',
                           'case': 'int',
                           'gene': 'string',
                           'VAF': 'float64',
                           'cytopenia_at_enrollment': 'int',
                           'cytoses_at_enrollment': 'int',
                           'hgb': 'float64',
                           'wbc': 'float64',
                           'plt': 'float64',
                           'mcv': 'float64',
                           'rdw': 'float64',
                           'persistent_anemia': 'int',
                           'persistent_leukopenia': 'int',
                           'persistent_thrombocytopenia': 'int',
                           'persistent_cytopenia': 'int',
                           'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
                           'first_cbc_datetime': 'datetime64[ns, UTC]',
                           'persistent_polycythemia': 'int', 
                           'persistent_leukocytoses': 'int', 
                           'persistent_thrombocytoses': 'int',
                           'persistent_cytoses': 'int',
                           'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
                           'persistent_abnormal_blood_counts': 'int',
                           'persistent_abnormal_counts_datetime': 'datetime64[ns, UTC]',
                           'last_cbc_datetime': 'datetime64[ns, UTC]',
                           'cbc_count': 'int',
                           'cbc_count_post_index': 'int',
                           'aml_date': 'datetime64[ns, UTC]',
                           'mds_date': 'datetime64[ns, UTC]',
                           'mf_date': 'datetime64[ns, UTC]',
                           'et_date': 'datetime64[ns, UTC]',
                           'pv_date': 'datetime64[ns, UTC]', 
                           'cll_date': 'datetime64[ns, UTC]',
                           'first_aml_mds_mf': 'string', 
                           'first_cll': 'string',
                           'aml_mds_mf_date': 'datetime64[ns, UTC]'})

c_aou['cohort'] = 'aou'
# https://support.researchallofus.org/hc/en-us/articles/360051661772-What-are-the-CDR-cutoff-dates
c_aou['cutoff_date'] = pd.to_datetime('2022-07-02').tz_localize('UTC')

# c_biovu['cohort'] = 'biovu'
# # https://vumc365.sharepoint.com/sites/Template/SitePages/Home.aspx?csf=1&web=1&e=mqC5ox
# c_biovu['cutoff_date'] = pd.to_datetime('2023-09-01').tz_localize('UTC')

c_ukb['cohort'] = 'ukb'
# # https://biobank.ctsu.ox.ac.uk/crystal/exinfo.cgi?src=Data_providers_and_dates
c_ukb['cutoff_date'] = pd.to_datetime('2020-12-31').tz_localize('UTC')

c_grouped = pd.concat([c_aou, 
#                        c_biovu, 
                       c_ukb])

# c_grouped = c_aou

c_grouped['cohort'] = c_grouped['cohort'].astype('string')

assert check_column_types(c_grouped, 
                          {'person_id': 'string',
                           'birth_datetime': 'datetime64[ns, UTC]',
                           'death_date': 'datetime64[ns, UTC]',
                           'index_datetime': 'datetime64[ns, UTC]', 
                           'gender': 'string',
                           'race': 'string',
                           'ethnicity': 'string',
                           'ever_smoker': 'int',
                           'age': 'float64',
                           'mca': 'int',
                           'prior_aml_mds_mf': 'int',
                           'prior_cll': 'int',
                           'case': 'int',
                           'gene': 'string',
                           'VAF': 'float64',
                           'cytopenia_at_enrollment': 'int',
                           'cytoses_at_enrollment': 'int',
                           'hgb': 'float64',
                           'wbc': 'float64',
                           'plt': 'float64',
                           'mcv': 'float64',
                           'rdw': 'float64',
                           'persistent_anemia': 'int',
                           'persistent_leukopenia': 'int',
                           'persistent_thrombocytopenia': 'int',
                           'persistent_cytopenia': 'int',
                           'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
                           'first_cbc_datetime': 'datetime64[ns, UTC]',
                           'persistent_polycythemia': 'int', 
                           'persistent_leukocytoses': 'int', 
                           'persistent_thrombocytoses': 'int',
                           'persistent_cytoses': 'int',
                           'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
                           'persistent_abnormal_blood_counts': 'int',
                           'persistent_abnormal_counts_datetime': 'datetime64[ns, UTC]',
                           'last_cbc_datetime': 'datetime64[ns, UTC]',
                           'cbc_count': 'int',
                           'cbc_count_post_index': 'int',
                           'aml_date': 'datetime64[ns, UTC]',
                           'mds_date': 'datetime64[ns, UTC]',
                           'mf_date': 'datetime64[ns, UTC]',
                           'et_date': 'datetime64[ns, UTC]',
                           'pv_date': 'datetime64[ns, UTC]', 
                           'cll_date': 'datetime64[ns, UTC]',
                           'first_aml_mds_mf': 'string', 
                           'aml_mds_mf_date': 'datetime64[ns, UTC]',
                           'first_cll': 'string',
                           'cohort': 'string', 
                           'cutoff_date': 'datetime64[ns, UTC]'})

# c_gwas_inputs = c_grouped[c_grouped['chip']==1][['person_id', 'cohort', 'age', 'gender', 'gene', 'VAF', 'persistent_anemia', 'persistent_leukopenia', 'persistent_thrombocytopenia', 'persistent_cytopenia']]
# save_to_bucket(c_gwas_inputs, file_path, f'{cohort_name}_chip_cytopenia_gwas_inputs')

We first prepare our dataset for survival analysis dataframe with the following columns:
1. duration: days from genetic sequencing to developing a persistent cytopenia OR being censored (final CBC, incident hematologic malignancy)
2. event: 1 for persistent cytopenia OR 0 for censoring (final CBC, incident hematologic malignancy)
    - For people without persistent cytopenia pull date of censoring (earliest of last blood sample or incident hematologic malignancy) 
    - For people with persistent cytopenia pull date of first persistent cytopenia
    - time 0 (blood sample for genetic sequencing)
    - time 1 (first persistent cytopenia or final blood sample if no persistent cytopenia)

In [ ]:
# Time to cytopenia, censoring (last CBC), or competing risk (hematologic malignancy, death)
c_grouped['time_cytopenia'] = (c_grouped[['last_cbc_datetime', 'persistent_cytopenia_datetime', 'aml_mds_mf_date']].min(axis=1) - c_grouped['index_datetime']).dt.days

# Create a DataFrame with the relevant columns
dates_df = c_grouped[['last_cbc_datetime', 'persistent_cytopenia_datetime', 'aml_mds_mf_date']]

# Compute the minimum date for each row
min_dates = dates_df.min(axis=1)

# Determine the event type using vectorized comparison
event_cytopenia = np.select(
    [
        c_grouped['last_cbc_datetime'] == min_dates,
        c_grouped['persistent_cytopenia_datetime'] == min_dates,
        c_grouped['aml_mds_mf_date'] == min_dates
    ],
    [0, 1, 2],
    default=np.nan
).astype(int)

# Assign the computed event types to the DataFrame
c_grouped['event_cytopenia'] = event_cytopenia

In [ ]:
# Time to cytopenia, censoring (last CBC), or competing risk (hematologic malignancy, death)
c_grouped['time_cytoses'] = (c_grouped[['last_cbc_datetime', 
                                        'persistent_cytoses_datetime', 
                                        'aml_mds_mf_date',
                                        'cll_date']].min(axis=1) - c_grouped['index_datetime']).dt.days

# Create a DataFrame with the relevant columns
dates_df = c_grouped[['last_cbc_datetime', 
                      'persistent_cytoses_datetime', 
                      'aml_mds_mf_date',
                     'cll_date']]

# Compute the minimum date for each row
min_dates = dates_df.min(axis=1)

# Determine the event type using vectorized comparison
event_cytoses = np.select(
    [
        c_grouped['last_cbc_datetime'] == min_dates,
        c_grouped['persistent_cytopenia_datetime'] == min_dates,
        c_grouped['aml_mds_mf_date'] == min_dates,
        c_grouped['cll_date'] == min_dates
    ],
    [0, 1, 2, 3],
    default=np.nan
).astype(int)

# Assign the computed event types to the DataFrame
c_grouped['event_cytoses'] = event_cytoses

In [ ]:
c_grouped = c_grouped[c_grouped['time_cytopenia']>0]

In [ ]:
p_grouped = p_grouped[p_grouped.person_id.isin(set(c_grouped.person_id))]

In [ ]:
# Time to malignancy, censoring (cutoff date), or competing risk (death)
c_grouped['time_malignancy'] = (c_grouped[['cutoff_date',
                                           'aml_mds_mf_date', 
                                           'cll_date',
                                           'death_date']].min(axis=1) - c_grouped['index_datetime']).dt.days

# Create a DataFrame with the relevant columns
dates_df = c_grouped[['cutoff_date', 'aml_mds_mf_date', 'cll_date', 'death_date']]

# Compute the minimum date for each row
min_dates = dates_df.min(axis=1)

# Determine the event type using vectorized comparison
event_malignancy = np.select(
    [
        c_grouped['cutoff_date'] == min_dates,
        c_grouped['aml_mds_mf_date'] == min_dates,
        c_grouped['cll_date'] == min_dates,
        c_grouped['death_date'] == min_dates
    ],
    [0, 1, 2, 3],
    default=np.nan
).astype(int)

# Assign the computed event types to the DataFrame
c_grouped['event_malignancy'] = event_malignancy

In [ ]:
# Function to determine gene_class for each group
def determine_gene_class(genes):
    
    # Define gene types
    high_risk_mcas = ['chr6 Loss', 'chr6q Loss', 
                      'chr11 Loss', 'chr11q Loss',
                      'chr13 Loss', 'chr13q Loss', 
                      'chr17 Loss', 'chr17p Loss', 'chr17q Loss',
                      'chr12 Gain', 'chr12p Gain', 'chr12q Gain',
                      'chr13 CN-LOH', 'chr13q CN-LOH']
    
    lymphoid_mcas = ['chr10 Loss', 'chr10p Loss', 'chr10q Loss',
                     'chr11 Loss', 'chr11q Loss', 
                     'chr13 Loss', 'chr13q Loss',
                     'chr14 Loss', 'chr14q Loss',
                     'chr15 Loss', 'chr15q Loss',
                     'chr17 Loss', 'chr17p Loss',
                     'chr1 Loss', 'chr1p Loss', 'chr1q Loss',
                     'chr22 Loss', 'chr22q Loss',
                     'chr6 Loss', 'chr6q Loss',
                     'chr7 Loss', 'chr7q Loss',
                     'chr8 Loss', 'chr8p Loss',
                     'chr12 Gain', 'chr12q Gain',
                     'chr15 Gain', 'chr15q Gain',
                     'chr17 Gain', 'chr17q Gain',
                     'chr22 Gain', 'chr22q Gain',
                     'chr2 Gain', 'chr2p Gain',
                     'chr3 Gain', 'chr3q Gain',
                     'chr8 Gain', 'chr8q Gain',
                     'chr9 Gain', 'chr9q Gain',
                     'chr16 CN-LOH', 'chr16p CN-LOH',
                     'chr1 CN-LOH', 'chr1q CN-LOH',
                     'chr7 CN-LOH', 'chr7q CN-LOH',
                     'chr13 CN-LOH', 'chr13q CN-LOH',
                     'chr12 CN-LOH', 'chr12q CN-LOH',
                     'chr9 CN-LOH', 'chr9q CN-LOH',
                     'chr18 Gain',
                     'chr19 Gain']

    myeloid_mcas = ['chr12q Loss', 'chr12 Loss',
                    'chr20q Loss', 'chr20 Loss',
                    'chr5 Loss', 'chr5q Loss',
                    'chr1 Gain', 'chr1q Gain', 
                    'chr9 Gain', 'chr9p Gain',
                    'chr22 CN-LOH', 'chr22q CN-LOH',
                    'chr9 CN-LOH', 'chr9p CN-LOH', 
                    'chr14 CN-LOH', 'chr14q CN-LOH',
                    'chr8 Gain']
    
    a_mcas = ['chr21 Gain', 'chr21q Gain',
              'chr11 CN-LOH', 'chr11 CN-LOH',
              'chr16 CN-LOH', 'chr16 CN-LOH',
              'chr1 CN-LOH', 'chr1p CN-LOH',
              'chr17 CN-LOH', 'chr17p CN-LOH']
        
    if genes.isna().all():
        return 'reference'
    
    unique_genes = genes.dropna().unique()
    
    if len(unique_genes) == 0:
        return 'reference'
    
    elif len(set(unique_genes).intersection(set(myeloid_mcas)))>0 and len(set(unique_genes).intersection(set(lymphoid_mcas)))>0:
        return 'a_mca'
    
    elif len(unique_genes) > 1:
        return 'multiple'
    
    else:
        gene = unique_genes[0]
        if gene in high_risk_mcas:
            return 'high_risk'
        elif gene in myeloid_mcas:
            return 'm_mca'
        elif gene in lymphoid_mcas:
            return 'l_mca'
        elif gene in a_mcas:
            return 'a_mca'
        else:
            return 'other'

In [ ]:
c_grouped['incident_aml_mds_mf'] = c_grouped['aml_mds_mf_date'].notna().astype(int)
c_grouped['incident_cll'] = c_grouped['cll_date'].notna().astype(int)
c_grouped['death'] = c_grouped['death_date'].notna().astype(int)

# Create age variables
# Groups: < 60, 60-80, > 80
c_grouped['age_over_65'] = c_grouped['age'].apply(lambda x: 1 if x >= 65 else 0)
c_grouped['age_group'] = pd.cut(c_grouped['age'], bins=[0, 60, 80, np.inf], labels=[0, 1, 2]).astype(int)

c_grouped['time_cytopenia_years'] = (c_grouped['time_cytopenia'] / 365.25).round(2)
c_grouped['time_cytoses_years'] = (c_grouped['time_cytoses'] / 365.25).round(2)
c_grouped['time_malignancy_years'] = (c_grouped['time_malignancy'] / 365.25).round(2)

# Create MCV and RDW group
c_grouped['mcv_100'] = c_grouped['mcv'].apply(lambda x: 1 if x >= 100 else 0)
c_grouped['rdw_15'] = c_grouped['rdw'].apply(lambda x: 1 if x >= 15 else 0)

# Gender as integer construct
c_grouped['gender_num'] = c_grouped['gender'].map({'female': 0, 'male': 1, 'other': 2})
c_grouped['gender_bin'] = c_grouped['gender'].map({'female': 0, 'male': 1, 'other': np.nan})

# Create VAF columns based on the specified ranges
c_grouped['VAF_10'] = (c_grouped['VAF'] >= 0.10).astype(int)

c_grouped['persistent_cytopenia_count'] = c_grouped['persistent_anemia'] + c_grouped['persistent_thrombocytopenia'] + c_grouped['persistent_leukopenia']

c_grouped['persistent_cytoses_count'] = c_grouped['persistent_polycythemia'] + c_grouped['persistent_thrombocytoses'] + c_grouped['persistent_leukocytoses']


# # Define the gene group mapping
# gene_group_mapping = {
#     'DNMT3A': 'DNMT3A',
#     'TET2': 'TET2',
#     'ASXL1': 'ASXL1',
#     'TP53': 'TP53_PPM1D',
#     'PPM1D': 'TP53_PPM1D',
#     'IDH1': 'IDH1_IDH2',
#     'IDH2': 'IDH1_IDH2',
#     'SF3B1': 'SF3B1_SRSF2_U2AF1_ZRSR2',
#     'SRSF2': 'SF3B1_SRSF2_U2AF1_ZRSR2',
#     'U2AF1': 'SF3B1_SRSF2_U2AF1_ZRSR2',
#     'ZRSR2': 'SF3B1_SRSF2_U2AF1_ZRSR2',
#     'JAK2': 'JAK2'
# }

# high_risk_genes_chrs = ['SRSF2', 'SF3B1', 'ZRSR2', 'IDH1', 'IDH2', 'FLT3', 'RUNX1', 'JAK2']

# high_risk_genes = ['SF3B1', 'SRSF2', 'U2AF1', 'ZRSR2', 'IDH1', 'IDH2', 'TP53', 'PPM1D']

# Map the genes to their groups
# c_grouped['gene_group'] = c_grouped['gene'].map(gene_group_mapping).fillna('reference').astype('string')
c_grouped['gene_group'] = c_grouped['gene'].fillna('reference').astype('string')


# Determine 'gene_class' for each group
c_grouped['gene_class'] = c_grouped.groupby('person_id')['gene'].transform(determine_gene_class).astype('string')

# # Map gene_class to gene_class_num
# gene_class_to_num = {
#     'reference': 0,
#     'DNMT3A': 1,
#     'TET2': 2,
#     'ASXL1': 3,
#     'JAK2': 4,
#     'TP53_or_PPM1D': 5,
#     'IDH1_or_IDH2': 6,
#     'SF3B1_or_SRSF2_or_U2AF1_or_ZRSR2': 7,
#     'other': 8,
#     'multiple': 9
# }

# Map gene_class to gene_class_num
gene_class_to_num = {
    'reference': 0,
    'high_risk': 1,
    'l_mca': 2,
    'm_mca': 3,
    'a_mca': 4,
    'multiple': 5,
    'other': 6,
}


high_risk_mcas = ['chr6 Loss', 'chr6q Loss', 
                  'chr11 Loss', 'chr11q Loss',
                  'chr13 Loss', 'chr13q Loss', 
                  'chr17 Loss', 'chr17p Loss', 'chr17q Loss',
                  'chr12 Gain', 'chr12p Gain', 'chr12q Gain',
                  'chr13 CN-LOH', 'chr13q CN-LOH']

c_grouped['gene_class_num'] = c_grouped['gene_class'].map(gene_class_to_num)

# Create the 'high_risk_genes' column
c_grouped['high_risk_genes'] = c_grouped['gene'].isin(high_risk_mcas).astype(int)
# c_grouped['high_risk_genes_chrs'] = c_grouped['gene'].isin(high_risk_mcas).astype(int)

# Filter out rows where 'gene' is NaN
c_non_na = c_grouped.dropna(subset=['gene'])

# Calculate the number of mutations for each person_id
mutation_counts = c_non_na.groupby('person_id')['gene'].size().reset_index(name='mcas')

# Merge the mutation_counts back into the original dataframe
c_grouped = c_grouped.merge(mutation_counts, on='person_id', how='left')
c_grouped

In [ ]:
c_grouped.columns

In [ ]:
c_grouped['mcas'] = c_grouped['mcas'].fillna(0)
c_grouped['mcas'] = c_grouped['mcas'].astype(int)

c_grouped['mcas_two_or_more'] = (c_grouped['mcas'] >= 2).astype(int)

# # Calculate the number of DNMT3A mutations for each person_id
# DNMT3A_counts = c_grouped[c_grouped['gene'] == 'DNMT3A'].groupby('person_id').size().reset_index(name='DNMT3A_count')

# # Merge the DNMT3A_counts back into the original dataframe
# c_grouped = c_grouped.merge(DNMT3A_counts, on='person_id', how='left').fillna({'DNMT3A_count': 0})

# Create a new column 'chip_mutation_group' based on 'chip_mutations'
c_grouped['mca_mutations_group'] = c_grouped['mcas'].apply(lambda x: 0 if x == 0 else (1 if x == 1 else 2))

# # Create a new column 'chip_mutations_DNMT3A' based on the conditions
# c_grouped['chip_mutations_DNMT3A'] = c_grouped.apply(
#     lambda row: '>=2 DNMT3A' if row['DNMT3A_count'] >= 2 else ('0' if row['chip_mutations'] == 0 else ('1' if row['chip_mutations'] == 1 else '>=2')), axis=1
# ).astype('string')

# # Drop the 'DNMT3A_count' column as it is no longer needed
# c_grouped.drop(columns=['DNMT3A_count'], inplace=True)

c_grouped['cohort_num'] = c_grouped['cohort'].map({'aou': 0, 'biovu': 1, 'ukb': 2})


########################################


survival_columns = ['person_id', 'mca', 'case', 'time_cytopenia', 'time_cytoses', 'time_malignancy', 
                    'event_cytopenia', 'event_cytoses', 'event_malignancy', 'gender', 'mcas', 
                    'mca_mutations_group', 
#                      'persistent_anemia', 'persistent_leukopenia',
#        'persistent_thrombocytopenia', 'persistent_polycythemia',
#        'persistent_leukocytoses', 'persistent_thrombocytoses',
                    'age_group', 'VAF_10', 
                    'gene_group', 'gene_class', 'gene_class_num', 'high_risk_genes', 
                    'rdw_15', 'mcv_100', 'ever_smoker', 'cohort', 
                    'cohort_num']

case_control_survival_df = c_grouped[survival_columns]

# case_control_survival_df.loc[:, 'chip_mutations_DNMT3A'] = case_control_survival_df.loc[:, 'chip_mutations_DNMT3A'].replace(0, '0')
# case_control_survival_df.loc[:, 'chip_mutations_DNMT3A'] = case_control_survival_df.loc[:, 'chip_mutations_DNMT3A'].replace(1, '1')


########################################


chip_enrollment = c_grouped[(c_grouped['mca']==1)]

# Select relevant columns for survival analysis
survival_df = chip_enrollment[['person_id', 'mca', 'time_cytopenia', 'time_cytoses', 'time_malignancy', 
                               'event_cytopenia', 'event_cytoses', 'event_malignancy', 
                               'age_over_65', 'age_group', 
                               'gender_num', 'gender_bin', 'ever_smoker', 'gene_class_num', 
                               'high_risk_genes', 'VAF_10', 'mcas', 
                               'mcas_two_or_more', 'mca_mutations_group', 'mcv_100', 
                               'rdw_15', 'cohort', 'cohort_num']].drop_duplicates('person_id').reset_index(drop=True)

In [ ]:
# assert check_column_types(c_grouped, {'person_id': 'string',
#                                       'birth_datetime': 'datetime64[ns, UTC]',
#                                       'death_date': 'datetime64[ns, UTC]',
#                                       'index_datetime': 'datetime64[ns, UTC]', 
#                                       'gender': 'string',
#                                       'race': 'string',
#                                       'ethnicity': 'string',
#                                       'ever_smoker': 'int',
#                                       'age': 'float64',
#                                       'mca': 'int',
#                                       'prior_aml_mds_mf': 'int',
#                                       'prior_cll': 'int',
#                                       'case': 'int',
#                                       'gene': 'string',
#                                       'VAF': 'float64',
#                                       'cytopenia_at_enrollment': 'int',
#                                       'cytoses_at_enrollment': 'int',
#                                       'hgb': 'float64',
#                                       'wbc': 'float64',
#                                       'plt': 'float64',
#                                       'mcv': 'float64',
#                                       'rdw': 'float64',
#                                       'persistent_anemia': 'int',
#                                       'persistent_leukopenia': 'int',
#                                       'persistent_thrombocytopenia': 'int',
#                                       'persistent_cytopenia': 'int',
#                                       'persistent_cytopenia_datetime': 'datetime64[ns, UTC]',
#                                       'persistent_polycythemia': 'int',
#                                       'persistent_leukocytoses': 'int',
#                                       'persistent_thrombocytoses': 'int',
#                                       'persistent_cytoses': 'int',
#                                       'persistent_cytoses_datetime': 'datetime64[ns, UTC]',
#                                       'first_cbc_datetime': 'datetime64[ns, UTC]',
#                                       'last_cbc_datetime': 'datetime64[ns, UTC]',
#                                       'cbc_count': 'int',
#                                       'cbc_count_post_index': 'int',
#                                       'persistent_abnormal_blood_counts': 'int',
#                                       'persistent_abnormal_counts_datetime': 'datetime64[ns, UTC]',
#                                       'aml_date': 'datetime64[ns, UTC]',
#                                       'mds_date': 'datetime64[ns, UTC]',
#                                       'mf_date': 'datetime64[ns, UTC]',
#                                       'et_date': 'datetime64[ns, UTC]',
#                                       'pv_date': 'datetime64[ns, UTC]', 
#                                       'first_aml_mds_mf': 'string', 
#                                       'aml_mds_mf_date': 'datetime64[ns, UTC]',
#                                       'first_cll': 'string',
#                                       'cll_date': 'datetime64[ns, UTC]',
#                                       'time_cytopenia': 'int',
#                                       'time_cytoses': 'int',
#                                       'time_malignancy': 'int',
#                                       'event_cytopenia': 'int',
#                                       'event_cytoses': 'int',
#                                       'event_malignancy': 'int', 
#                                       'cohort': 'string',
#                                       'cohort_num': 'int',
#                                       'cutoff_date': 'datetime64[ns, UTC]',
#                                       'incident_aml_mds_mf': 'int',
#                                       'incident_cll':'int',
#                                       'death': 'int',
#                                       'age_over_65': 'int', 
#                                       'age_group': 'int', 
#                                       'time_cytopenia_years': 'float64',
#                                       'time_cytoses_years': 'float64',
#                                       'time_malignancy_years': 'float64',
#                                       'mcv_100': 'int', 
#                                       'rdw_15': 'int', 
#                                       'gender_num': 'int',
#                                       'gender_bin': 'float64',
#                                       'VAF_20': 'int', 
#                                       'persistent_cytopenia_count': 'int', 
#                                       'persistent_cytoses_count': 'int',
#                                       'gene_group': 'string', 
#                                       'gene_class': 'string', 
#                                       'gene_class_num': 'int', 
#                                       'high_risk_genes': 'int',
#                                       'mcas': 'int', 
#                                       'mcas_two_or_more': 'int', 
#                                       'mca_mutations_group': 'int'})
# #                                       'chip_mutations_DNMT3A': 'string'}

# assert check_column_types(p_grouped, {'person_id': 'string', 
#                                       'birth_datetime': 'datetime64[ns, UTC]',
#                                       'gender': 'string',
#                                       'race': 'string',
#                                       'ethnicity': 'string',
#                                       'ever_smoker': 'int',
#                                       'age': 'float64',
#                                       'mca': 'int',
#                                       'cytopenia_at_enrollment': 'int',
#                                       'cytoses_at_enrollment': 'int',
#                                       'cbc_count': 'int',
#                                       'cbc_count_post_index': 'int', 
#                                       'prior_aml_mds_mf': 'int', 
#                                       'prior_cll': 'int', 
#                                       'eligible': 'int', 
#                                       'case': 'int', 
#                                       'cohort': 'string'})

save_to_bucket(c_grouped, file_path, f'{cohort_name}_cohort')
save_to_bucket(p_grouped, file_path, f'{cohort_name}_participants')
save_to_bucket(case_control_survival_df, file_path, f'{cohort_name}_case_control_survival')
save_to_bucket(survival_df, file_path, f'{cohort_name}_chip_survival')

In [ ]:
# !gsutil cp {BUCKET}/{file_path}/'all_case_control_survival.csv' broganjf/chip_cytopenia/
# !gsutil cp {BUCKET}/{file_path}/'all_cohort.csv' broganjf/chip_cytopenia/

## Figure 1; Supplemental Figure 1A, 1B, 1C; Cohort overview statistics 
Flow diagrams for cohort enrollment

In [ ]:
def criteria_summary(df, criteria, cohorts=None):
    def apply_criteria(df, criterion):
        # Apply specific criteria to the dataframe and return the count of unique person_id
        if criterion == 'n_sequenced':
            return df['person_id'].nunique()
        elif criterion == 'n_prior_ca':
            return df[df['prior_aml_mds_mf'] == 1]['person_id'].nunique()
        elif criterion == 'n_insufficient_cbc':
            return df[(df['prior_aml_mds_mf'] == 0) & 
                      (df['cbc_count'] < 3) & 
                      (df['cbc_count_post_index'] < 1)]['person_id'].nunique()
        elif criterion == 'n_potential':
            return df[(df['prior_aml_mds_mf'] == 0) & 
                      (df['cbc_count'] >= 3) & 
                      (df['cbc_count_post_index'] >= 1)]['person_id'].nunique()
        elif criterion == 'n_enrollment_cytopenia':
            return df[(df['prior_aml_mds_mf'] == 0) & 
                      (df['cytopenia_at_enrollment'] == 1)]['person_id'].nunique()
        elif criterion == 'n_eligible':
            return df[df['eligible'] == 1]['person_id'].nunique()
        elif criterion == 'n_eligible_mca':
            return df[(df['eligible'] == 1) & (df['mca'] == 1)]['person_id'].nunique()
        elif criterion == 'n_eligible_no_mca':
            return df[(df['eligible'] == 1) & (df['mca'] == 0)]['person_id'].nunique()
        elif criterion == 'n_cases':
            return df[df['case'] == 1]['person_id'].nunique()
        elif criterion == 'n_controls':
            return df[df['case'] == 0]['person_id'].nunique()
        else:
            raise ValueError(f"Unknown criterion: {criterion}")

    if cohorts:
        if 'cohort' not in df.columns:
            raise ValueError("The dataframe does not have a 'cohort' column.")
        available_cohorts = df['cohort'].unique()
        if not set(cohorts).issubset(available_cohorts):
            raise ValueError(f"Some specified cohorts are not in the DataFrame. Available cohorts: {available_cohorts}")

    # Create a dictionary to hold the results
    results = {'criteria': criteria,
               'all_cohorts': []}
    if cohorts:
        results.update({cohort: [] for cohort in cohorts})

    # Calculate counts for all cohorts
    counts = {}
    for criterion in results['criteria']:
        counts[criterion] = apply_criteria(df, criterion)
        results['all_cohorts'].append(counts[criterion])
        
    # Check assertions
    assert counts['n_sequenced'] == counts['n_prior_ca'] + counts['n_insufficient_cbc'] + counts['n_enrollment_cytopenia'] + counts['n_eligible'], "Assertion failed: n_sequenced does not match the sum of n_prior_ca, n_insufficient_cbc, n_enrollment_cytopenia, and n_eligible"
    assert counts['n_eligible'] == counts['n_eligible_mca'] + counts['n_eligible_no_mca'], "Assertion failed: n_eligible does not match the sum of n_eligible_chip and n_eligible_no_chip"
    
    if cohorts:
        for cohort in cohorts:
            cohort_df = df[df['cohort'] == cohort]
            for criterion in results['criteria']:
                count = apply_criteria(cohort_df, criterion)
                results[cohort].append(count)

    result_df = pd.DataFrame(results)
    return result_df


def cohort_age_summary(df):
    """
    Computes the median, 25th and 75th percentiles, min, and max age for unique person_id in each cohort.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing 'person_id', 'cohort', and 'age' columns.

    Returns:
        pd.DataFrame: Summary statistics by cohort.
    """
    # Ensure only unique person_id per cohort
    df_unique = df.drop_duplicates(subset=['person_id', 'cohort'])

    # Group by cohort and calculate the required statistics
    summary = df_unique.groupby('cohort')['age'].agg(
        median_age='median',
        age_25th_percentile=lambda x: x.quantile(0.25),
        age_75th_percentile=lambda x: x.quantile(0.75),
        min_age='min',
        max_age='max'
    ).reset_index()

    return summary

In [ ]:
criteria = ['n_sequenced', 
            'n_prior_ca', 
            'n_insufficient_cbc', 
            # 'n_potential', 
            'n_enrollment_cytopenia', 
            'n_eligible', 
            'n_eligible_mca', 
            'n_eligible_no_mca', 
            'n_cases', 
            'n_controls']

result = criteria_summary(p_grouped, criteria, cohorts=['aou', 
#                                                         'biovu', 
                                                        'ukb'])
# result = criteria_summary(p_grouped, criteria, cohorts=['aou'])

result

In [ ]:
cohort_age_summary(p_grouped)

In [ ]:
np.median(c_grouped[c_grouped['mca']==1].time_cytoses)

In [ ]:
np.median(c_grouped[c_grouped['mca']==0].time_cytoses)

In [ ]:
case_control_survival_df

## Table 1 and Supplemental Table 1A, 1B, 1C

In [ ]:
cases = c_grouped[c_grouped['case']==1]
controls = c_grouped[c_grouped['case']==0]

aou_cases = c_grouped[(c_grouped['case']==1) & (c_grouped['cohort']=='aou')]
aou_controls = c_grouped[(c_grouped['case']==0) & (c_grouped['cohort']=='aou')]

# biovu_cases = c_grouped[(c_grouped['case']==1) & (c_grouped['cohort']=='biovu')]
# biovu_controls = c_grouped[(c_grouped['case']==0) & (c_grouped['cohort']=='biovu')]

ukb_cases = c_grouped[(c_grouped['case']==1) & (c_grouped['cohort']=='ukb')]
ukb_controls = c_grouped[(c_grouped['case']==0) & (c_grouped['cohort']=='ukb')]

cohorts_dict = {'cases': cases, 'controls': controls}

aou_dict = {'cases': aou_cases, 'controls': aou_controls}
# biovu_dict = {'cases': biovu_cases, 'controls': biovu_controls}
ukb_dict = {'cases': ukb_cases, 'controls': ukb_controls}


table_one_map = {
    'person_id': ['id', 'unique'],
    'age': ['continuous', 'median'], 
    'gender': ['categorical', 'count'], 
    'ever_smoker': ['categorical', 'count'],
    'hgb': ['continuous', 'median'],
    'plt': ['continuous', 'median'],
    'wbc': ['continuous', 'median'],
    'mcv': ['continuous', 'median'],
    'rdw': ['continuous', 'median'],
    'persistent_cytopenia_count': ['categorical', 'count'],
    'persistent_anemia': ['categorical', 'count'],
    'persistent_thrombocytopenia': ['categorical', 'count'],
    'persistent_leukopenia': ['categorical', 'count'],
    'persistent_polycythemia': ['categorical', 'count'],
    'persistent_thrombocytoses': ['categorical', 'count'],
    'persistent_leukocytoses': ['categorical', 'count'],
    'persistent_cytoses_count': ['categorical', 'count'],
}

In [ ]:
compute_statistics(cohorts_dict, table_one_map)

In [ ]:
compute_incidence_rate(cases, 'persistent_cytopenia', 'time_cytopenia_years'), compute_incidence_rate(controls, 'persistent_cytopenia', 'time_cytopenia_years')

In [ ]:
compute_incidence_rate(cases, 'persistent_cytoses', 'time_cytoses_years'), compute_incidence_rate(controls, 'persistent_cytoses', 'time_cytoses_years')

In [ ]:
compute_statistics(aou_dict, table_one_map)

In [ ]:
compute_incidence_rate(aou_cases, 'persistent_cytopenia', 'time_cytopenia_years'), compute_incidence_rate(aou_controls, 'persistent_cytopenia', 'time_cytopenia_years')

In [ ]:
compute_incidence_rate(aou_cases, 'persistent_cytoses', 'time_cytoses_years'), compute_incidence_rate(aou_controls, 'persistent_cytopenia', 'time_cytopenia_years')

In [ ]:
# compute_statistics(biovu_dict, table_one_map)

In [ ]:
# compute_incidence_rate(biovu_cases, 'persistent_cytopenia', 'time_cytopenia_years'), compute_incidence_rate(biovu_controls, 'persistent_cytopenia', 'time_cytopenia_years')

In [ ]:
compute_statistics(ukb_dict, table_one_map)

In [ ]:
compute_incidence_rate(ukb_cases, 'persistent_cytopenia', 'time_cytopenia_years'), compute_incidence_rate(ukb_controls, 'persistent_cytopenia', 'time_cytopenia_years')

In [ ]:
compute_incidence_rate(ukb_cases, 'persistent_cytoses', 'time_cytoses_years'), compute_incidence_rate(ukb_controls, 'persistent_cytopenia', 'time_cytopenia_years')

## Figure 2
Risk of incident cytopenia and cumulative incidence of cytopenia at 2 years

### Figure 2A

In [ ]:
plot_cumulative_incidence(case_control_survival_df, 
                          'case',
                          'time_cytopenia',
                          'event_cytopenia',
                          f'Figure 2A. Cumulative incidence of cytopenia in all cases and controls', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'no mCA', 1: '≥ 1 mCAs'},
                          {0: 'black', 1: 'red'},
                          f'{cohort_name}_ci_case_control_cytopenia_by_mca_presence.pdf',
                          x_max = 7,
                          y_max = 0.3)

In [ ]:
plot_cumulative_incidence(case_control_survival_df[case_control_survival_df['time_cytoses']>0], 
                          'case',
                          'time_cytoses',
                          'event_cytoses',
                          f'Figure 2A. Cumulative incidence of cytoses in all cases and controls', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'no mCA', 1: '≥ 1 mCAs'},
                          {0: 'black', 1: 'red'},
                          f'{cohort_name}_ci_case_control_cytoses_by_mca_presence.svg',
                          x_max = 7,
                          y_max = 0.3)

### Figure 2B

In [ ]:
df_hr_ci = analyze_cohorts(c_grouped, 'time_cytopenia', 'event_cytopenia', 2, unit_of_time)

In [ ]:
df_hr_ci

In [ ]:
df_hr_ci = analyze_cohorts(c_grouped, 'time_cytoses', 'event_cytoses', 2, unit_of_time)

In [ ]:
df_hr_ci

In [ ]:
# # Define the dictionary
# hr_ci_dict = {
#     'cohort': ['all', 'aou', 'ukb'],
#     'hazard_ratio': [1.76, 1.77, 3.53],
#     'lower_95_CI': [1.32, 1.32, 0.94],
#     'upper_95_CI': [2.32, 2.36, 13.14],
#     'p_value': [0.000098, 0.000131, 0.060278],
#     'ci_cases_2_years': [0.0858, 0.1488, 0.0000],
#     'ci_controls_2_years': [0.0490, 0.0848, 0.0011]
# }

# # Convert the dictionary into a DataFrame
# hr_ci = pd.DataFrame(hr_ci_dict)

In [ ]:
# custom_order = ['all', 'ukb', 'aou']
# cohort_mapping = {'aou': 'All of Us', 'ukb': 'UK Biobank', 'all': 'Composite'}

# forest_plot(hr_ci, 
#             'Figure 2B', 
#             f'{cohort_name}_hr_ci.pdf', 
#             cohort_order=custom_order, 
#             cohort_map=cohort_mapping)

### Figure 2C

In [ ]:
c_grouped.columns

In [ ]:
case_control_survival_df

In [ ]:
c_grouped[c_grouped['person_id']=='u3708027']

In [ ]:
case_control_survival_df

In [ ]:
test = case_control_survival_df.merge(c_grouped[['person_id', 'persistent_anemia', 'persistent_leukopenia',
       'persistent_thrombocytopenia', 'persistent_polycythemia',
       'persistent_leukocytoses', 'persistent_thrombocytoses']].drop_duplicates(),
                               on='person_id',
                               how='inner')

In [ ]:
control_variables = []
duration = 'time_cytopenia'
event = 'event_cytopenia'

In [ ]:
# gene_forest_label = {'gene_class_num_1': 'DNMT3A', 'gene_class_num_2': 'TET2','gene_class_num_3': 'ASXL1', 
#                      'gene_class_num_4': 'JAK2', 'gene_class_num_5': 'TP53 or PPM1D', 'gene_class_num_6': 'IDH1 or IDH2',
#                      'gene_class_num_7': 'Spliceosome genes', 'gene_class_num_8': 'Other', 'gene_class_num_9': '≥ 2 mutations'}
gene_forest_label = {'gene_class_num_1': 'high_risk', 
                     'gene_class_num_2': 'l_mca',
                     'gene_class_num_3': 'm_mca',
                     'gene_class_num_4': 'a_mca',
                     'gene_class_num_5': 'multiple',
                     'gene_class_num_6': 'other'}

In [ ]:
Counter(case_control_survival_df['gene_group'])

In [ ]:
def create_forest_plot(df, 
                       variables_of_interest, 
                       control_variables, 
                       duration_col, 
                       event_col,
                       label_dict, 
                       file_path, 
                       title, 
                       cols,
                       counts=False, 
                       sort='count', 
                       reference_dict={},
                       MIN_NUM = 10):
    hr_list, ci_lower_list, ci_upper_list = [], [], []
    p_values_list, labels, n_counts = [], [], []
    mca_list = []
    max_col_list = []

    for var in variables_of_interest:
        # Identify reference category
        reference_mask = df[var].str.contains('reference', case=False, na=False)
        reference_category = df.loc[reference_mask, var].iloc[0] if any(reference_mask) else None
        
        if reference_category is None:
            print(f"Warning: No reference category found for {var}")
            continue
            
        # Create dummy variables but don't drop first category
        dummies = pd.get_dummies(df[var], prefix=var)
        ref_col = [col for col in dummies.columns if reference_category in col][0]
        
        # Drop reference column after identifying it
        non_ref_cols = [col for col in dummies.columns if col != ref_col]
        dummies = dummies[non_ref_cols]
        df = pd.concat([df, dummies], axis=1)
        
        for dummy_var in dummies.columns:
            n_unique = df[df[dummy_var]==1]['person_id'].nunique()
            if n_unique < MIN_NUM:
                print(f"skipping {dummy_var}")
                continue
            
            all_vars = [dummy_var] + control_variables + [duration_col, event_col]
            label = label_dict.get(dummy_var, f'{var}={dummy_var.split("_")[-1]} vs Reference')
            mca = dummy_var.split("_")[-1]
            if counts:
                n_counts.append(n_unique)
                label = f'{label} (N={n_unique})'
            else:
                n_counts.append(None)

            labels.append(label)

            cph = CoxPHFitter()
            cph.fit(df[all_vars], duration_col=duration_col, event_col=event_col)
            summary = cph.summary
            
            col_sums = df[df[dummy_var]==1][cols].sum()
            max_col = col_sums.idxmax()

            hr = summary.loc[dummy_var, 'exp(coef)']
            ci_lower = summary.loc[dummy_var, 'exp(coef) lower 95%']
            ci_upper = summary.loc[dummy_var, 'exp(coef) upper 95%']
            p_value = summary.loc[dummy_var, 'p']

            hr_list.append(hr)
            ci_lower_list.append(ci_lower)
            ci_upper_list.append(ci_upper)
            p_values_list.append(p_value)
            mca_list.append(mca)
            max_col_list.append(max_col)
    
    if reference_dict:
        hr_list.append(reference_dict['hr'])
        n_counts.append(reference_dict['count'])
        labels.append(f'{reference_dict["label"]} (N={reference_dict["count"]})')
        ci_lower_list.append(np.nan)
        ci_upper_list.append(np.nan)
        p_values_list.append(np.nan)
        mca_list.append('reference')
        max_col_list.append('')
        
    # Create a DataFrame for the results
    results_df = pd.DataFrame({
        'Variable': mca_list,
        'Label': labels,
        'HR': hr_list,
        'CI Lower': ci_lower_list,
        'CI Upper': ci_upper_list,
        'P Value': p_values_list,
        'Count': n_counts,
        'Driving Cell Line': max_col_list
    })

    # Sort the DataFrame based on the 'sort' parameter
    if sort == 'count' and counts:
        results_df.sort_values(by='Count', ascending=True, inplace=True)
    elif sort == 'alphabetical':
        results_df.sort_values(by='Label', ascending=True, inplace=True)

    # Prepare the data for the plot
    hr = results_df['HR']
    ci_lower = results_df['CI Lower']
    ci_upper = results_df['CI Upper']
    p_values = results_df['P Value']
    labels = results_df['Label']
    
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)

    
    ax.tick_params(top=False,
                   bottom=False,
                   left=False,
                   right=False,
                   labelleft=True,
                   labelbottom=True)

    # Set x-axis to be logarithmic
    ax.set_xlabel('Hazard Ratio')

    
    # Remove all spines except the bottom one
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Remove all grid lines
    ax.grid(False)

    # Calculate the position for text labels
    text_x_position = max(ci_upper) * 1.1

    # Add text beside each point
    for i, (h, cl, cu, p) in enumerate(zip(hr, ci_lower, ci_upper, p_values)):
        if np.isnan(p):
            ax.text(text_x_position, i, 'Reference', verticalalignment='center', fontsize=10)
        else:
            text = f'{h:.2f} [{cl:.2f}, {cu:.2f}], p = {p:.2e}'
            ax.text(text_x_position, i, text, verticalalignment='center', fontsize=10)

    ax.text(0, len(hr), title, fontsize=10, fontweight='bold', va='bottom', ha='left')
        
    # Save the plot as a PDF file
#     plt.savefig(file_path, format='svg', bbox_inches='tight')
    
    # Show plot
    plt.tight_layout()
    plt.show()
    
    return results_df

In [ ]:
control_variables = []
duration = 'time_cytopenia'
event = 'event_cytopenia'

cytopenia_df = create_forest_plot(test,
                   ['gene_group'], 
                   control_variables,
                   duration, 
                   event, 
                   gene_forest_label, 
                   'all_gene_forest_plot.pdf', 
                   title='Figure 2C', 
                   cols=['persistent_anemia', 'persistent_thrombocytopenia', 'persistent_leukopenia'],
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'gene_group_reference', 'hr': 1, 'count': 2386}
                  )

In [ ]:
cytopenia_df

In [ ]:
control_variables = []
duration = 'time_cytoses'
event = 'event_cytoses'
cytoses_df = create_forest_plot(test,
#     case_control_survival_df, 
                   ['gene_group'], 
                   control_variables,
                   duration, 
                   event, 
                   gene_forest_label, 
                   'all_gene_forest_plot.pdf', 
                   title='Figure 2C', 
                   cols=['persistent_polycythemia', 'persistent_thrombocytoses', 'persistent_leukocytoses'],
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'gene_group_reference', 'hr': 1, 'count': 2386}
                  )

In [ ]:
cytopenia_df

In [ ]:
cytoses_df

In [ ]:
cell_line_label_dict = {
    'persistent_polycythemia': 'R',
    'persistent_thrombocytoses': 'P',
    'persistent_leukocytoses': 'W',
    'persistent_anemia': 'R',
    'persistent_thrombocytopenia': 'P',
    'persistent_leukopenia': 'W'
}

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Prepare data
def create_mca_heatmap(cytopenia_df, cytoses_df, bonferroni_threshold=0.05/43):
    # Combine dataframes
    cytopenia_df['outcome'] = 'Cytopenia'
    cytoses_df['outcome'] = 'Cytoses'
    combined_df = pd.concat([cytopenia_df, cytoses_df])
    combined_df['type'] = combined_df['Variable'].str.split(" ").str[1]
    combined_df['chr'] = combined_df['Variable'].str.split(" ").str[0]
    combined_df['chr'] = combined_df['chr'].str.replace('chr', '').str.replace('p', '').str.replace('q', '')
    
    combined_df = combined_df[combined_df['chr']!='reference']
    combined_df['chr'] = combined_df['chr'].astype(int)
    
    combined_df = combined_df.sort_values(by=['chr', 'type'])
    
    # Pivot data for heatmap
    hr_pivot = combined_df.pivot(index='outcome', columns='Variable', values='HR')
    p_pivot = combined_df.pivot(index='outcome', columns='Variable', values='P Value')
    cell_pivot = combined_df.pivot(index='outcome', columns='Variable', values='Driving Cell Line')

    hr_pivot = hr_pivot[combined_df['Variable'].unique()]
    p_pivot = p_pivot[combined_df['Variable'].unique()]
    cell_pivot = cell_pivot[combined_df['Variable'].unique()]
    
    # Calculate color values (deviation from 1)
    color_values = np.log2(hr_pivot)  # log2 for symmetric visualization
        
#     color_values = hr_pivot
    
    # Create figure
    plt.figure(figsize=(15, 4))
    
    # Create heatmap
    ax = sns.heatmap(color_values, 
                     cmap='RdBu_r',
                     center=0,
#                      annot=hr_pivot.round(2),
                     fmt='.2f',
                     cbar_kws={'label': 'log2(Hazard Ratio)'},
                     linewidths=1)
    
    # Add stars for significant p-values
    for i in range(len(hr_pivot.index)):
        for j in range(len(hr_pivot.columns)):
            if p_pivot.iloc[i, j] < bonferroni_threshold:
                value = color_values.iloc[i, j]
                text_color = 'white' if abs(value) > 1.75 else 'black'
                ax.text(j + 0.5, i + 0.5, cell_line_label_dict[cell_pivot.iloc[i,j]],
                        ha='center', va='center',
                        color=text_color, fontweight='bold')
    
    plt.xlabel('mCA')
    plt.ylabel('')
    # Adjust label rotation 
#     plt.xticks(rotation=45) 
    plt.yticks(rotation=0) # Default is already 0 (90 would be vertical)
    plt.tight_layout()
    
    plt.savefig('cytoses_cytopenia_by_mca_heatmap.svg')
    plt.show()

In [ ]:
# Use function
create_mca_heatmap(cytopenia_df[~cytopenia_df['Variable'].isin(['chr1 CN-LOH', 
                                                                'chr15q Gain',
                                                                'chr20q CN-LOH'])], 
                   cytoses_df[~cytoses_df['Variable'].isin(['chr1 CN-LOH', 
                                                            'chr15q Gain',
                                                           'chr20q CN-LOH'])])

In [ ]:
create_forest_plot(case_control_survival_df[case_control_survival_df['cohort']=='aou'], 
                   ['gene_class_num'], 
                   [],
                   duration, 
                   event,
                   gene_forest_label, 
                   'aou_gene_forest_plot.pdf', 
                   title='Supplement 3A (All of Us)',
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'No CHIP', 'hr': 1, 'count': 5109}
                  )

In [ ]:
# create_forest_plot(case_control_survival_df[case_control_survival_df['cohort']=='biovu'], 
#                    ['gene_class_num'], 
#                    [],
#                    duration, 
#                    event,
#                    gene_forest_label, 
#                    'biovu_gene_forest_plot.pdf', 
#                    title='Supplement 3B (BioVU)',
#                    counts=True, 
#                    sort='count', 
#                    reference_dict={'label': 'No CHIP', 'hr': 1, 'count': 4314}
#                   )

In [ ]:
create_forest_plot(case_control_survival_df[case_control_survival_df['cohort']=='ukb'], 
                   ['gene_class_num'], 
                   [],
                   duration, 
                   event,
                   gene_forest_label, 
                   'ukb_gene_forest_plot.pdf', 
                   title='Supplement 3C (UKB)',
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'No CHIP', 'hr': 1, 'count': 15326}
                  )

In [ ]:
control_variables = []
duration = 'time_cytoses'
event = 'event_cytoses'

# gene_forest_label = {'gene_class_num_1': 'DNMT3A', 'gene_class_num_2': 'TET2','gene_class_num_3': 'ASXL1', 
#                      'gene_class_num_4': 'JAK2', 'gene_class_num_5': 'TP53 or PPM1D', 'gene_class_num_6': 'IDH1 or IDH2',
#                      'gene_class_num_7': 'Spliceosome genes', 'gene_class_num_8': 'Other', 'gene_class_num_9': '≥ 2 mutations'}
gene_forest_label = {'gene_class_num_1': 'high_risk', 
                     'gene_class_num_2': 'l_mca',
                     'gene_class_num_3': 'm_mca',
                     'gene_class_num_4': 'a_mca',
                     'gene_class_num_5': 'multiple',
                     'gene_class_num_6': 'other'}

Counter(case_control_survival_df['gene_class'])

create_forest_plot(case_control_survival_df, 
                   ['gene_class_num'], 
                   control_variables,
                   duration, 
                   event, 
                   gene_forest_label, 
                   'all_gene_forest_plot.pdf', 
                   title='Figure 2C', 
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'No mCA', 'hr': 1, 'count': 24749}
                  )

create_forest_plot(case_control_survival_df[case_control_survival_df['cohort']=='aou'], 
                   ['gene_class_num'], 
                   [],
                   duration, 
                   event,
                   gene_forest_label, 
                   'aou_gene_forest_plot.pdf', 
                   title='Supplement 3A (All of Us)',
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'No CHIP', 'hr': 1, 'count': 5109}
                  )

# create_forest_plot(case_control_survival_df[case_control_survival_df['cohort']=='biovu'], 
#                    ['gene_class_num'], 
#                    [],
#                    duration, 
#                    event,
#                    gene_forest_label, 
#                    'biovu_gene_forest_plot.pdf', 
#                    title='Supplement 3B (BioVU)',
#                    counts=True, 
#                    sort='count', 
#                    reference_dict={'label': 'No CHIP', 'hr': 1, 'count': 4314}
#                   )

create_forest_plot(case_control_survival_df[case_control_survival_df['cohort']=='ukb'], 
                   ['gene_class_num'], 
                   [],
                   duration, 
                   event,
                   gene_forest_label, 
                   'ukb_gene_forest_plot.pdf', 
                   title='Supplement 3C (UKB)',
                   counts=True, 
                   sort='count', 
                   reference_dict={'label': 'No CHIP', 'hr': 1, 'count': 15326}
                  )

### Figure 2D

In [ ]:
mca_forest_variables = ['age_over_65', 'rdw_15', 'mcv_100', 'high_risk_genes',
                         'mcas_two_or_more', 'VAF_10', 'gender_bin', 'ever_smoker']

mca_forest_labels = {'age_over_65_1': 'Age ≥ 65', 'rdw_15_1': 'RDW ≥ 15', 
                      'mcv_100_1': 'MCV ≥ 100', 'high_risk_genes_1': 'High risk genes',
                      'mcas_two_or_more_1': '≥ 2 mutations', 'VAF_10_1': 'VAF ≥ 0.10', 
                      'gender_bin_1.0': 'Male', 'gender_num_2': 'Other gender', 
                      'ever_smoker_1': 'Smoker'}

In [ ]:
event

In [ ]:
x = create_forest_plot(survival_df, mca_forest_variables, control_variables, duration, event, mca_forest_labels, 
                   'all_mca_forest_plot.svg', 'Figure 2D', counts=True)

In [ ]:
x

In [ ]:
x.to_csv('cytoses_forest_plot.tsv', sep='\t', index=False)

In [ ]:
create_forest_plot(survival_df[survival_df['cohort']=='aou'], mca_forest_variables, [], duration, event, 
                   mca_forest_labels, 
                   'aou_mcaforest_plot.pdf', title='Supplement 4A (All of Us)')

In [ ]:
# create_forest_plot(survival_df[survival_df['cohort']=='biovu'], chip_forest_variables, [], duration, event, chip_forest_labels, 
#                    'biovu_chip_forest_plot.pdf', title='Supplement 4B (BioVU)')

In [ ]:
create_forest_plot(survival_df[survival_df['cohort']=='ukb'], mca_forest_variables, [], duration, event, 
                   mca_forest_labels, 
                   'ukb_mcaforest_plot.pdf', title='Supplement 4A (UKB)')

## Figure 3

### Figure 3A

In [ ]:
def add_feature_count_columns(df, columns_list):
    """
    Adds two new columns to the DataFrame:
    1. 'feature_count': Number of columns in columns_list that are == 1 for each row.
    2. 'three_or_more_features': 1 if 'feature_count' >= 3, else 0.

    Parameters:
    df (pd.DataFrame): The input DataFrame.
    columns_list (list): List of column names to check for value == 1.

    Returns:
    pd.DataFrame: The DataFrame with the two new columns added.
    """
    # Ensure all columns in the list are present in the DataFrame
    if not all(col in df.columns for col in columns_list):
        raise ValueError("One or more columns in columns_list are not in the DataFrame")
    
    # Calculate feature_count
    df['feature_count'] = df[columns_list].sum(axis=1)
    
    # Calculate three_or_more_features
    df['three_or_more_features'] = (df['feature_count'] >= 3).astype(int)
    
    return df

def map_feature_count_to_group(count):
    if count == 0:
        return 0
    elif count == 1:
        return 1
    elif count in [2]:
        return 2
    elif count in [3, 4, 5]:
        return 3
    else:
        return 3  # or handle values outside specified ranges if needed

In [ ]:
cases = c_grouped[(c_grouped['time_cytoses']>0) & (c_grouped['case']==1)].copy()

features = ['mcas_two_or_more', 'high_risk_genes', 'VAF_10', 'rdw_15']

cases_features = add_feature_count_columns(cases, features)
# Apply the function to create the new column
cases_features['feature_split_group'] = cases_features['feature_count'].apply(map_feature_count_to_group)

In [ ]:
plot_cumulative_incidence(cases_features, 
                          'feature_split_group',
                          'time_cytoses',
                          'event_cytoses',
                          'Figure 3A: All participants cumulative incidence of cytoses', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: '0 high risk features', 
                           1: '1 high risk features', 
                           2: '2 high risk features', 
                           3: '≥ 3 high risk features'},
                          {0: 'black', 1: 'red', 2: 'gray', 3: 'blue'},
                          'all_ci_case_cytopenia_by_high_risk_feature_groups.pdf',
                          x_max = 7,
                          y_max = 0.7, 
                          ci=False)


# plot_cumulative_incidence(cases_features[cases_features['cohort']=='aou'], 
#                           'feature_split_group',
#                           'time_cytoses',
#                           'event_cytoses',
#                           'Supplement 5A: All of Us cumulative incidence of cytoses', 
#                           unit_of_time,
#                           unit_of_time_label,
#                           {0: '0 high risk features', 
#                            1: '1 high risk features', 
#                            2: '2 or 3 high risk features', 
#                            3: '4 or 5 high risk features'},
#                           {0: 'black', 1: 'red', 2: 'gray', 3: 'blue'},
#                           'aou_ci_case_cytopenia_by_high_risk_feature_groups.pdf',
#                           x_max = 3,
#                           y_max = 0.50,
#                           ci=False)


# # plot_cumulative_incidence(cases_features[cases_features['cohort']=='biovu'], 
# #                           'feature_split_group',
# #                           'time_cytoses',
# #                           'event_cytoses',
# #                           'Supplement 5B: BioVU cumulative incidence of cytoses', 
# #                           unit_of_time,
# #                           unit_of_time_label,
# #                           {0: '0 high risk features', 
# #                            1: '1 high risk features', 
# #                            2: '2 or 3 high risk features', 
# #                            3: '4 or 5 high risk features'},
# #                           {0: 'black', 1: 'red', 2: 'gray', 3: 'blue'},
# #                           'biovu_ci_case_cytopenia_by_high_risk_feature_groups.pdf',
# #                           x_max = 7,
# #                           y_max = 0.70,
# #                           ci=False)


# plot_cumulative_incidence(cases_features[cases_features['cohort']=='ukb'], 
#                           'feature_split_group',
#                           'time_cytopenia',
#                           'event_cytopenia',
#                           'Supplement 5C: UKB cumulative incidence of cytopenia', 
#                           unit_of_time,
#                           unit_of_time_label,
#                           {0: '0 high risk features', 
#                            1: '1 high risk features', 
#                            2: '2 or 3 high risk features', 
#                            3: '4 or 5 high risk features'},
#                           {0: 'black', 1: 'red', 2: 'gray', 3: 'blue'},
#                           'ukb_ci_case_cytopenia_by_high_risk_feature_groups.pdf',
#                           x_max = 7,
#                           y_max = 0.70,
#                           ci=False)

### Figure 3B

In [ ]:
def hrf_to_forest(df, 
                 title, 
                 file_path, 
                 figsize=(14, 6)):

    # Sort the DataFrame by 'hazard_ratio' or any other column that determines the order
    df = df.sort_values(by='group', ascending=False)
    
    # Prepare the data for the plot
    hr = df['hazard_ratio']
    ci_lower = df['ci_lower']
    ci_upper = df['ci_upper']
    ci_p = df['ci_string']
    labels = df['group']

    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)
    
    ax.tick_params(top=False,
                   bottom=False,
                   left=False,
                   right=False,
                   labelleft=True,
                   labelbottom=True)

    # Set x-axis to be linear with custom ticks and a narrower range
    ax.set_xscale('linear')
    ax.set_xlim(0.8, 8)
    ax.set_xticks([1, 3, 5, 8])
    ax.set_xlabel('Hazard Ratio')

    # Remove all spines except the bottom one
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    # Remove all grid lines
    ax.grid(False)

    # Calculate the position for text labels
    hr_text_x_position = 9

    # Add column headers
    ax.text(hr_text_x_position, len(hr), 'Hazard Ratio', fontweight='bold', verticalalignment='bottom', fontsize=8)
    
    # Add text beside each point
    for i, (h, cip) in enumerate(zip(hr, ci_p)):
        hr_text = cip
        ax.text(hr_text_x_position, i, hr_text, verticalalignment='center', fontsize=10)
        
    # Adjust layout to prevent clipping
    plt.tight_layout()
    
    ax.text(0.7, len(hr), title, fontsize=10, fontweight='bold', va='bottom', ha='left')

    # Save the plot as a PDF file
    plt.savefig(file_path, format='pdf', bbox_inches='tight')
    
    # Show the plot
    plt.show()

In [ ]:
# # Define the dictionary
# hrf_dict = {
#     'group': ['0 high risk features', '1 high risk feature', '2 or 3 high risk features', '4 or 5 high risk features'],
#     'hazard_ratio': [1, 1.45, 2.69, 5.30],
#     'ci_lower': [np.nan, 1.21, 2.16, 3.68],
#     'ci_upper': [np.nan, 1.74, 3.35, 7.61],
#     'ci_string': ['Reference', 
#                   '1.45 [1.21, 1.74], p = 6.70e-05', 
#                   '2.69 [2.16, 3.35], p < 2.00e-16', 
#                   '5.30 [3.68, 7.61], p < 2.00e-16']
# }

# # Convert the dictionary into a DataFrame
# hrf_df = pd.DataFrame(hrf_dict)

In [ ]:
# hrf_to_forest(hrf_df, 
#               'Figure 3B', 
#               'hrf_hazard_ratios.pdf')

### Figure 3C

In [ ]:
c_grouped_features = add_feature_count_columns(c_grouped, features)
c_grouped_features['feature_split_group'] = c_grouped_features['feature_count'].apply(map_feature_count_to_group)

In [ ]:
def cumulative_incidence_table(df, time_point, duration_col, event_col, cohort_col='cohort', feature_col='feature_split_group'):
    """
    Compute cumulative incidence for each combination of cohort and feature count.

    Parameters:
    - df: DataFrame containing the data
    - cohort_col: Column name for cohort labels
    - feature_col: Column name for feature counts
    - duration_col: Column name for duration
    - event_col: Column name for event status (1 for event occurred, 0 for censored)

    Returns:
    - result_df: DataFrame with cumulative incidences for each feature count and cohort combination
    """
    # Initialize Kaplan-Meier fitter
    kmf = KaplanMeierFitter()

    # Get unique cohorts and feature counts
    cohorts = df[cohort_col].unique()
    feature_counts = df[feature_col].unique()

    # Prepare an empty list to store results
    results = []

    for feature_count in feature_counts:
        row = {'feature_count': feature_count}
        for cohort in cohorts:
            # Slice the DataFrame for the current cohort and feature count
            sub_df = df[(df[cohort_col] == cohort) & (df[feature_col] == feature_count)]

            if sub_df.empty:
                # If there's no data for this combination, set cumulative incidence to NaN
                row[f'{cohort}'] = np.nan
                continue
            
            # Compute cumulative incidence at the end of the duration
            ci_stats = compute_cumulative_incidence(sub_df, time_point, duration_col, event_col)

            row[f'{cohort}_ci'] = (ci_stats['cumulative_incidence']).round(2)
            row[f'{cohort}_lower_95'] = (ci_stats['confidence_interval_lower']).round(2)
            row[f'{cohort}_upper_95'] = (ci_stats['confidence_interval_upper']).round(2)
            row[f'{cohort}_count'] = sub_df['person_id'].nunique()

        # Append the row to results list
        results.append(row)

    # Create the result DataFrame
    result_df = pd.DataFrame(results)
    
    return result_df.sort_values('feature_count').reset_index(drop=True)

In [ ]:
def compute_cumulative_incidence(df, time_point, duration_col, event_col):
    """
    Compute the cumulative incidence at a specified time point using the Kaplan-Meier estimator.
    
    Parameters:
    - df: DataFrame with columns 'duration' and 'event'.
    - time_point: Time point at which to compute the cumulative incidence.
    
    Returns:
    - A dictionary with the cumulative incidence estimate and 95% confidence intervals.
    """
    # Initialize the Kaplan-Meier Fitter
    kmf = KaplanMeierFitter()
    
    # Fit the model
    kmf.fit(durations=df[duration_col], event_observed=df[event_col])
    
    # Compute cumulative incidence function
    survival_function = kmf.survival_function_
    confidence_intervals = kmf.confidence_interval_

    # Compute cumulative incidence from survival function
    cumulative_incidence = 1 - survival_function
    ci_lower = 1 - confidence_intervals['KM_estimate_upper_0.95']
    ci_upper = 1 - confidence_intervals['KM_estimate_lower_0.95']
    
    # Extract the cumulative incidence and confidence intervals at the specified time point
    if time_point in cumulative_incidence.index:
        estimate_at_time = cumulative_incidence.loc[time_point, 'KM_estimate']
        ci_lower_at_time = ci_lower.loc[time_point]
        ci_upper_at_time = ci_upper.loc[time_point]
    else:
        # Interpolation if the time point is not in the index
        if time_point < cumulative_incidence.index.min() or time_point > cumulative_incidence.index.max():
            # Time point is outside the range of the observed data
            estimate_at_time = np.nan
            ci_lower_at_time = np.nan
            ci_upper_at_time = np.nan
        else:
            estimate_at_time = 1 - np.interp(time_point, survival_function.index, survival_function['KM_estimate'])
            ci_lower_at_time = 1 - np.interp(time_point, confidence_intervals.index, confidence_intervals['KM_estimate_upper_0.95'])
            ci_upper_at_time = 1 - np.interp(time_point, confidence_intervals.index, confidence_intervals['KM_estimate_lower_0.95'])
    
    return {
        'cumulative_incidence': estimate_at_time,
        'confidence_interval_lower': ci_lower_at_time,
        'confidence_interval_upper': ci_upper_at_time
    }

In [ ]:
ci_2_yr_by_feature = cumulative_incidence_table(c_grouped_features[c_grouped_features['mca']==1], 730.5, 'time_cytoses', 'event_cytoses')

In [ ]:
ci_2_yr_by_feature

In [ ]:
def plot_timed_incidence(df, cohort_mapping, title, file_path, figsize=(10, 7)):
    fig, ax = plt.subplots(figsize=figsize)
    
    jitter_mapping = {
        'aou': 0,
#         'biovu': 0.05,
        'ukb': -0.05
    }

    for cohort, display_name in cohort_mapping.items():
        fc = df['feature_count']
        ci = df[f'{cohort}_ci']
        lower_95 = df[f'{cohort}_lower_95']
        upper_95 = df[f'{cohort}_upper_95']
        
        jitter_strength = jitter_mapping[cohort]
        
        # Add fixed jitter to the x-values in the same direction for each point
        fc_jittered = fc + jitter_strength
        
        ax.errorbar(fc_jittered, ci, yerr=[ci - lower_95, upper_95 - ci], fmt='o', label=display_name, alpha=0.7, capsize=5)

    ax.set_xlabel('Feature count')
    ax.set_ylabel('Cumulative Incidence of Cytoses')
    ax.legend()
    ax.set_title(title)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(False)
    
    ax.set_yticks([0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60])
    
    x_label_mapping = {0: '0', 1: '1', 2: '2', 3: '≥3'}
    ticks = [0, 1, 2, 3]
    ax.set_xticks(ticks)
    ax.set_xticklabels([x_label_mapping[tick] for tick in ticks])

    table_data = []
    for cohort, _ in cohort_mapping.items():
        cohort_data = df.groupby('feature_count')[f'{cohort}_count'].sum()
        cohort_data = cohort_data.reindex(range(4), fill_value=0)
        table_data.append(cohort_data.tolist())

    cell_text = [[f'{int(x):,}' for x in row] for row in table_data]

    # Adjust the bbox to align with x-axis ticks
    table = ax.table(cellText=cell_text,
                     rowLabels=list(cohort_mapping.values()),
                     loc='bottom',
                     cellLoc='center',
                     colWidths=[.2, .1, .1, .4],
                     bbox=[-0.38, -0.25, 1.12, 0.1])  # Adjust these values as needed

    for key, cell in table.get_celld().items():
        cell.set_linewidth(0)
        cell.set_fontsize(9)
        cell.set_height(0.05)
        
        # Center-align row labels
        if key[1] == -1:
            cell._loc = 'center'
        
    # Automatically adjust column widths
    table.auto_set_column_width(range(len(ticks)))

    plt.tight_layout()
    plt.subplots_adjust(bottom=0.2)
    
    # Save the plot as a PDF file
    plt.savefig(file_path, format='pdf', bbox_inches='tight')
    
    plt.show()

In [ ]:
cohort_mapping = {
    'aou': 'All of Us',
#     'biovu': 'BioVU',
    'ukb': 'UKB'
}

plot_timed_incidence(ci_2_yr_by_feature, cohort_mapping, 'Figure 3C', f'{cohort_name}_2_year_ci_risk_features.pdf')

## Figure 4

### Figure 4A

In [ ]:
c_grouped = c_grouped[c_grouped['time_cytoses_years']>0]

In [ ]:
# Convert datetime columns to datetime type if they aren't already
datetime_columns = ['index_datetime',
                    'persistent_cytopenia_datetime',
                    'persistent_cytoses_datetime',
                    'aml_mds_mf_date', 
                    'death_date',
                    'cll_date']
for col in datetime_columns:
    c_grouped[col] = pd.to_datetime(c_grouped[col])

# Calculate duration (in days) until event or censoring for all rows
c_grouped['duration_cll'] = (c_grouped[['index_datetime',
                    'persistent_cytopenia_datetime',
                    'persistent_cytoses_datetime',
                    'aml_mds_mf_date', 
                    'death_date',
                    'et_date', 'pv_date', 'cll_date']].min(axis=1) - c_grouped['index_datetime']).dt.days

# Convert duration from days to years
c_grouped['duration_cll_years'] = c_grouped['duration_cll'] / 365.25

# Create a mask for valid durations (not NaN and not negative)
valid_duration_mask = (
    (c_grouped['incident_cll'] == 1) & 
    ((c_grouped['persistent_cytoses_datetime'].isna()) | 
     (c_grouped['cll_date'] >= c_grouped['persistent_cytoses_datetime']))
)

# Determine if the event (incident myeloid neoplasm) occurred for rows with valid duration
c_grouped['event_cll'] = np.where(valid_duration_mask, 1, 0)

In [ ]:
# Select relevant columns for survival analysis
c_grouped_mn = c_grouped[['person_id', 'mca', 'persistent_cytoses', 'duration_cll', 'event_cll']].drop_duplicates('person_id').reset_index(drop=True)

In [ ]:
# Define conditions and corresponding values
conditions = [
    (c_grouped_mn['mca'] == 0) & (c_grouped_mn['persistent_cytoses'] == 0),
    (c_grouped_mn['mca'] == 0) & (c_grouped_mn['persistent_cytoses'] == 1),
    (c_grouped_mn['mca'] == 1) & (c_grouped_mn['persistent_cytoses'] == 0),
    (c_grouped_mn['mca'] == 1) & (c_grouped_mn['persistent_cytoses'] == 1)
]

values = [0, 1, 2, 3]

# Create the new column based on conditions
c_grouped_mn['mca_cytoses'] = np.select(conditions, values)

In [ ]:
def mn_stats(df):
    
    time = 365
    
    # Filter for individuals with CHIP but no persistent cytopenia
    chip_no_cytopenia = df[(df['mca'] == 1) & (df['persistent_cytoses'] == 0)]
    
    # Number of individuals who had event_mn == 1
    chip_no_cytopenia_event = chip_no_cytopenia[chip_no_cytopenia['event_cll'] == 1]
    num_chip_no_cytopenia_event = len(chip_no_cytopenia_event)
    
    # Median, 25th, and 75th percentiles of duration_mn
    median_duration_no_cytopenia = chip_no_cytopenia_event['duration_cll'].median() / time
    percentile_25_no_cytopenia = chip_no_cytopenia_event['duration_cll'].quantile(0.25) / time
    percentile_75_no_cytopenia = chip_no_cytopenia_event['duration_cll'].quantile(0.75) / time
    
    # Filter for individuals with CHIP and persistent cytopenia
    chip_with_cytopenia = df[(df['mca'] == 1) & (df['persistent_cytoses'] == 1)]
    
    # Number of individuals who had event_mn == 1
    chip_with_cytopenia_event = chip_with_cytopenia[chip_with_cytopenia['event_cll'] == 1]
    num_chip_with_cytopenia_event = len(chip_with_cytopenia_event)
    
    # Median, 25th, and 75th percentiles of duration_mn
    median_duration_with_cytopenia = chip_with_cytopenia_event['duration_cll'].median() / time
    percentile_25_with_cytopenia = chip_with_cytopenia_event['duration_cll'].quantile(0.25) / time
    percentile_75_with_cytopenia = chip_with_cytopenia_event['duration_cll'].quantile(0.75) / time
    
    return {
        'No persistent cytoses': {
            'Total': chip_no_cytopenia['person_id'].nunique(),
            'Number with event_cll == 1': num_chip_no_cytopenia_event,
            'Median duration_cll': median_duration_no_cytopenia,
            '25th percentile duration_cll': percentile_25_no_cytopenia,
            '75th percentile duration_cll': percentile_75_no_cytopenia
        },
        'With persistent cytoses': {
            'Total': chip_with_cytopenia['person_id'].nunique(),
            'Number with event_cll == 1': num_chip_with_cytopenia_event,
            'Median duration_cll': median_duration_with_cytopenia,
            '25th percentile duration_cll': percentile_25_with_cytopenia,
            '75th percentile duration_cll': percentile_75_with_cytopenia
        }
    }

In [ ]:
mn_stats(c_grouped_mn)

In [ ]:
plot_cumulative_incidence(c_grouped_mn, 
                          'mca_cytoses',
                          'duration_cll',
                          'event_cll',
                          'Figure 4A. Cumulative incidence of AML, MDS, MF', 
                          unit_of_time,
                          unit_of_time_label,
                          {0: 'Controls without incident cytopenia', 1: 'Controls with incident cytopenia', 
                           2: 'Cases without incident cytopenia', 3: 'Cases with incident cytopenia'},
                          {0: 'black', 1: 'red', 2: 'gray', 3: 'blue'},
                          f'{cohort_name}_ci_neoplasm_by_cytopenia.pdf',
                          x_max = 7,
                          y_max = 0.16, 
                          ci=False)

### Figure 4B

In [ ]:
def mn_to_forest(df, 
                 title, 
                 file_path, 
                 figsize=(14, 6)):

    # Sort the DataFrame by 'hazard_ratio' or any other column that determines the order
    df = df.sort_values(by='hazard_ratio', ascending=False)
    
    # Prepare the data for the plot
    hr = df['hazard_ratio']
    ci_lower = df['ci_lower']
    ci_upper = df['ci_upper']
    ci_p = df['ci_string']
    labels = df['group']

    # Create the plot
    fig, ax = plt.subplots(figsize=figsize)

    # Plot the hazard ratios with 95% CI
    ax.errorbar(hr, range(len(hr)), xerr=[hr - ci_lower, ci_upper - hr], fmt='o', color='black', ecolor='gray', capsize=3)

    # Add a vertical line at HR=1
    ax.axvline(x=1, linestyle='--', color='black')

    # Set y-axis labels
    ax.set_yticks(range(len(hr)))
    ax.set_yticklabels(labels)
    
    ax.tick_params(top=False,
                   bottom=False,
                   left=False,
                   right=False,
                   labelleft=True,
                   labelbottom=True)

    # Set x-axis to be linear with custom ticks and a narrower range
    ax.set_xscale('linear')
    ax.set_xlim(0.8, 1000)
    ax.set_xticks([1, 10, 100, 1000])
    ax.set_xlabel('Hazard Ratio')

    # Remove all spines except the bottom one
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    # Remove all grid lines
    ax.grid(False)

    # Calculate the position for text labels
    hr_text_x_position = 1000

    # Add column headers
    ax.text(hr_text_x_position, len(hr), 'Hazard Ratio', fontweight='bold', verticalalignment='bottom', fontsize=8)
    
    # Add text beside each point
    for i, (h, cip) in enumerate(zip(hr, ci_p)):
        hr_text = cip
        ax.text(hr_text_x_position, i, hr_text, verticalalignment='center', fontsize=10)
        
    # Adjust layout to prevent clipping
    plt.tight_layout()
    
    ax.text(0.7, len(hr), title, fontsize=10, fontweight='bold', va='bottom', ha='left')
    
    # Save the plot as a PDF file
    plt.savefig(file_path, format='pdf', bbox_inches='tight')
    
    # Show the plot
    plt.show()

In [ ]:
# Define the dictionary
mn_dict = {
    'group': ['Controls without cytopenia', 'Cases without cytopenia', 'Controls with cytopenia', 'Cases with cytopenia'],
    'hazard_ratio': [1, 5.38, 44.6, 131],
    'ci_lower': [np.nan, 2.0, 18.8, 55.8],
    'ci_upper': [np.nan, 14.3, 106, 310],
    'ci_string': ['Reference', 
                  '5.38 [2.0, 14.3], p = 7.60e-04', 
                  '44.6 [18.8, 106], p < 2.00e-16', 
                  '131 [55.8, 310], p < 2.00e-16']
}

# Convert the dictionary into a DataFrame
mn_df = pd.DataFrame(mn_dict)

In [ ]:
mn_to_forest(mn_df, 'Figure 4B', 'mn_hazard_ratios.pdf')

## Table 2

In [ ]:
def compute_myeloid_neoplasm_stats(df, group_col):
    # Initialize an empty DataFrame to store results
    results = pd.DataFrame()
    
    # Get unique values from the grouping column
    group_values = df[group_col].unique()
    
    for value in group_values:
        # Filter data for the current group value
        group_df = df[df[group_col] == value].copy()
        
        total = group_df['person_id'].nunique()
        
        # Compute total unique person_id with incident_aml_mds_mf == 1
        total_cancer = group_df[group_df['incident_aml_mds_mf'] == 1]['person_id'].nunique()
        total_cancer_p = 100*total_cancer/total
        
        # Compute unique person_id count with additional filtering
        filtered_df = group_df[
            (group_df['incident_aml_mds_mf'] == 1) & 
            (group_df['aml_mds_mf_date'] > group_df['persistent_cytopenia_datetime'])
        ].copy()
        unique_person_count = filtered_df['person_id'].nunique()
        unique_person_p = 100*unique_person_count/total
        
        # Convert date columns to datetime if not already
        filtered_df['persistent_cytopenia_datetime'] = pd.to_datetime(filtered_df['persistent_cytopenia_datetime'])
        filtered_df['index_datetime'] = pd.to_datetime(filtered_df['index_datetime'])
        filtered_df['aml_mds_mf_date'] = pd.to_datetime(filtered_df['aml_mds_mf_date'])
        
        # Compute time to cytopenia for filtered data
        filtered_df = filtered_df.dropna(subset=['persistent_cytopenia_datetime', 'index_datetime'])
        filtered_df['time_to_cytopenia'] = (filtered_df['persistent_cytopenia_datetime'] - filtered_df['index_datetime']).dt.days / 365.25
        
        # Compute time from cytopenia to malignancy for filtered data
        filtered_df['time_from_cytopenia_to_malignancy'] = (filtered_df['aml_mds_mf_date'] - filtered_df['persistent_cytopenia_datetime']).dt.days / 365.25
        
        time_to_cytopenia_values = filtered_df['time_to_cytopenia'].dropna()
        time_from_cytopenia_to_malignancy_values = filtered_df['time_from_cytopenia_to_malignancy'].dropna()
        
        # Compute means and percentiles
        time_to_cytopenia_median = time_to_cytopenia_values.median() if not time_to_cytopenia_values.empty else np.nan
        time_to_cytopenia_25th = np.percentile(time_to_cytopenia_values, 25) if not time_to_cytopenia_values.empty else np.nan
        time_to_cytopenia_75th = np.percentile(time_to_cytopenia_values, 75) if not time_to_cytopenia_values.empty else np.nan
        
        time_from_cytopenia_to_malignancy_median = time_from_cytopenia_to_malignancy_values.median() if not time_from_cytopenia_to_malignancy_values.empty else np.nan
        time_from_cytopenia_to_malignancy_25th = np.percentile(time_from_cytopenia_to_malignancy_values, 25) if not time_from_cytopenia_to_malignancy_values.empty else np.nan
        time_from_cytopenia_to_malignancy_75th = np.percentile(time_from_cytopenia_to_malignancy_values, 75) if not time_from_cytopenia_to_malignancy_values.empty else np.nan
        
        # Append results to the DataFrame
        results[value] = pd.Series({
            'Participants': total,
            'Total AML, MDS, MF': total_cancer,
            'Percent total AML, MDS, MF': total_cancer_p,
            'Incident AML, MDS, MF': unique_person_count,
            'Percent incident AML, MDS, MF': unique_person_p,
            'Median Time to Cytopenia (years)': time_to_cytopenia_median,
            '25th Percentile Time to Cytopenia (years)': time_to_cytopenia_25th,
            '75th Percentile Time to Cytopenia (years)': time_to_cytopenia_75th,
            'Median Time from Cytopenia to Malignancy (years)': time_from_cytopenia_to_malignancy_median,
            '25th Percentile Time from Cytopenia to Malignancy (years)': time_from_cytopenia_to_malignancy_25th,
            '75th Percentile Time from Cytopenia to Malignancy (years)': time_from_cytopenia_to_malignancy_75th
        })
    
    return results

In [ ]:
compute_myeloid_neoplasm_stats(c_grouped, 'case')

In [ ]:
c_grouped[(c_grouped['case']==0)].drop_duplicates(subset='person_id')['first_aml_mds_mf'].value_counts()

In [ ]:
c_grouped[(c_grouped['case']==1) & (c_grouped['aml_mds_mf_date'] > c_grouped['persistent_cytopenia_datetime'])].drop_duplicates(subset='person_id')['first_aml_mds_mf'].value_counts()

# Miscellaneous

## Move to bucket

In [ ]:
!ls

In [ ]:
!gsutil cp broganjf/chip_cytopenia/ukb_cohort.csv {BUCKET}/broganjf/chip_cytopenia

## Hematologic malignancy ICD9 and ICD10 codes

In [ ]:
heme_malig_table = pd.read_csv(os.path.join('ICD_9_10_hematologic_malignancy_exclusion_codes.csv'))

In [ ]:
icd_10_codes

In [ ]:
def aggregate_phenotypes(df):
    # Ensure datetime columns are in datetime format
    df['observation_datetime'] = pd.to_datetime(df['observation_datetime'])
    
    # Group by person_id and concept_name, then aggregate
    grouped_df = df.groupby(['person_id']).agg(
        concept_name=('concept_name', 'first'),
        date_first_heme_ca=('observation_datetime', 'min')
    ).reset_index()
    
    return grouped_df

In [ ]:
icd_heme_malig_list = heme_malig_table['ICD_CODE'].to_list()
query = f"""
        SELECT person_id
            , observation_datetime
            , concept_name
            , vocabulary_id
            , concept_code 
        FROM 
            {DATASET}.observation    
        LEFT JOIN `{DATASET}.concept` as c on c.concept_id = observation_source_concept_id
        WHERE vocabulary_id IN ('ICD9CM', 'ICD10CM') AND concept_code IN UNNEST({icd_heme_malig_list})
        ORDER BY
            person_id
        """

heme_malig_obs = pd.read_gbq(query, use_bqstorage_api=True, progress_bar_type='tqdm_notebook')

In [ ]:
heme_phenotypes = aggregate_phenotypes(heme_malig_obs)
heme_phenotypes

In [ ]:
heme_phenotypes['person_id'].nunique()

In [ ]:
save_to_bucket(heme_phenotypes, 'broganjf/chip_to_ccus', 'heme_phenotypes')

## Covariates

In [ ]:
pd.read_csv('gs://fc-secure-cb192ac6-30ba-46b9-92ee-896a6e36c63e/hpoisner/v6_v7_covariates.txt', delimiter='\t')

## Queries

In [ ]:
visit_type_query = f"""
SELECT  
    ve.visit_occurrence_id,  -- Unique visit encounter ID
    ve.person_id,
    ve.visit_concept_id AS encounter_type_num,
    ec.concept_name AS encounter_type_name,
    ve.visit_start_datetime,
    ve.visit_end_datetime,
    DATE_DIFF(ve.visit_end_datetime, ve.visit_start_datetime, DAY) AS visit_duration_days,
    dc.discharge_status,
    ve.visit_type_concept_id,
    vt.concept_name AS visit_type
FROM 
    `{DATASET}.visit_occurrence` ve
LEFT JOIN 
    `{DATASET}.visit_detail` vd 
    ON ve.visit_occurrence_id = vd.visit_occurrence_id
LEFT JOIN 
    (SELECT 
        concept_id, 
        concept_name AS discharge_status 
     FROM 
        `{DATASET}.concept`) dc
    ON vd.discharge_to_concept_id = dc.concept_id
LEFT JOIN 
    (SELECT 
        concept_id, 
        concept_name 
     FROM 
        `{DATASET}.concept`) vt
    ON ve.visit_type_concept_id = vt.concept_id
LEFT JOIN 
    (SELECT 
        concept_id, 
        concept_name 
     FROM 
        `{DATASET}.concept`) ec
    ON ve.visit_concept_id = ec.concept_id
-- WHERE 
--    ve.visit_concept_id IN (9201, 262)  -- Hospital admissions concept IDs
"""

visit_type = pandas_gbq.read_gbq(
    visit_type_query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

visit_type.head()

In [ ]:
visits = visit_type[visit_type['person_id'].isin(participants['person_id'].unique())]

In [ ]:
save_to_bucket(visits, 'broganjf/chip_to_ccus', 'visits')

In [ ]:
visit_type_query = f"""
SELECT 
    DISTINCT ve.visit_concept_id, 
    c.concept_name AS visit_concept_name, 
    c.domain_id, 
    c.vocabulary_id
FROM 
    `{DATASET}.visit_occurrence` ve
JOIN 
    `{DATASET}.concept` c 
    ON ve.visit_concept_id = c.concept_id
ORDER BY 
    ve.visit_concept_id
"""

visit_type_table = pandas_gbq.read_gbq(
    visit_type_query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

visit_type_table

In [ ]:
measurement_id_query = f"""
SELECT
    concept_id,
    LOWER(concept_name) AS concept_name
FROM
    `{DATASET}.concept`
WHERE
    vocabulary_id = 'LOINC';
"""

measurement_id = pandas_gbq.read_gbq(
    measurement_id_query,
    dialect="standard",
    use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
    progress_bar_type="tqdm_notebook"
)

measurement_id

In [ ]:
substring = 'lymphocytes'
measurement_id[measurement_id['concept_name'].str.contains(substring)] #.sort_values(by='concept_id')

## Plots

In [ ]:
import matplotlib.pyplot as plt

def plot_participants_data(df, lab_name):
    # Convert 'measurement_datetime' to datetime if it's not already
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    
    # Plot each participant's data separately
    for person_id, group in df.groupby('person_id'):
        plt.figure()  # Create a new figure for each participant
        plt.scatter(group['measurement_datetime'], group[lab_name])
        plt.xlabel('Measurement Date')
        plt.ylabel('Lab Value')
        plt.title(f'{lab_name} Over Time (Participant {person_id})')
        plt.show()

# Example usage:
# Assuming df is your timeseries DataFrame with columns ['person_id', 'measurement_datetime', 'lab_name', 'value_as_number']
# plot_participants_data(df, 'lab_name')

In [ ]:
plot_participants_data(chip_cbc_annotated[chip_cbc_annotated['person_id'].isin(chip_cbc_annotated['person_id'].unique()[:10])].copy(deep=True), 'hemoglobin')

## Monoclonal B cell Lymphocytosis to Chronic Lymphocytic Leukemia

In [ ]:
# Read in tables for unit processing
tables = read_tables('v0_2')
metadata = tables['metadata']
unit_map = tables['unit_map']
unit_reduce = tables['unit_reduce']

m_cids = [3004327]
m_vars = metadata[metadata['measurement_concept_id'].isin(m_cids)]['lab_name'].to_list()
print(f'Preparing laboratory measurements for the following variables: {m_vars}')

m = omop_query(m_cids)
query_summary(m)

In [ ]:
# Run quality control process
m_preprocessed = preprocess(m)
m_harmonized = harmonize(m_preprocessed, metadata, unit_map, unit_reduce)
m_final = trim(m_harmonized)
m_final = m_final.sort_values(by=['person_id', 'measurement_datetime'])

# Descriptive statistics after outliers are removed
m_unitdata = units_dist(m_harmonized)

In [ ]:
save_to_bucket(m_final, 'pershy1/mbl_to_cll', 'alc_06282024')

## Deprecated code

In [ ]:
def find_persistent_condition(df, condition):
    # Convert 'measurement_datetime' to datetime if it's not already
    df['measurement_datetime'] = pd.to_datetime(df['measurement_datetime'])
    
    # Set 'measurement_datetime' as the index
    df.set_index('measurement_datetime', inplace=True)
    
    # Resample the DataFrame by month and take the minimum value of the condition for each month
    df_resampled = df.groupby('person_id')[condition].resample('M').min().reset_index()
    
    # Forward fill missing data
    df_resampled[condition] = df_resampled.groupby('person_id')[condition].ffill()
    
    # Initialize an empty list to store results
    results = []
    
    # Iterate over unique person_ids
    for person_id, group in df_resampled.groupby('person_id'):
        # Apply a rolling window of 4 months and check if condition persists
        rolling_condition = group[condition].rolling(window=4, min_periods=1).sum()
        
        # Check if condition persists for at least 4 months
        if (rolling_condition >= 4).any():
            # Get the first occurrence of persistent condition
            time_of_persistent_condition = group.loc[rolling_condition >= 4].iloc[0]['measurement_datetime']
            
            # Append the result
            results.append({'person_id': person_id, f'persistent_{condition}': 1, f'persistent_{condition}_datetime': time_of_persistent_condition})
    
    # Convert results to DataFrame
    persistent_condition_df = pd.DataFrame(results)
    
    return persistent_condition_df


def calculate_time_to_cytopenia(df, index_datetime_col, cytopenia_col):
    # Ensure 'index_datetime' column is in datetime format
    df[index_datetime_col] = pd.to_datetime(df[index_datetime_col])

    # Find the first occurrence of persistent cytopenia for each person
    first_cytopenia = df.groupby('person_id')[cytopenia_col].idxmax()

    # Create a DataFrame with the first occurrence of persistent cytopenia for each person
    first_cytopenia_df = df.loc[first_cytopenia, ['person_id', cytopenia_col]]

    # Rename the column to indicate the first cytopenia occurrence
    first_cytopenia_df.rename(columns={cytopenia_col: 'first_' + cytopenia_col}, inplace=True)

    # Merge the first cytopenia occurrence DataFrame with the original DataFrame
    df = pd.merge(df, first_cytopenia_df, on='person_id', how='left')

    # Calculate the time to cytopenia as the difference between the first cytopenia occurrence and the index_datetime
    df['time_to_' + cytopenia_col] = df['first_' + cytopenia_col] - df[index_datetime_col]

    return df


def annotate_cbc_data(cbc_data, demographics):
    # Merge dataframes and calculate time differences
    cbc_annotated = pd.merge(cbc_data, demographics[['person_id', 'gender', 'index_datetime']].drop_duplicates(), on='person_id', how='left')
    cbc_annotated['index_to_measurement_timediff'] = (cbc_annotated['measurement_datetime'] - cbc_annotated['index_datetime']).dt.total_seconds() / 86400
    cbc_annotated['abs_diff'] = (cbc_annotated['measurement_datetime'] - cbc_annotated['index_datetime']).abs()

    # Determine index and last CBC
    idx_min = cbc_annotated.groupby('person_id')['abs_diff'].idxmin()
    idx_max = cbc_annotated.groupby('person_id')['measurement_datetime'].idxmax()
    
    cbc_annotated['index_cbc'] = 0
    cbc_annotated.loc[idx_min, 'index_cbc'] = 1

    index_dates = cbc_annotated.loc[idx_min, ['person_id', 'measurement_datetime']].rename(columns={'measurement_datetime': 'index_measurement_datetime'})
    last_dates = cbc_annotated.loc[idx_max, ['person_id', 'measurement_datetime']].rename(columns={'measurement_datetime': 'last_measurement_datetime'})
    merged_dates = pd.merge(index_dates, last_dates, on='person_id')
    merged_dates['max_index_timediff'] = (merged_dates['last_measurement_datetime'] - merged_dates['index_measurement_datetime']).dt.total_seconds() / 86400.0

    cbc_annotated = pd.merge(cbc_annotated, merged_dates[['person_id', 'max_index_timediff']], on='person_id', how='left')
    cbc_annotated = cbc_annotated.drop(columns=['abs_diff'])

    # Filter and count post-index measurements
    filtered_df = cbc_annotated[cbc_annotated['index_to_measurement_timediff'] > 0]
    cbc_counts = filtered_df.groupby('person_id').size().reset_index(name='cbc_count_post_index')
    cbc_annotated = cbc_annotated.merge(cbc_counts, on='person_id', how='left')
    cbc_annotated['cbc_count_post_index'] = cbc_annotated['cbc_count_post_index'].fillna(0).astype(int)

    # Initialize cytopenia columns
    cbc_annotated['anemia'] = 0
    cbc_annotated['leukopenia'] = 0
    cbc_annotated['thrombocytopenia'] = 0

    # Assign gender 'Other' for non-standard values
    cbc_annotated.loc[~cbc_annotated['gender'].isin(['Male', 'Female']), 'gender'] = 'Other'

    # Apply conditions for cytopenia
    anemia_cutoffs = {'Male': 13.5, 'Female': 12.0, 'Other': 12.0}
    cbc_annotated.loc[(cbc_annotated['gender'] == 'Male') & (cbc_annotated['hemoglobin'] < anemia_cutoffs['Male']), 'anemia'] = 1
    cbc_annotated.loc[(cbc_annotated['gender'] == 'Female') & (cbc_annotated['hemoglobin'] < anemia_cutoffs['Female']), 'anemia'] = 1
    cbc_annotated.loc[(cbc_annotated['gender'] == 'Other') & (cbc_annotated['hemoglobin'] < anemia_cutoffs['Other']), 'anemia'] = 1
    cbc_annotated.loc[cbc_annotated['leukocyte count'] < 3.7, 'leukopenia'] = 1
    cbc_annotated.loc[cbc_annotated['platelets'] < 150, 'thrombocytopenia'] = 1
    
    # Create column cytopenia based on the condition
    cbc_annotated['cytopenia'] = cbc_annotated[['anemia', 'leukopenia', 'thrombocytopenia']].any(axis=1).astype(int)

    # Define and merge persistent conditions
    persistent_anemia = find_persistent_condition(cbc_annotated.copy(deep=True), 'anemia')
    persistent_leukopenia = find_persistent_condition(cbc_annotated.copy(deep=True), 'leukopenia')
    persistent_thrombocytopenia = find_persistent_condition(cbc_annotated.copy(deep=True), 'thrombocytopenia')

    cbc_annotated = cbc_annotated.merge(persistent_anemia, on='person_id', how='left')
    cbc_annotated = cbc_annotated.merge(persistent_leukopenia, on='person_id', how='left')
    cbc_annotated = cbc_annotated.merge(persistent_thrombocytopenia, on='person_id', how='left')

    cbc_annotated[['persistent_anemia', 'persistent_leukopenia', 'persistent_thrombocytopenia']] = cbc_annotated[['persistent_anemia', 'persistent_leukopenia', 'persistent_thrombocytopenia']].fillna(0).astype(int)

    # Calculate first and last CBC dates, and persistent cytopenia
    cbc_annotated['first_cbc_datetime'] = cbc_annotated.groupby('person_id')['measurement_datetime'].transform('min')
    cbc_annotated['last_cbc_datetime'] = cbc_annotated.groupby('person_id')['measurement_datetime'].transform('max')
    cbc_annotated['persistent_cytopenia'] = (cbc_annotated[['persistent_anemia', 'persistent_leukopenia', 'persistent_thrombocytopenia']].any(axis=1)).astype(int)
    cbc_annotated['persistent_cytopenia_datetime'] = cbc_annotated[['persistent_anemia_datetime', 'persistent_leukopenia_datetime', 'persistent_thrombocytopenia_datetime']].min(axis=1)

    return cbc_annotated


def condense_cbc_data(df):
    # Filter the DataFrame where index_cbc == 1
    filtered_df = df[df['index_cbc'] == 1].reset_index(drop=True)

    # Select the desired columns
    condensed_df = filtered_df[['person_id', 'src_id', 'measurement_datetime', 'hemoglobin', 'leukocyte count', 'platelets', 'gender', 'index_datetime', 'index_to_measurement_timediff', 'index_cbc', 'max_index_timediff']]

    # Define the aggregation dictionary
    aggregation = {
        'first_cbc_datetime': 'first',
        'last_cbc_datetime': 'first',
        'cbc_count_post_index': 'first',
        'anemia': 'max',
        'leukopenia': 'max',
        'thrombocytopenia': 'max',
        'cytopenia': 'max',
        'persistent_anemia': 'max',
        'persistent_leukopenia': 'max',
        'persistent_thrombocytopenia': 'max',
        'persistent_cytopenia': 'max',
        'persistent_cytopenia_datetime': 'max'
    }

    # Group by 'person_id' and apply the aggregation
    aggregated_df = df.groupby('person_id').agg(aggregation).reset_index()

    # Merge the aggregated DataFrame with the filtered DataFrame based on 'person_id'
    condensed_df = pd.merge(condensed_df, aggregated_df, on='person_id', how='left')

    return condensed_df

# chip_cbc_annotated = pd.merge(chip_participants_cbc, chip_participants[['person_id', 'gender', 'index_datetime']].drop_duplicates(), on='person_id', how='left')
# chip_cbc_annotated['index_to_measurement_timediff'] = (chip_cbc_annotated['measurement_datetime'] - chip_cbc_annotated['index_datetime']).dt.total_seconds() / 86400

# # Calculate the absolute difference between measurement_datetime and index_datetime
# chip_cbc_annotated['abs_diff'] = (chip_cbc_annotated['measurement_datetime'] - chip_cbc_annotated['index_datetime']).abs()

# # Find the row with the minimum difference for each person_id
# idx_min = chip_cbc_annotated.groupby('person_id')['abs_diff'].idxmin()

# # Create the index_cbc column and set it to 0
# chip_cbc_annotated['index_cbc'] = 0

# # Set index_cbc to 1 for the rows with the minimum difference
# chip_cbc_annotated.loc[idx_min, 'index_cbc'] = 1

# # Find the last measurement for each person_id
# idx_max = chip_cbc_annotated.groupby('person_id')['measurement_datetime'].idxmax()

# # Merge the index CBC and last CBC dates into a new DataFrame
# index_dates = chip_cbc_annotated.loc[idx_min, ['person_id', 'measurement_datetime']].rename(columns={'measurement_datetime': 'index_measurement_datetime'})
# last_dates = chip_cbc_annotated.loc[idx_max, ['person_id', 'measurement_datetime']].rename(columns={'measurement_datetime': 'last_measurement_datetime'})

# # Merge index and last dates on person_id
# merged_dates = pd.merge(index_dates, last_dates, on='person_id')

# # Calculate the time difference and convert it to days
# merged_dates['max_index_timediff'] = (merged_dates['last_measurement_datetime'] - merged_dates['index_measurement_datetime']).dt.total_seconds() / 86400.0

# # Merge this information back into the original DataFrame
# chip_cbc_annotated = pd.merge(chip_cbc_annotated, merged_dates[['person_id', 'max_index_timediff']], on='person_id', how='left')

# # Drop the temporary 'abs_diff' column
# chip_cbc_annotated = chip_cbc_annotated.drop(columns=['abs_diff'])

# # Filter rows where index_to_measurement_timediff > 0
# filtered_df = chip_cbc_annotated[chip_cbc_annotated['index_to_measurement_timediff'] > 0]

# # Count the number of rows for each person_id in the filtered DataFrame
# cbc_counts = filtered_df.groupby('person_id').size().reset_index(name='cbc_count_post_index')

# # Merge the counts back into the original DataFrame
# chip_cbc_annotated = chip_cbc_annotated.merge(cbc_counts, on='person_id', how='left')

# # Fill NaN values with 0 (for person_ids that have no post-index measurements)
# chip_cbc_annotated['cbc_count_post_index'] = chip_cbc_annotated['cbc_count_post_index'].fillna(0).astype(int)

# # Creating new columns for cytopenias
# chip_cbc_annotated['anemia'] = 0
# chip_cbc_annotated['leukopenia'] = 0
# chip_cbc_annotated['thrombocytopenia'] = 0

# # Select rows where the value in the 'gender' column is not 'Male' or 'Female' and assign them 'Other'
# other_gender_rows = chip_cbc_annotated[~chip_cbc_annotated['gender'].isin(['Male', 'Female'])]
# chip_cbc_annotated.loc[other_gender_rows.index, 'gender'] = 'Other'

# # Cytopenia conditions
# anemia_cutoff_male = 13.5
# anemia_cutoff_female = 12.0
# anemia_cutoff_other = 12.0
# leukopenia_cutoff = 3.7
# thrombocytopenia_cutoff = 150

# # Applying conditions
# chip_cbc_annotated.loc[(chip_cbc_annotated['gender'] == 'Male') & (chip_cbc_annotated['hemoglobin'] < anemia_cutoff_male), 'anemia'] = 1
# chip_cbc_annotated.loc[(chip_cbc_annotated['gender'] == 'Female') & (chip_cbc_annotated['hemoglobin'] < anemia_cutoff_female), 'anemia'] = 1
# chip_cbc_annotated.loc[(chip_cbc_annotated['gender'] == 'Other') & (chip_cbc_annotated['hemoglobin'] < anemia_cutoff_female), 'anemia'] = 1
# chip_cbc_annotated.loc[(chip_cbc_annotated['leukocyte count'] < leukopenia_cutoff), 'leukopenia'] = 1
# chip_cbc_annotated.loc[(chip_cbc_annotated['platelets'] < thrombocytopenia_cutoff), 'thrombocytopenia'] = 1

# # Create column cytopenia based on the condition
# chip_cbc_annotated['cytopenia'] = chip_cbc_annotated[['anemia', 'leukopenia', 'thrombocytopenia']].any(axis=1).astype(int)

# # Apply the function for each condition
# persistent_anemia = find_persistent_condition(chip_cbc_annotated.copy(deep=True), 'anemia')
# persistent_leukopenia = find_persistent_condition(chip_cbc_annotated.copy(deep=True), 'leukopenia')
# persistent_thrombocytopenia = find_persistent_condition(chip_cbc_annotated.copy(deep=True), 'thrombocytopenia')

# chip_cbc_annotated = pd.merge(chip_cbc_annotated, persistent_anemia, on='person_id', how='left')
# chip_cbc_annotated = pd.merge(chip_cbc_annotated, persistent_leukopenia, on='person_id', how='left')
# chip_cbc_annotated = pd.merge(chip_cbc_annotated, persistent_thrombocytopenia, on='person_id', how='left')

# # Fill NaN values in the specified columns with zeros
# chip_cbc_annotated['persistent_anemia'].fillna(0, inplace=True)
# chip_cbc_annotated['persistent_leukopenia'].fillna(0, inplace=True)
# chip_cbc_annotated['persistent_thrombocytopenia'].fillna(0, inplace=True)

# # 1. Create the first_cbc_datetime column
# chip_cbc_annotated['first_cbc_datetime'] = chip_cbc_annotated.groupby('person_id')['measurement_datetime'].transform('min')

# # 2. Create the last_cbc_datetime column
# chip_cbc_annotated['last_cbc_datetime'] = chip_cbc_annotated.groupby('person_id')['measurement_datetime'].transform('max')

# # 3. Create the persistent_cytopenia column
# chip_cbc_annotated['persistent_cytopenia'] = (
#     (chip_cbc_annotated['persistent_anemia'] == 1) |
#     (chip_cbc_annotated['persistent_leukopenia'] == 1) |
#     (chip_cbc_annotated['persistent_thrombocytopenia'] == 1)
# ).astype(int)

# # 4. Create the persistent_cytopenia_datetime column
# chip_cbc_annotated['persistent_cytopenia_datetime'] = chip_cbc_annotated[
#     ['persistent_anemia_datetime', 'persistent_thrombocytopenia_datetime', 'persistent_leukopenia_datetime']
# ].min(axis=1)a

# # Display the resulting dataframe
# chip_cbc_annotated